In [ ]:
# ============================================================
# COSMX CELL 0 — INSTALL DEPENDENCIES + SET PATHS
# ============================================================

!pip -q install scanpy anndata scipy pandas numpy matplotlib seaborn openpyxl pyarrow tables

import os
import re
import gzip
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# User-editable paths
# ------------------------------------------------------------
COSMX_ROOT = Path("/content/cosmx_nsclc")
SCRNA_ROOT = Path("/content/gse131907")

COSMX_ROOT.mkdir(parents=True, exist_ok=True)
SCRNA_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PATH SETUP")
print("=" * 80)
print(f"COSMX_ROOT : {COSMX_ROOT}")
print(f"SCRNA_ROOT : {SCRNA_ROOT}")
print("\nPut/unzip the CosMx NSCLC files inside COSMX_ROOT.")
print("Put/unzip the GSE131907 files inside SCRNA_ROOT.")

In [ ]:
# ============================================================
# COSMX CELL 1 — DOWNLOAD + EXTRACT Lung5_Rep1 SMI FLAT DATA
# ============================================================

import os
import tarfile
import hashlib
import urllib.request
from pathlib import Path

print("=" * 80)
print("COSMX CELL 1 — DOWNLOAD + EXTRACT Lung5_Rep1 SMI FLAT DATA")
print("=" * 80)

# ------------------------------------------------------------
# 1. Define sample and download information
# ------------------------------------------------------------

SAMPLE_NAME = "Lung5_Rep1"

# This URL follows the public NanoString/Bruker SMI compressed-data structure.
# Spaces are encoded as %20.
COSMX_FLAT_DATA_URL = (
    "https://nanostring-public-share.s3.us-west-2.amazonaws.com/"
    "SMI-Compressed/Lung5_Rep1/Lung5_Rep1%20SMI%20Flat%20data.tar.gz"
)

EXPECTED_MD5 = "58d02b183fb2446c8c6f9b5c15ba1321"

# We keep everything inside Colab under COSMX_ROOT.
# COSMX_ROOT was already defined in COSMX CELL 0.
SAMPLE_DIR = COSMX_ROOT / SAMPLE_NAME
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

TAR_PATH = SAMPLE_DIR / "Lung5_Rep1 SMI Flat data.tar.gz"

print(f"Sample name     : {SAMPLE_NAME}")
print(f"Download URL    : {COSMX_FLAT_DATA_URL}")
print(f"Download target : {TAR_PATH}")
print(f"Extract folder  : {SAMPLE_DIR}")

# ------------------------------------------------------------
# 2. Download the file if it is not already downloaded
# ------------------------------------------------------------

if TAR_PATH.exists():
    print("\nFile already exists. Skipping download.")
    print(f"Existing file size: {TAR_PATH.stat().st_size / 1e9:.2f} GB")
else:
    print("\nDownloading Lung5_Rep1 SMI Flat data...")
    print("This is about ~1.5 GB, so it may take several minutes.")

    urllib.request.urlretrieve(COSMX_FLAT_DATA_URL, TAR_PATH)

    print("\nDownload complete.")
    print(f"Downloaded file size: {TAR_PATH.stat().st_size / 1e9:.2f} GB")

# ------------------------------------------------------------
# 3. Verify MD5 checksum
# ------------------------------------------------------------

def compute_md5(file_path, chunk_size=1024 * 1024):
    md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            md5.update(chunk)
    return md5.hexdigest()

print("\nVerifying MD5 checksum...")
actual_md5 = compute_md5(TAR_PATH)

print(f"Expected MD5: {EXPECTED_MD5}")
print(f"Actual MD5  : {actual_md5}")

if actual_md5 != EXPECTED_MD5:
    raise ValueError(
        "MD5 checksum mismatch. The file may be incomplete or corrupted. "
        "Delete the tar.gz file and rerun this cell."
    )

print("MD5 check passed.")

# ------------------------------------------------------------
# 4. Extract tar.gz file
# ------------------------------------------------------------

# Marker file so we do not extract again unnecessarily
EXTRACT_MARKER = SAMPLE_DIR / ".extracted_successfully"

if EXTRACT_MARKER.exists():
    print("\nExtraction marker found. Skipping extraction.")
else:
    print("\nExtracting tar.gz file...")
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        tar.extractall(path=SAMPLE_DIR)

    EXTRACT_MARKER.touch()
    print("Extraction complete.")

# ------------------------------------------------------------
# 5. List extracted files
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXTRACTED FILES")
print("=" * 80)

all_files = sorted([p for p in SAMPLE_DIR.rglob("*") if p.is_file()])

for p in all_files:
    size_mb = p.stat().st_size / 1e6
    print(f"{p.relative_to(SAMPLE_DIR)}  ({size_mb:.2f} MB)")

print("\nTotal extracted/downloaded files:", len(all_files))

# ------------------------------------------------------------
# 6. Automatically identify expected CosMx files
# ------------------------------------------------------------

def find_one(patterns):
    hits = []
    for p in all_files:
        name = p.name.lower()
        for pat in patterns:
            if pat.lower() in name:
                hits.append(p)
                break
    return sorted(hits)

tx_candidates = find_one(["tx_file"])
expr_candidates = find_one(["exprmat_file", "exprmat"])
metadata_candidates = find_one(["metadata_file", "metadata"])
fov_candidates = find_one(["fov_positions_file", "fov_positions"])
cell_label_candidates = find_one(["celllabels"])
compartment_label_candidates = find_one(["compartmentlabels"])

print("\n" + "=" * 80)
print("EXPECTED PIPELINE FILE CHECK")
print("=" * 80)

def print_candidates(label, candidates):
    status = "FOUND" if candidates else "MISSING"
    print(f"\n[{status}] {label}")
    if candidates:
        for c in candidates:
            print(f"  - {c}")

print_candidates("Transcript / molecule file", tx_candidates)
print_candidates("Cell × gene expression matrix", expr_candidates)
print_candidates("Cell metadata file", metadata_candidates)
print_candidates("FOV positions file", fov_candidates)
print_candidates("CellLabels images", cell_label_candidates)
print_candidates("CompartmentLabels images", compartment_label_candidates)

# ------------------------------------------------------------
# 7. Save selected paths for later cells
# ------------------------------------------------------------

COSMX_TRANSCRIPTS_PATH = tx_candidates[0] if tx_candidates else None
COSMX_EXPR_PATH = expr_candidates[0] if expr_candidates else None
COSMX_METADATA_PATH = metadata_candidates[0] if metadata_candidates else None
COSMX_FOV_PATH = fov_candidates[0] if fov_candidates else None

COSMX_CELL_LABEL_PATHS = cell_label_candidates
COSMX_COMPARTMENT_LABEL_PATHS = compartment_label_candidates

print("\n" + "=" * 80)
print("SELECTED PATHS FOR NEXT CELLS")
print("=" * 80)

print(f"COSMX_TRANSCRIPTS_PATH : {COSMX_TRANSCRIPTS_PATH}")
print(f"COSMX_EXPR_PATH        : {COSMX_EXPR_PATH}")
print(f"COSMX_METADATA_PATH    : {COSMX_METADATA_PATH}")
print(f"COSMX_FOV_PATH         : {COSMX_FOV_PATH}")
print(f"CellLabels count       : {len(COSMX_CELL_LABEL_PATHS)}")
print(f"CompartmentLabels count: {len(COSMX_COMPARTMENT_LABEL_PATHS)}")

if COSMX_TRANSCRIPTS_PATH is None or COSMX_EXPR_PATH is None or COSMX_METADATA_PATH is None:
    raise FileNotFoundError(
        "One or more required CosMx flat files were not found after extraction. "
        "Check the extracted file list above."
    )

print("\nCOSMX CELL 1 finished successfully.")
print("Next step: load and inspect the transcript/molecule file.")

In [ ]:
# ============================================================
# REPLACEMENT COSMX CELL 2D — LOAD FULL TRANSCRIPT CSV AT ONCE
# High-RAM version for Colab Pro.
#
# Purpose:
#   Load Lung5_Rep1_tx_file.csv fully into memory,
#   standardize it into Xenium-like molecule columns,
#   save one clean parquet file for downstream cells.
#
# This replaces the chunked Cell 2B/2C.
# ============================================================

import os
import gc
import json
import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 80)
print("REPLACEMENT COSMX CELL 2D — LOAD FULL TRANSCRIPT CSV AT ONCE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check transcript file
# ------------------------------------------------------------

COSMX_TRANSCRIPTS_PATH = Path(COSMX_TRANSCRIPTS_PATH)

if not COSMX_TRANSCRIPTS_PATH.exists():
    raise FileNotFoundError(f"Transcript file not found: {COSMX_TRANSCRIPTS_PATH}")

print(f"Transcript file: {COSMX_TRANSCRIPTS_PATH}")
print(f"File size      : {COSMX_TRANSCRIPTS_PATH.stat().st_size / 1e9:.2f} GB")

# ------------------------------------------------------------
# 2. Define expected columns and dtypes
# ------------------------------------------------------------

required_tx_cols = [
    "fov",
    "cell_ID",
    "x_global_px",
    "y_global_px",
    "x_local_px",
    "y_local_px",
    "z",
    "target",
    "CellComp",
]

dtype_map = {
    "fov": "int32",
    "cell_ID": "int32",
    "x_global_px": "float32",
    "y_global_px": "float32",
    "x_local_px": "float32",
    "y_local_px": "float32",
    "z": "float32",
    "target": "string",
    "CellComp": "string",
}

# ------------------------------------------------------------
# 3. Preview first rows
# ------------------------------------------------------------

print("\nReading preview...")
tx_preview = pd.read_csv(COSMX_TRANSCRIPTS_PATH, nrows=10)

print("\nOriginal transcript preview:")
display(tx_preview)

print("\nColumns:")
print(list(tx_preview.columns))

missing_cols = [c for c in required_tx_cols if c not in tx_preview.columns]
if missing_cols:
    raise KeyError(f"Missing required transcript columns: {missing_cols}")

print("\nAll required transcript columns are present.")

# ------------------------------------------------------------
# 4. Load full transcript CSV
# ------------------------------------------------------------

print("\nLoading full transcript CSV into memory...")
print("This may take several minutes and may use substantial RAM.")

transcripts_df = pd.read_csv(
    COSMX_TRANSCRIPTS_PATH,
    usecols=required_tx_cols,
    dtype=dtype_map,
    low_memory=False,
)

print("\nLoaded full transcripts_df.")
print(f"Shape       : {transcripts_df.shape}")
print(f"Memory usage: {transcripts_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Reset index to avoid any pandas alignment problems.
transcripts_df = transcripts_df.reset_index(drop=True)

# ------------------------------------------------------------
# 5. Standardize into Xenium-like molecule table
# ------------------------------------------------------------

print("\nStandardizing molecule table...")

n = len(transcripts_df)

# Clean gene and compartment columns.
target = transcripts_df["target"].fillna("Unknown").astype(str).str.strip()
cellcomp = transcripts_df["CellComp"].fillna("Unknown").astype(str).str.strip()
cellcomp_lower = cellcomp.str.lower()

# Build standardized table using numpy arrays to avoid index-alignment bugs.
molecules_raw = pd.DataFrame({
    "transcript_id": np.arange(n, dtype=np.int64).astype(str),

    "fov": transcripts_df["fov"].to_numpy(dtype=np.int32),
    "cell_ID_original": transcripts_df["cell_ID"].to_numpy(dtype=np.int32),

    # CosMx cell_ID is only unique within one FOV.
    # So the global cell ID must combine fov and cell_ID.
    "cell_id": (
        transcripts_df["fov"].astype(str).to_numpy()
        + "_"
        + transcripts_df["cell_ID"].astype(str).to_numpy()
    ),

    # CosMx calls gene names "target".
    "gene_id": target.to_numpy(),

    # Use global pixel coordinates for tissue-wide coordinates.
    "x": transcripts_df["x_global_px"].to_numpy(dtype=np.float32),
    "y": transcripts_df["y_global_px"].to_numpy(dtype=np.float32),
    "z": transcripts_df["z"].fillna(0).to_numpy(dtype=np.float32),

    # Local FOV coordinates.
    "x_local": transcripts_df["x_local_px"].to_numpy(dtype=np.float32),
    "y_local": transcripts_df["y_local_px"].to_numpy(dtype=np.float32),

    # Subcellular compartment.
    "CellComp": cellcomp.to_numpy(),

    # CosMx replacement for Xenium overlaps_nucleus.
    "overlaps_nucleus": cellcomp_lower.str.contains(
        "nuclear", na=False
    ).astype(np.int8).to_numpy(),

    # CosMx flat data does not have Xenium-like QV.
    # Keep placeholder for code compatibility.
    "quality": np.full(n, 40.0, dtype=np.float32),

    # cell_ID = 0 means transcript was not assigned to a segmented cell.
    "is_unassigned": (
        transcripts_df["cell_ID"].to_numpy(dtype=np.int32) == 0
    ),
})

# Free original CSV dataframe to save RAM.
del transcripts_df
gc.collect()

print("\nStandardized molecules_raw created.")
print(f"Shape       : {molecules_raw.shape}")
print(f"Memory usage: {molecules_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")

display(molecules_raw.head(10))

# ------------------------------------------------------------
# 6. Diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MOLECULE TABLE DIAGNOSTICS")
print("=" * 80)

gene_lower = molecules_raw["gene_id"].astype(str).str.lower()
comp_lower = molecules_raw["CellComp"].astype(str).str.lower()

n_none_gene = int(gene_lower.eq("none").sum())
n_unknown_gene = int(gene_lower.eq("unknown").sum())

n_assigned = int((~molecules_raw["is_unassigned"]).sum())
n_unassigned = int(molecules_raw["is_unassigned"].sum())

n_nuclear = int(comp_lower.str.contains("nuclear", na=False).sum())
n_membrane = int(comp_lower.str.contains("membrane", na=False).sum())
n_cytoplasm = int(comp_lower.str.contains("cytoplasm|cytoplasmic", na=False).sum())
n_extracellular = int(
    (comp_lower.eq("0") | comp_lower.str.contains("extracellular", na=False)).sum()
)

print(f"Total transcript rows       : {len(molecules_raw):,}")
print(f"Assigned transcripts        : {n_assigned:,}")
print(f"Unassigned transcripts      : {n_unassigned:,}")
print(f"Unique FOVs                 : {molecules_raw['fov'].nunique():,}")
print(f"Unique genes/targets        : {molecules_raw['gene_id'].nunique():,}")
print(f"gene_id='None' rows         : {n_none_gene:,}")
print(f"gene_id='Unknown' rows      : {n_unknown_gene:,}")
print(f"Nuclear transcripts         : {n_nuclear:,}")
print(f"Membrane transcripts        : {n_membrane:,}")
print(f"Cytoplasm transcripts       : {n_cytoplasm:,}")
print(f"Extracellular/0 transcripts : {n_extracellular:,}")

print("\nTop 30 genes/targets:")
display(molecules_raw["gene_id"].value_counts().head(30).reset_index().rename(
    columns={"index": "gene_id", "gene_id": "count"}
))

print("\nCellComp distribution:")
display(molecules_raw["CellComp"].value_counts(dropna=False).head(20).reset_index())

# Critical safety check.
if n_none_gene > 0:
    raise ValueError(
        f"Found {n_none_gene:,} rows with gene_id='None'. "
        "This should not happen. Stop and inspect the target column."
    )

if n_unknown_gene > 1000:
    raise ValueError(
        f"Found {n_unknown_gene:,} rows with gene_id='Unknown'. "
        "This suggests gene names were not read correctly."
    )

# ------------------------------------------------------------
# 7. Save one parquet file, compatible with later chunk-based code
# ------------------------------------------------------------

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_CHUNK_DIR = WORK_DIR / "molecule_parquet_chunks"
PARQUET_CHUNK_DIR.mkdir(parents=True, exist_ok=True)

# Delete old/corrupted chunks.
old_chunks = list(PARQUET_CHUNK_DIR.glob("molecules_chunk_*.parquet"))
print(f"\nDeleting old molecule parquet chunks: {len(old_chunks)}")
for old in old_chunks:
    old.unlink()

single_parquet_path = PARQUET_CHUNK_DIR / "molecules_chunk_0000.parquet"

print("\nSaving full standardized molecule table as one parquet file...")
print(f"Output: {single_parquet_path}")

molecules_raw.to_parquet(single_parquet_path, index=False)

print("Saved parquet successfully.")
print(f"Parquet file size: {single_parquet_path.stat().st_size / 1e9:.2f} GB")

# ------------------------------------------------------------
# 8. Save summary JSON
# ------------------------------------------------------------

summary_to_save = {
    "sample_name": SAMPLE_NAME,
    "method": "full_csv_loaded_at_once",
    "transcript_csv_path": str(COSMX_TRANSCRIPTS_PATH),
    "parquet_chunk_dir": str(PARQUET_CHUNK_DIR),
    "single_parquet_path": str(single_parquet_path),
    "n_parquet_chunks": 1,
    "total_rows_read": int(len(molecules_raw)),
    "assigned_rows": int(n_assigned),
    "unassigned_rows": int(n_unassigned),
    "n_unique_fovs": int(molecules_raw["fov"].nunique()),
    "unique_fovs": sorted([int(x) for x in molecules_raw["fov"].unique()]),
    "n_unique_genes": int(molecules_raw["gene_id"].nunique()),
    "none_gene_rows": int(n_none_gene),
    "unknown_gene_rows": int(n_unknown_gene),
    "nuclear_rows": int(n_nuclear),
    "membrane_rows": int(n_membrane),
    "cytoplasm_rows": int(n_cytoplasm),
    "extracellular_rows": int(n_extracellular),
    "example_genes": sorted(molecules_raw["gene_id"].astype(str).unique().tolist())[:50],
}

summary_path = WORK_DIR / "cosmx_cell2D_full_transcript_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary_to_save, f, indent=2)

# Keep variables for later cells.
COSMX_MOLECULE_PARQUET_CHUNK_DIR = PARQUET_CHUNK_DIR
COSMX_TRANSCRIPT_SUMMARY_PATH = summary_path

print("\n" + "=" * 80)
print("CELL 2D SUMMARY")
print("=" * 80)

print(f"Total rows read        : {summary_to_save['total_rows_read']:,}")
print(f"Assigned rows          : {summary_to_save['assigned_rows']:,}")
print(f"Unassigned rows        : {summary_to_save['unassigned_rows']:,}")
print(f"Unique FOVs            : {summary_to_save['n_unique_fovs']:,}")
print(f"Unique genes/targets   : {summary_to_save['n_unique_genes']:,}")
print(f"gene_id='None' rows    : {summary_to_save['none_gene_rows']:,}")
print(f"gene_id='Unknown' rows : {summary_to_save['unknown_gene_rows']:,}")
print(f"Parquet files saved    : {summary_to_save['n_parquet_chunks']:,}")
print(f"Summary saved to       : {summary_path}")

print("\nREPLACEMENT COSMX CELL 2D finished successfully.")
print("Next step: run Cell 3, Cell 4, Cell 4.5, then Replacement Cell 5.")

In [ ]:
# ============================================================
# COSMX CELL 3 — LOAD COSMX CELL × GENE EXPRESSION MATRIX
# Xenium equivalent:
#   cell_by_gene_adata = sc.read_10x_h5("/content/outs/cell_feature_matrix.h5")
#
# For CosMx:
#   We load Lung5_Rep1_exprMat_file.csv
#   Build an AnnData object:
#       cells × biological genes
#   Store negative-probe/control counts separately in obs for QC.
# ============================================================

import os
import gc
import json
import pandas as pd
import numpy as np
import anndata as ad
from scipy import sparse
from pathlib import Path

print("=" * 80)
print("COSMX CELL 3 — LOAD COSMX CELL × GENE EXPRESSION MATRIX")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check expression matrix path
# ------------------------------------------------------------

COSMX_EXPR_PATH = Path(COSMX_EXPR_PATH)

if not COSMX_EXPR_PATH.exists():
    raise FileNotFoundError(f"Expression matrix file not found: {COSMX_EXPR_PATH}")

print(f"Expression matrix file: {COSMX_EXPR_PATH}")
print(f"File size             : {COSMX_EXPR_PATH.stat().st_size / 1e6:.2f} MB")

# ------------------------------------------------------------
# 2. Read a small preview first
# ------------------------------------------------------------

print("\nReading first 5 rows for inspection...")
expr_preview = pd.read_csv(COSMX_EXPR_PATH, nrows=5)

print("\nExpression matrix columns:")
print(list(expr_preview.columns[:30]))
print(f"Total columns in preview: {len(expr_preview.columns):,}")

print("\nPreview:")
display(expr_preview.head())

# ------------------------------------------------------------
# 3. Verify required columns
# ------------------------------------------------------------

required_expr_cols = ["fov", "cell_ID"]
missing_expr_cols = [c for c in required_expr_cols if c not in expr_preview.columns]

if missing_expr_cols:
    raise KeyError(
        f"Missing required expression-matrix columns: {missing_expr_cols}\n"
        f"Available columns: {list(expr_preview.columns[:50])}"
    )

print("\nRequired expression matrix columns are present.")

# ------------------------------------------------------------
# 4. Load full expression matrix
# ------------------------------------------------------------
# This file is about 198 MB, so it should be manageable in Colab.

print("\nLoading full expression matrix...")
expr_df = pd.read_csv(COSMX_EXPR_PATH)

print("Loaded expr_df.")
print(f"expr_df shape : {expr_df.shape}")
print(f"Memory usage  : {expr_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# ------------------------------------------------------------
# 5. Create globally unique CosMx cell IDs
# ------------------------------------------------------------
# CosMx cell_ID is only unique inside each FOV.
# So the true cell key must be fov + cell_ID.

expr_df["cell_id"] = (
    expr_df["fov"].astype(str)
    + "_"
    + expr_df["cell_ID"].astype(str)
)

print("\nCell ID diagnostics:")
print(f"Rows in expression matrix     : {len(expr_df):,}")
print(f"Unique fov values             : {expr_df['fov'].nunique():,}")
print(f"Unique raw cell_ID values     : {expr_df['cell_ID'].nunique():,}")
print(f"Unique global cell_id values  : {expr_df['cell_id'].nunique():,}")

# Check whether expression matrix includes cell_ID 0
n_cell0 = int((expr_df["cell_ID"] == 0).sum())
print(f"Rows with cell_ID == 0        : {n_cell0:,}")

# ------------------------------------------------------------
# 6. Detect gene/count columns
# ------------------------------------------------------------

metadata_cols = {"fov", "cell_ID", "cell_id"}

all_count_cols = [c for c in expr_df.columns if c not in metadata_cols]

# CosMx expression matrix includes biological targets and negative probes.
# Negative probes are useful QC/background controls, but should not be treated
# as biological genes for denoising/reference overlap.
def is_negative_or_control(col):
    c = str(col).lower()
    return (
        c.startswith("negprb")
        or c.startswith("negative")
        or "negative" in c
        or "control" in c
        or c.startswith("blank")
        or c.startswith("systemcontrol")
    )

negative_probe_cols = [c for c in all_count_cols if is_negative_or_control(c)]
bio_gene_cols = [c for c in all_count_cols if c not in negative_probe_cols]

print("\nColumn classification:")
print(f"All count columns             : {len(all_count_cols):,}")
print(f"Biological gene columns       : {len(bio_gene_cols):,}")
print(f"Negative/control columns      : {len(negative_probe_cols):,}")

print("\nFirst 30 biological genes:")
print(bio_gene_cols[:30])

print("\nNegative/control columns:")
print(negative_probe_cols[:30])

if len(bio_gene_cols) == 0:
    raise ValueError("No biological gene columns detected in expression matrix.")

# ------------------------------------------------------------
# 7. Build AnnData object for biological genes only
# ------------------------------------------------------------

print("\nBuilding AnnData object using biological genes only...")

# Make sure counts are numeric.
X_counts = (
    expr_df[bio_gene_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .to_numpy(dtype=np.float32)
)

cell_by_gene_adata = ad.AnnData(X=sparse.csr_matrix(X_counts))
cell_by_gene_adata.obs_names = pd.Index(expr_df["cell_id"].astype(str), name="cell_id")
cell_by_gene_adata.var_names = pd.Index([str(g) for g in bio_gene_cols], name="gene_id")
cell_by_gene_adata.var_names_make_unique()

# Basic obs metadata
cell_by_gene_adata.obs["fov"] = expr_df["fov"].astype(str).values
cell_by_gene_adata.obs["cell_ID_original"] = expr_df["cell_ID"].astype(str).values

# Store raw counts
cell_by_gene_adata.layers["raw"] = cell_by_gene_adata.X.copy()

# ------------------------------------------------------------
# 8. Store negative probe/control count summaries in obs
# ------------------------------------------------------------

if len(negative_probe_cols) > 0:
    neg_counts = (
        expr_df[negative_probe_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .to_numpy(dtype=np.float32)
    )

    cell_by_gene_adata.obs["neg_probe_total_counts"] = neg_counts.sum(axis=1)
    cell_by_gene_adata.obs["neg_probe_detected_features"] = (neg_counts > 0).sum(axis=1)

    # Save negative probe names separately
    cell_by_gene_adata.uns["negative_probe_columns"] = negative_probe_cols
else:
    cell_by_gene_adata.obs["neg_probe_total_counts"] = 0.0
    cell_by_gene_adata.obs["neg_probe_detected_features"] = 0
    cell_by_gene_adata.uns["negative_probe_columns"] = []

# ------------------------------------------------------------
# 9. Add standard QC summaries
# ------------------------------------------------------------

total_counts = np.asarray(cell_by_gene_adata.X.sum(axis=1)).ravel()
detected_genes = np.asarray((cell_by_gene_adata.X > 0).sum(axis=1)).ravel()

cell_by_gene_adata.obs["total_counts"] = total_counts
cell_by_gene_adata.obs["detected_genes"] = detected_genes

gene_total_counts = np.asarray(cell_by_gene_adata.X.sum(axis=0)).ravel()
gene_detected_cells = np.asarray((cell_by_gene_adata.X > 0).sum(axis=0)).ravel()

cell_by_gene_adata.var["total_counts"] = gene_total_counts
cell_by_gene_adata.var["detected_cells"] = gene_detected_cells

# ------------------------------------------------------------
# 10. Print summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COSMX SPATIAL AnnData SUMMARY")
print("=" * 80)

print(cell_by_gene_adata)

print("\nMatrix diagnostics:")
print(f"Cells                         : {cell_by_gene_adata.n_obs:,}")
print(f"Biological genes              : {cell_by_gene_adata.n_vars:,}")
print(f"Total biological counts       : {cell_by_gene_adata.X.sum():,.0f}")
print(f"Mean counts per cell          : {total_counts.mean():,.2f}")
print(f"Median counts per cell        : {np.median(total_counts):,.2f}")
print(f"Mean detected genes per cell  : {detected_genes.mean():,.2f}")
print(f"Median detected genes per cell: {np.median(detected_genes):,.2f}")

print("\nTop 20 genes by total counts:")
top_gene_df = pd.DataFrame({
    "gene": cell_by_gene_adata.var_names,
    "total_counts": gene_total_counts,
    "detected_cells": gene_detected_cells,
})
display(top_gene_df.sort_values("total_counts", ascending=False).head(20))

print("\nFirst 10 cell IDs:")
print(cell_by_gene_adata.obs_names[:10].tolist())

print("\nFirst 20 genes:")
print(cell_by_gene_adata.var_names[:20].tolist())

print("\nobs preview:")
display(cell_by_gene_adata.obs.head())

# ------------------------------------------------------------
# 11. Save a checkpoint
# ------------------------------------------------------------

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

adata_path = WORK_DIR / "cosmx_cell3_raw_cell_by_gene_adata.h5ad"

print(f"\nSaving raw CosMx AnnData checkpoint to:")
print(adata_path)

cell_by_gene_adata.write_h5ad(adata_path)

# Save metadata about selected columns
cell3_summary = {
    "sample_name": SAMPLE_NAME,
    "expr_csv_path": str(COSMX_EXPR_PATH),
    "n_cells": int(cell_by_gene_adata.n_obs),
    "n_biological_genes": int(cell_by_gene_adata.n_vars),
    "n_all_count_columns": int(len(all_count_cols)),
    "n_negative_control_columns": int(len(negative_probe_cols)),
    "biological_gene_cols": list(map(str, bio_gene_cols)),
    "negative_probe_cols": list(map(str, negative_probe_cols)),
    "total_biological_counts": float(cell_by_gene_adata.X.sum()),
}

summary_path = WORK_DIR / "cosmx_cell3_expression_summary.json"

with open(summary_path, "w") as f:
    json.dump(cell3_summary, f, indent=2)

print(f"Saved Cell 3 summary to:")
print(summary_path)

# Clean memory
del X_counts
gc.collect()

print("\nCOSMX CELL 3 finished successfully.")
print("Next step: load cell metadata and merge it into AnnData.obs.")

In [ ]:
# ============================================================
# COSMX CELL 4 — LOAD CELL METADATA + MERGE INTO AnnData + REMOVE UNASSIGNED PSEUDO-CELLS
# Xenium equivalent:
#   - Load cells.csv.gz
#   - Add cell centroid/area metadata
#   - Keep valid segmented cells only
#
# CosMx-specific:
#   - metadata_file.csv contains real segmented cells
#   - cell_ID = 0 rows are not real cells, so remove them
# ============================================================

import os
import gc
import json
import pandas as pd
import numpy as np
import anndata as ad
from pathlib import Path

print("=" * 80)
print("COSMX CELL 4 — LOAD METADATA + MERGE + REMOVE cell_ID=0 PSEUDO-CELLS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check metadata path
# ------------------------------------------------------------

COSMX_METADATA_PATH = Path(COSMX_METADATA_PATH)

if not COSMX_METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata file not found: {COSMX_METADATA_PATH}")

print(f"Metadata file: {COSMX_METADATA_PATH}")
print(f"File size    : {COSMX_METADATA_PATH.stat().st_size / 1e6:.2f} MB")

# ------------------------------------------------------------
# 2. Load metadata
# ------------------------------------------------------------

metadata_df = pd.read_csv(COSMX_METADATA_PATH)

print("\nLoaded metadata_df.")
print(f"metadata_df shape: {metadata_df.shape}")
print("\nMetadata columns:")
print(list(metadata_df.columns))

print("\nMetadata preview:")
display(metadata_df.head())

# ------------------------------------------------------------
# 3. Verify required columns
# ------------------------------------------------------------

required_meta_cols = [
    "fov",
    "cell_ID",
    "Area",
    "CenterX_local_px",
    "CenterY_local_px",
    "CenterX_global_px",
    "CenterY_global_px",
    "Width",
    "Height",
]

missing_meta_cols = [c for c in required_meta_cols if c not in metadata_df.columns]

if missing_meta_cols:
    raise KeyError(
        f"Missing required metadata columns: {missing_meta_cols}\n"
        f"Available columns: {list(metadata_df.columns)}"
    )

print("\nAll required metadata columns are present.")

# ------------------------------------------------------------
# 4. Create globally unique cell_id in metadata
# ------------------------------------------------------------

metadata_df["cell_id"] = (
    metadata_df["fov"].astype(str)
    + "_"
    + metadata_df["cell_ID"].astype(str)
)

print("\nMetadata cell diagnostics:")
print(f"Rows in metadata                 : {len(metadata_df):,}")
print(f"Unique global cell_id in metadata: {metadata_df['cell_id'].nunique():,}")
print(f"Rows with cell_ID == 0 in metadata: {(metadata_df['cell_ID'] == 0).sum():,}")

# ------------------------------------------------------------
# 5. Check current AnnData from Cell 3
# ------------------------------------------------------------

if "cell_by_gene_adata" not in globals():
    raise NameError("cell_by_gene_adata not found. Run COSMX CELL 3 first.")

print("\nCurrent AnnData before metadata merge:")
print(cell_by_gene_adata)

adata_cell_ids = set(cell_by_gene_adata.obs_names.astype(str))
metadata_cell_ids = set(metadata_df["cell_id"].astype(str))

common_cells = sorted(list(adata_cell_ids.intersection(metadata_cell_ids)))

print("\nCell matching diagnostics:")
print(f"Cells in AnnData expression matrix : {len(adata_cell_ids):,}")
print(f"Cells in metadata                  : {len(metadata_cell_ids):,}")
print(f"Shared real cells                  : {len(common_cells):,}")
print(f"AnnData-only cells                 : {len(adata_cell_ids - metadata_cell_ids):,}")
print(f"Metadata-only cells                : {len(metadata_cell_ids - adata_cell_ids):,}")

print("\nFirst 20 AnnData-only cells:")
print(sorted(list(adata_cell_ids - metadata_cell_ids))[:20])

# ------------------------------------------------------------
# 6. Remove cell_ID == 0 pseudo-cells from AnnData
# ------------------------------------------------------------
# In CosMx, cell_ID=0 means transcripts not assigned to cells.
# The expression matrix has one cell_ID=0 row per FOV.
# These are not biological segmented cells, so we remove them.

cell_id_original_str = cell_by_gene_adata.obs["cell_ID_original"].astype(str)
is_real_cell = ~cell_id_original_str.isin(["0", "0.0", "nan", "None", ""])

n_before = cell_by_gene_adata.n_obs
n_pseudo = int((~is_real_cell).sum())

print("\nRemoving pseudo-cells:")
print(f"Cells before removal       : {n_before:,}")
print(f"cell_ID=0 pseudo-cell rows : {n_pseudo:,}")

# Keep a backup reference name in case we need it later
cell_by_gene_adata_with_unassigned = cell_by_gene_adata.copy()

cell_by_gene_adata = cell_by_gene_adata[is_real_cell.values, :].copy()

print(f"Cells after removal        : {cell_by_gene_adata.n_obs:,}")

# ------------------------------------------------------------
# 7. Merge metadata into AnnData.obs
# ------------------------------------------------------------

metadata_join = metadata_df.set_index("cell_id")

common_cells_after = cell_by_gene_adata.obs_names.intersection(metadata_join.index)

print("\nMerging metadata:")
print(f"Real AnnData cells after removing cell_ID=0: {cell_by_gene_adata.n_obs:,}")
print(f"Cells with matching metadata             : {len(common_cells_after):,}")

if len(common_cells_after) != cell_by_gene_adata.n_obs:
    print("WARNING: Some AnnData cells do not have metadata.")
    missing_meta = cell_by_gene_adata.obs_names.difference(metadata_join.index)
    print("First 20 missing metadata cells:")
    print(missing_meta[:20].tolist())

# Add metadata columns into obs
for col in metadata_join.columns:
    # Avoid overwriting existing columns unless values are useful
    new_col = f"metadata_{col}" if col in cell_by_gene_adata.obs.columns else col
    cell_by_gene_adata.obs[new_col] = np.nan

    # Assign values only for matched cells
    cell_by_gene_adata.obs.loc[common_cells_after, new_col] = metadata_join.loc[common_cells_after, col].values

# ------------------------------------------------------------
# 8. Add micron-scaled coordinate/geometry columns
# ------------------------------------------------------------
# README says pixel edge length is 180 nm = 0.18 microns.
# We keep original pixel coordinates, and also add micron versions.

PIXEL_SIZE_UM = 0.18

# Make sure numeric columns are numeric
numeric_cols = [
    "Area",
    "CenterX_local_px",
    "CenterY_local_px",
    "CenterX_global_px",
    "CenterY_global_px",
    "Width",
    "Height",
]

for col in numeric_cols:
    if col in cell_by_gene_adata.obs.columns:
        cell_by_gene_adata.obs[col] = pd.to_numeric(cell_by_gene_adata.obs[col], errors="coerce")

# Pixel to micron conversions
cell_by_gene_adata.obs["cell_area_px"] = cell_by_gene_adata.obs["Area"].astype(float)
cell_by_gene_adata.obs["cell_area_um2"] = cell_by_gene_adata.obs["cell_area_px"] * (PIXEL_SIZE_UM ** 2)

cell_by_gene_adata.obs["center_x_global_px"] = cell_by_gene_adata.obs["CenterX_global_px"].astype(float)
cell_by_gene_adata.obs["center_y_global_px"] = cell_by_gene_adata.obs["CenterY_global_px"].astype(float)

cell_by_gene_adata.obs["center_x_um"] = cell_by_gene_adata.obs["center_x_global_px"] * PIXEL_SIZE_UM
cell_by_gene_adata.obs["center_y_um"] = cell_by_gene_adata.obs["center_y_global_px"] * PIXEL_SIZE_UM

cell_by_gene_adata.obs["width_px"] = cell_by_gene_adata.obs["Width"].astype(float)
cell_by_gene_adata.obs["height_px"] = cell_by_gene_adata.obs["Height"].astype(float)

cell_by_gene_adata.obs["width_um"] = cell_by_gene_adata.obs["width_px"] * PIXEL_SIZE_UM
cell_by_gene_adata.obs["height_um"] = cell_by_gene_adata.obs["height_px"] * PIXEL_SIZE_UM

# ------------------------------------------------------------
# 9. Recompute QC after removing cell_ID=0 pseudo-cells
# ------------------------------------------------------------

from scipy import sparse

total_counts = np.asarray(cell_by_gene_adata.X.sum(axis=1)).ravel()
detected_genes = np.asarray((cell_by_gene_adata.X > 0).sum(axis=1)).ravel()

cell_by_gene_adata.obs["total_counts"] = total_counts
cell_by_gene_adata.obs["detected_genes"] = detected_genes

gene_total_counts = np.asarray(cell_by_gene_adata.X.sum(axis=0)).ravel()
gene_detected_cells = np.asarray((cell_by_gene_adata.X > 0).sum(axis=0)).ravel()

cell_by_gene_adata.var["total_counts"] = gene_total_counts
cell_by_gene_adata.var["detected_cells"] = gene_detected_cells

# ------------------------------------------------------------
# 10. Print summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COSMX AnnData AFTER METADATA MERGE")
print("=" * 80)

print(cell_by_gene_adata)

print("\nCell-level summary after removing pseudo-cells:")
print(f"Real cells                         : {cell_by_gene_adata.n_obs:,}")
print(f"Genes                              : {cell_by_gene_adata.n_vars:,}")
print(f"Total biological counts            : {cell_by_gene_adata.X.sum():,.0f}")
print(f"Mean counts per real cell          : {total_counts.mean():,.2f}")
print(f"Median counts per real cell        : {np.median(total_counts):,.2f}")
print(f"Mean detected genes per real cell  : {detected_genes.mean():,.2f}")
print(f"Median detected genes per real cell: {np.median(detected_genes):,.2f}")

print("\nGeometry summary:")
geom_cols = [
    "cell_area_px",
    "cell_area_um2",
    "center_x_global_px",
    "center_y_global_px",
    "width_px",
    "height_px",
    "Mean.DAPI",
    "Max.DAPI",
]

available_geom_cols = [c for c in geom_cols if c in cell_by_gene_adata.obs.columns]
display(cell_by_gene_adata.obs[available_geom_cols].describe().T)

print("\nobs preview:")
display(cell_by_gene_adata.obs.head())

# ------------------------------------------------------------
# 11. Save checkpoint
# ------------------------------------------------------------

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

adata_path = WORK_DIR / "cosmx_cell4_spatial_adata_with_metadata_real_cells.h5ad"
metadata_clean_path = WORK_DIR / "cosmx_cell4_metadata_with_global_cell_id.csv"
summary_path = WORK_DIR / "cosmx_cell4_metadata_merge_summary.json"

print("\nSaving Cell 4 outputs...")

cell_by_gene_adata.write_h5ad(adata_path)
metadata_df.to_csv(metadata_clean_path, index=False)

cell4_summary = {
    "sample_name": SAMPLE_NAME,
    "metadata_path": str(COSMX_METADATA_PATH),
    "n_metadata_rows": int(len(metadata_df)),
    "n_metadata_unique_cell_ids": int(metadata_df["cell_id"].nunique()),
    "n_cells_before_removing_cell_id_0": int(n_before),
    "n_cell_id_0_pseudo_cells_removed": int(n_pseudo),
    "n_real_cells_after_removal": int(cell_by_gene_adata.n_obs),
    "n_genes": int(cell_by_gene_adata.n_vars),
    "total_biological_counts_after_removal": float(cell_by_gene_adata.X.sum()),
    "pixel_size_um": float(PIXEL_SIZE_UM),
    "n_cells_with_matching_metadata": int(len(common_cells_after)),
}

with open(summary_path, "w") as f:
    json.dump(cell4_summary, f, indent=2)

print(f"Saved AnnData:")
print(f"  {adata_path}")

print(f"Saved metadata with global cell_id:")
print(f"  {metadata_clean_path}")

print(f"Saved summary:")
print(f"  {summary_path}")

gc.collect()

print("\nCOSMX CELL 4 finished successfully.")
print("Next step: clean molecule table and verify molecule-derived counts against the expression matrix.")

In [ ]:
# ============================================================
# COSMX CELL 4.5 — DOWNLOAD GSE131907 REFERENCE + CHECK GENE OVERLAP
# Purpose:
#   Before cleaning molecule counts, check whether CosMx Lung5_Rep1
#   is pairable with the GSE131907 scRNA-seq reference.
#
# What this cell downloads:
#   1. GSE131907_Lung_Cancer_cell_annotation.txt.gz
#   2. GSE131907_Lung_Cancer_raw_UMI_matrix.txt.gz
#
# Important:
#   This cell does NOT load the full raw UMI matrix into memory.
#   It only extracts the reference gene names for overlap checking.
# ============================================================

import os
import gzip
import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 80)
print("COSMX CELL 4.5 — DOWNLOAD GSE131907 REFERENCE + CHECK GENE OVERLAP")
print("=" * 80)

# ------------------------------------------------------------
# 1. Setup GSE131907 download folder
# ------------------------------------------------------------

GSE_ID = "GSE131907"

# SCRNA_ROOT was defined in COSMX CELL 0.
# If it is missing for any reason, recreate it.
if "SCRNA_ROOT" not in globals():
    SCRNA_ROOT = Path("/content/gse131907")
else:
    SCRNA_ROOT = Path(SCRNA_ROOT)

SCRNA_ROOT.mkdir(parents=True, exist_ok=True)

GSE_DIR = SCRNA_ROOT / GSE_ID
GSE_DIR.mkdir(parents=True, exist_ok=True)

print(f"GSE directory: {GSE_DIR}")

# ------------------------------------------------------------
# 2. Define public GEO supplementary file URLs
# ------------------------------------------------------------

BASE_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl"

GSE_CELL_ANNOT_URL = (
    f"{BASE_URL}/GSE131907_Lung_Cancer_cell_annotation.txt.gz"
)

GSE_RAW_UMI_URL = (
    f"{BASE_URL}/GSE131907_Lung_Cancer_raw_UMI_matrix.txt.gz"
)

GSE_FEATURE_SUMMARY_URL = (
    f"{BASE_URL}/GSE131907_Lung_Cancer_Feature_Summary.xlsx"
)

SCRNA_ANNOT_PATH = GSE_DIR / "GSE131907_Lung_Cancer_cell_annotation.txt.gz"
SCRNA_RAW_UMI_PATH = GSE_DIR / "GSE131907_Lung_Cancer_raw_UMI_matrix.txt.gz"
SCRNA_FEATURE_SUMMARY_PATH = GSE_DIR / "GSE131907_Lung_Cancer_Feature_Summary.xlsx"

files_to_download = [
    ("cell annotation", GSE_CELL_ANNOT_URL, SCRNA_ANNOT_PATH),
    ("raw UMI matrix", GSE_RAW_UMI_URL, SCRNA_RAW_UMI_PATH),
    ("feature summary", GSE_FEATURE_SUMMARY_URL, SCRNA_FEATURE_SUMMARY_PATH),
]

# ------------------------------------------------------------
# 3. Download helper
# ------------------------------------------------------------

def download_if_missing(url, out_path):
    out_path = Path(out_path)

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"Already exists, skipping: {out_path.name}")
        print(f"  size: {out_path.stat().st_size / 1e6:.2f} MB")
        return

    print(f"\nDownloading: {out_path.name}")
    print(f"Target path: {out_path}")
    urllib.request.urlretrieve(url, out_path)
    print(f"Downloaded: {out_path.name}")
    print(f"  size: {out_path.stat().st_size / 1e6:.2f} MB")

# ------------------------------------------------------------
# 4. Download files
# ------------------------------------------------------------

print("\nDownloading/checking required GSE131907 files...")
for label, url, path in files_to_download:
    print(f"\nFile type: {label}")
    download_if_missing(url, path)

# ------------------------------------------------------------
# 5. Load small cell annotation file
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD GSE131907 CELL ANNOTATION")
print("=" * 80)

scrna_annot = pd.read_csv(SCRNA_ANNOT_PATH, sep="\t", compression="gzip")

print(f"scrna_annot shape: {scrna_annot.shape}")
print("Annotation columns:")
print(list(scrna_annot.columns))
print("\nAnnotation preview:")
display(scrna_annot.head())

print("\nPossible cell-type / annotation columns:")
for col in scrna_annot.columns:
    col_lower = col.lower()
    if any(key in col_lower for key in ["cell", "type", "cluster", "anno", "major", "sub"]):
        print(f"\nColumn: {col}")
        print(f"Unique values: {scrna_annot[col].nunique(dropna=True):,}")
        display(scrna_annot[col].value_counts(dropna=False).head(20))

# ------------------------------------------------------------
# 6. Extract scRNA reference gene names without loading full matrix
# ------------------------------------------------------------
# The raw UMI matrix is large.
# We only need the gene names for now.
#
# Expected format:
#   first column = gene name
#   remaining columns = single-cell barcodes
#
# We stream through the gzip file line by line and collect the first field.

print("\n" + "=" * 80)
print("EXTRACT GENE NAMES FROM GSE131907 RAW UMI MATRIX")
print("=" * 80)

def sniff_delimiter_from_gzip(gz_path):
    with gzip.open(gz_path, "rt") as f:
        header = f.readline()
    if "\t" in header:
        return "\t"
    elif "," in header:
        return ","
    else:
        return None

delimiter = sniff_delimiter_from_gzip(SCRNA_RAW_UMI_PATH)
print(f"Detected delimiter: {repr(delimiter)}")

if delimiter is None:
    raise ValueError("Could not detect delimiter in raw UMI matrix header.")

# Read first few lines for display
print("\nFirst 3 lines of raw UMI matrix:")
with gzip.open(SCRNA_RAW_UMI_PATH, "rt") as f:
    for i in range(3):
        line = f.readline()
        print(line[:500] + ("..." if len(line) > 500 else ""))

# Stream gene names
scrna_genes = []
n_lines = 0

with gzip.open(SCRNA_RAW_UMI_PATH, "rt") as f:
    header = f.readline()
    header_parts = header.rstrip("\n").split(delimiter)

    print("\nHeader diagnostics:")
    print(f"Number of header columns: {len(header_parts):,}")
    print("First 10 header fields:")
    print(header_parts[:10])

    for line in f:
        n_lines += 1
        if not line.strip():
            continue
        gene = line.split(delimiter, 1)[0].strip().strip('"')
        if gene:
            scrna_genes.append(gene)

scrna_genes = pd.Index(scrna_genes.astype(str) if hasattr(scrna_genes, "astype") else [str(g) for g in scrna_genes])
scrna_genes = pd.Index(scrna_genes).drop_duplicates()

print("\nReference gene extraction complete.")
print(f"Raw matrix data lines read: {n_lines:,}")
print(f"Unique GSE131907 genes    : {len(scrna_genes):,}")
print("First 30 reference genes:")
print(scrna_genes[:30].tolist())

# ------------------------------------------------------------
# 7. Get CosMx biological genes from current AnnData
# ------------------------------------------------------------

if "cell_by_gene_adata" not in globals():
    raise NameError("cell_by_gene_adata not found. Run COSMX CELL 3 and CELL 4 first.")

cosmx_genes = pd.Index(cell_by_gene_adata.var_names.astype(str)).drop_duplicates()

print("\n" + "=" * 80)
print("COSMX PANEL GENES")
print("=" * 80)

print(f"CosMx biological genes: {len(cosmx_genes):,}")
print("First 30 CosMx genes:")
print(cosmx_genes[:30].tolist())

# ------------------------------------------------------------
# 8. Direct gene overlap check
# ------------------------------------------------------------

cosmx_gene_set = set(cosmx_genes.astype(str))
scrna_gene_set = set(scrna_genes.astype(str))

shared_genes = [g for g in cosmx_genes if g in scrna_gene_set]
missing_from_scrna = [g for g in cosmx_genes if g not in scrna_gene_set]

overlap_pct = 100 * len(shared_genes) / max(len(cosmx_genes), 1)

print("\n" + "=" * 80)
print("GENE OVERLAP: COSMX Lung5_Rep1 vs GSE131907")
print("=" * 80)

print(f"CosMx biological genes       : {len(cosmx_genes):,}")
print(f"GSE131907 reference genes    : {len(scrna_genes):,}")
print(f"Shared genes                 : {len(shared_genes):,}")
print(f"CosMx panel covered by ref   : {overlap_pct:.2f}%")

print("\nFirst 50 shared genes:")
print(shared_genes[:50])

print("\nCosMx genes missing from GSE131907:")
print(f"Missing count: {len(missing_from_scrna):,}")
print(missing_from_scrna[:100])

# ------------------------------------------------------------
# 9. Case-insensitive rescue check
# ------------------------------------------------------------
# Sometimes gene symbols differ only by capitalization.
# This checks whether missing genes can be rescued by case-insensitive matching.

scrna_upper_to_original = {}
for g in scrna_genes:
    gu = str(g).upper()
    if gu not in scrna_upper_to_original:
        scrna_upper_to_original[gu] = str(g)

case_rescued = []
still_missing = []

for g in missing_from_scrna:
    gu = str(g).upper()
    if gu in scrna_upper_to_original:
        case_rescued.append((g, scrna_upper_to_original[gu]))
    else:
        still_missing.append(g)

print("\nCase-insensitive rescue check:")
print(f"Could rescue by capitalization only: {len(case_rescued):,}")
print(case_rescued[:50])
print(f"Still missing after case rescue    : {len(still_missing):,}")
print(still_missing[:100])

# If capitalization rescue is useful, create mapping.
gene_symbol_mapping = {g: g for g in shared_genes}
for cos_g, ref_g in case_rescued:
    gene_symbol_mapping[cos_g] = ref_g

shared_genes_mapped = [g for g in cosmx_genes if g in gene_symbol_mapping]

print("\nFinal usable shared genes after optional case rescue:")
print(f"Usable shared CosMx genes: {len(shared_genes_mapped):,}")
print(f"Coverage after rescue    : {100 * len(shared_genes_mapped) / len(cosmx_genes):.2f}%")

# ------------------------------------------------------------
# 10. Pairability verdict
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PAIRABILITY VERDICT")
print("=" * 80)

if len(shared_genes_mapped) >= 0.80 * len(cosmx_genes):
    verdict = "STRONG"
    explanation = (
        "Gene overlap is high. This pair is suitable for reference-guided "
        "cell-level denoising and downstream label transfer."
    )
elif len(shared_genes_mapped) >= 0.60 * len(cosmx_genes):
    verdict = "MODERATE"
    explanation = (
        "Gene overlap is usable but not ideal. The pair may still work, but "
        "missing genes should be inspected."
    )
else:
    verdict = "WEAK"
    explanation = (
        "Gene overlap is low. This reference may not be suitable without "
        "gene-symbol correction or another reference."
    )

print(f"Verdict: {verdict}")
print(explanation)

# ------------------------------------------------------------
# 11. Save overlap summary
# ------------------------------------------------------------

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

overlap_summary = {
    "sample_name": SAMPLE_NAME,
    "reference": "GSE131907",
    "scrna_annotation_path": str(SCRNA_ANNOT_PATH),
    "scrna_raw_umi_path": str(SCRNA_RAW_UMI_PATH),
    "n_cosmx_genes": int(len(cosmx_genes)),
    "n_scrna_genes": int(len(scrna_genes)),
    "n_shared_genes_direct": int(len(shared_genes)),
    "n_missing_from_scrna_direct": int(len(missing_from_scrna)),
    "n_case_rescued": int(len(case_rescued)),
    "n_shared_genes_after_case_rescue": int(len(shared_genes_mapped)),
    "coverage_direct_pct": float(overlap_pct),
    "coverage_after_case_rescue_pct": float(100 * len(shared_genes_mapped) / len(cosmx_genes)),
    "verdict": verdict,
    "shared_genes_direct": list(map(str, shared_genes)),
    "missing_from_scrna_direct": list(map(str, missing_from_scrna)),
    "case_rescued_pairs": [(str(a), str(b)) for a, b in case_rescued],
    "shared_genes_mapped": list(map(str, shared_genes_mapped)),
    "gene_symbol_mapping": {str(k): str(v) for k, v in gene_symbol_mapping.items()},
}

overlap_json_path = WORK_DIR / "cosmx_gse131907_gene_overlap_summary.json"

with open(overlap_json_path, "w") as f:
    json.dump(overlap_summary, f, indent=2)

# Save shared genes as plain text too
shared_genes_txt_path = WORK_DIR / "cosmx_gse131907_shared_genes.txt"
with open(shared_genes_txt_path, "w") as f:
    for g in shared_genes_mapped:
        f.write(str(g) + "\n")

print("\nSaved overlap summary:")
print(f"  {overlap_json_path}")
print(f"  {shared_genes_txt_path}")

# Keep useful variables for later cells
SCRNA_ANNOT_PATH = SCRNA_ANNOT_PATH
SCRNA_RAW_UMI_PATH = SCRNA_RAW_UMI_PATH
SCRNA_GENES = scrna_genes
COSMX_GENES = cosmx_genes
SHARED_GENES_COSMX_GSE131907 = shared_genes_mapped
GENE_SYMBOL_MAPPING_COSMX_TO_GSE131907 = gene_symbol_mapping

print("\nCOSMX CELL 4.5 finished successfully.")
print("Next step can be either:")
print("  1. If overlap is strong/moderate: continue to Cell 5 clean molecule table.")
print("  2. If overlap is weak: inspect gene naming or choose another reference.")

In [ ]:
# ============================================================
# REPLACEMENT COSMX CELL 5 — CLEAN SHARED-GENE DATA SAFELY
# Run this AFTER:
#   Cell 2D  = corrected full transcript loading
#   Cell 3   = load CosMx expression matrix
#   Cell 4   = merge metadata and remove cell_ID=0 pseudo-cells
#   Cell 4.5 = download GSE131907 and compute 951 shared genes
#
# Main rule:
#   Keep ONLY the 951 CosMx genes shared with GSE131907.
#
# Important:
#   We remove unassigned transcripts where cell_ID_original == 0.
#   We remove transcripts from non-shared genes.
#   We remove transcripts from invalid cells.
#   We DO NOT remove CellComp == "0" if the transcript is assigned to a real cell.
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
from scipy import sparse
from pathlib import Path

print("=" * 80)
print("REPLACEMENT COSMX CELL 5 — CLEAN SHARED-GENE DATA SAFELY")
print("=" * 80)

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Check required variables
# ------------------------------------------------------------

required_vars = [
    "cell_by_gene_adata",
    "COSMX_MOLECULE_PARQUET_CHUNK_DIR",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Required variable missing: {v}")

# ------------------------------------------------------------
# 2. Load shared genes from Cell 4.5
# ------------------------------------------------------------

if "SHARED_GENES_COSMX_GSE131907" not in globals():
    shared_genes_path = WORK_DIR / "cosmx_gse131907_shared_genes.txt"

    if not shared_genes_path.exists():
        raise FileNotFoundError(
            "Shared genes file not found. Run COSMX Cell 4.5 first."
        )

    SHARED_GENES_COSMX_GSE131907 = [
        line.strip() for line in open(shared_genes_path) if line.strip()
    ]

shared_genes = [str(g).strip() for g in SHARED_GENES_COSMX_GSE131907]
shared_gene_set = set(shared_genes)

print(f"Shared genes to keep: {len(shared_genes):,}")
print("First 30 shared genes:")
print(shared_genes[:30])

if len(shared_genes) < 900:
    raise ValueError(
        f"Too few shared genes found: {len(shared_genes)}. "
        "Expected about 951. Stop and rerun Cell 4.5."
    )

# ------------------------------------------------------------
# 3. Safety check: Cell 4 AnnData should still have ~100k cells
# ------------------------------------------------------------

print("\nCurrent AnnData before Cell 5:")
print(cell_by_gene_adata)

if cell_by_gene_adata.n_obs < 90_000:
    raise ValueError(
        "cell_by_gene_adata has too few cells. "
        "It looks like a failed Cell 5 may have overwritten it. "
        "Reload/rerun Cell 4 before running this Replacement Cell 5."
    )

if cell_by_gene_adata.n_vars < 900:
    raise ValueError(
        "cell_by_gene_adata has too few genes before shared-gene subsetting. "
        "Expected the 960-gene CosMx matrix from Cell 4."
    )

# ------------------------------------------------------------
# 4. Subset AnnData to the 951 shared genes only
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SUBSET CELL × GENE MATRIX TO SHARED GENES ONLY")
print("=" * 80)

adata_gene_order = [str(g).strip() for g in cell_by_gene_adata.var_names]
shared_genes_in_adata_order = [g for g in adata_gene_order if g in shared_gene_set]

missing_shared_from_adata = [g for g in shared_genes if g not in set(adata_gene_order)]

print(f"Genes before subsetting          : {cell_by_gene_adata.n_vars:,}")
print(f"Shared genes found in AnnData    : {len(shared_genes_in_adata_order):,}")
print(f"Shared genes missing from AnnData: {len(missing_shared_from_adata):,}")

if missing_shared_from_adata:
    print("Missing shared genes:")
    print(missing_shared_from_adata[:50])

if len(shared_genes_in_adata_order) < 900:
    raise ValueError("Too few shared genes found in AnnData. Stop.")

# Subset to shared genes in CosMx AnnData order.
cell_by_gene_adata = cell_by_gene_adata[:, shared_genes_in_adata_order].copy()
cell_by_gene_adata.var_names = pd.Index(shared_genes_in_adata_order)

# Store official expression matrix shared-gene counts.
cell_by_gene_adata.layers["raw_expr_original_shared"] = cell_by_gene_adata.X.copy()

# Remove cells with zero official expression counts across shared genes.
cell_total_shared = np.asarray(cell_by_gene_adata.X.sum(axis=1)).ravel()
keep_nonzero_expr_cells = cell_total_shared > 0

n_cells_before_expr_filter = cell_by_gene_adata.n_obs
n_zero_expr_cells = int((~keep_nonzero_expr_cells).sum())

print(f"\nCells before zero-count expression filter : {n_cells_before_expr_filter:,}")
print(f"Zero shared-gene expression cells          : {n_zero_expr_cells:,}")

if n_zero_expr_cells > 0:
    cell_by_gene_adata = cell_by_gene_adata[keep_nonzero_expr_cells, :].copy()
    cell_by_gene_adata.layers["raw_expr_original_shared"] = cell_by_gene_adata.X.copy()

print("\nAfter shared-gene subsetting:")
print(cell_by_gene_adata)

expr_sum = float(cell_by_gene_adata.X.sum())
print(f"Official expression shared-gene sum: {expr_sum:,.0f}")

# ------------------------------------------------------------
# 5. Prepare molecule parquet files
# ------------------------------------------------------------

COSMX_MOLECULE_PARQUET_CHUNK_DIR = Path(COSMX_MOLECULE_PARQUET_CHUNK_DIR)

if not COSMX_MOLECULE_PARQUET_CHUNK_DIR.exists():
    raise FileNotFoundError(
        f"Molecule parquet folder not found: {COSMX_MOLECULE_PARQUET_CHUNK_DIR}"
    )

chunk_paths = sorted(COSMX_MOLECULE_PARQUET_CHUNK_DIR.glob("molecules_chunk_*.parquet"))

if not chunk_paths:
    raise FileNotFoundError(
        "No molecule parquet files found. Run corrected Cell 2D first."
    )

print("\nMolecule parquet files found:")
for p in chunk_paths:
    print(f"  {p.name} ({p.stat().st_size / 1e9:.2f} GB)")

# Since Cell 2D creates one parquet file, this is usually 1.
# But this code also works if there are multiple files.
valid_cell_ids = set(cell_by_gene_adata.obs_names.astype(str))
gene_to_col = {g: j for j, g in enumerate(cell_by_gene_adata.var_names.astype(str))}
cell_to_row = {c: i for i, c in enumerate(cell_by_gene_adata.obs_names.astype(str))}

# ------------------------------------------------------------
# 6. Filter molecule table to:
#      assigned transcripts only
#      valid real cells only
#      951 shared genes only
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FILTER MOLECULE TABLE TO REAL CELLS + SHARED GENES")
print("=" * 80)

CLEAN_CHUNK_DIR = WORK_DIR / "clean_shared_molecule_parquet_chunks"
CLEAN_CHUNK_DIR.mkdir(parents=True, exist_ok=True)

# Remove old clean files if rerunning.
for old in CLEAN_CHUNK_DIR.glob("clean_molecules_chunk_*.parquet"):
    old.unlink()

summary = {
    "input_rows": 0,
    "kept_rows": 0,
    "removed_unassigned": 0,
    "removed_nonshared_gene": 0,
    "removed_invalid_cell": 0,
    "none_gene_rows_seen": 0,
    "unknown_gene_rows_seen": 0,
    "nuclear_rows": 0,
    "membrane_rows": 0,
    "cytoplasm_rows": 0,
    "extracellular_rows": 0,
}

all_row_idx = []
all_col_idx = []
all_counts = []
clean_chunk_paths = []

for k, chunk_path in enumerate(chunk_paths):
    print(f"\nProcessing molecule parquet {k + 1}/{len(chunk_paths)}: {chunk_path.name}")

    chunk = pd.read_parquet(chunk_path)

    summary["input_rows"] += len(chunk)

    # Normalize strings.
    chunk["gene_id"] = chunk["gene_id"].astype(str).str.strip()
    chunk["cell_id"] = chunk["cell_id"].astype(str).str.strip()
    chunk["CellComp"] = chunk["CellComp"].astype(str).str.strip()

    # Safety checks for the previous bug.
    gene_lower = chunk["gene_id"].str.lower()
    none_gene_rows = int(gene_lower.eq("none").sum())
    unknown_gene_rows = int(gene_lower.eq("unknown").sum())

    summary["none_gene_rows_seen"] += none_gene_rows
    summary["unknown_gene_rows_seen"] += unknown_gene_rows

    if none_gene_rows > 0:
        raise ValueError(
            f"Found {none_gene_rows:,} rows with gene_id='None' in {chunk_path.name}. "
            "This indicates corrupted molecule parquet conversion. "
            "Rerun Cell 2D and do not continue."
        )

    if unknown_gene_rows > 1000:
        raise ValueError(
            f"Found {unknown_gene_rows:,} rows with gene_id='Unknown'. "
            "This suggests gene names were not read correctly."
        )

    # Main filtering masks.
    mask_assigned = ~chunk["is_unassigned"].astype(bool)
    mask_gene = chunk["gene_id"].isin(shared_gene_set)
    mask_cell = chunk["cell_id"].isin(valid_cell_ids)

    summary["removed_unassigned"] += int((~mask_assigned).sum())
    summary["removed_nonshared_gene"] += int((mask_assigned & ~mask_gene).sum())
    summary["removed_invalid_cell"] += int((mask_assigned & mask_gene & ~mask_cell).sum())

    # IMPORTANT:
    # We do NOT remove CellComp == "0" here.
    # If a transcript is assigned to a real cell and belongs to a shared gene,
    # it stays in the clean molecule table.
    clean = chunk.loc[mask_assigned & mask_gene & mask_cell].copy()

    # Enforce clean datatypes.
    clean["cell_id"] = clean["cell_id"].astype(str)
    clean["gene_id"] = clean["gene_id"].astype(str)
    clean["fov"] = clean["fov"].astype(np.int32)
    clean["cell_ID_original"] = clean["cell_ID_original"].astype(np.int32)
    clean["overlaps_nucleus"] = clean["overlaps_nucleus"].astype(np.int8)
    clean["quality"] = clean["quality"].astype(np.float32)

    comp_lower = clean["CellComp"].astype(str).str.lower()

    summary["nuclear_rows"] += int(comp_lower.str.contains("nuclear", na=False).sum())
    summary["membrane_rows"] += int(comp_lower.str.contains("membrane", na=False).sum())
    summary["cytoplasm_rows"] += int(
        comp_lower.str.contains("cytoplasm|cytoplasmic", na=False).sum()
    )
    summary["extracellular_rows"] += int(
        (comp_lower.eq("0") | comp_lower.str.contains("extracellular", na=False)).sum()
    )

    summary["kept_rows"] += len(clean)

    print(f"  Input rows             : {len(chunk):,}")
    print(f"  Kept clean rows        : {len(clean):,}")
    print(f"  Removed unassigned     : {(~mask_assigned).sum():,}")
    print(f"  Removed non-shared gene: {(mask_assigned & ~mask_gene).sum():,}")
    print(f"  Removed invalid cell   : {(mask_assigned & mask_gene & ~mask_cell).sum():,}")

    # Save clean molecule table.
    clean_chunk_path = CLEAN_CHUNK_DIR / f"clean_molecules_chunk_{k:04d}.parquet"
    clean.to_parquet(clean_chunk_path, index=False)
    clean_chunk_paths.append(str(clean_chunk_path))

    # Build count entries from clean molecule rows.
    if len(clean) > 0:
        counts = (
            clean.groupby(["cell_id", "gene_id"])
            .size()
            .reset_index(name="count")
        )

        rows = counts["cell_id"].map(cell_to_row).to_numpy()
        cols = counts["gene_id"].map(gene_to_col).to_numpy()
        vals = counts["count"].to_numpy(dtype=np.int32)

        ok = (~pd.isna(rows)) & (~pd.isna(cols))

        if ok.sum() > 0:
            all_row_idx.append(rows[ok].astype(np.int32))
            all_col_idx.append(cols[ok].astype(np.int32))
            all_counts.append(vals[ok].astype(np.int32))

    del chunk, clean
    gc.collect()

print("\nFinished filtering molecule table.")

# ------------------------------------------------------------
# 7. Build molecule-derived raw count matrix
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BUILD MOLECULE-DERIVED RAW COUNT MATRIX")
print("=" * 80)

if all_row_idx:
    row_idx = np.concatenate(all_row_idx)
    col_idx = np.concatenate(all_col_idx)
    vals = np.concatenate(all_counts)
else:
    row_idx = np.array([], dtype=np.int32)
    col_idx = np.array([], dtype=np.int32)
    vals = np.array([], dtype=np.int32)

X_raw_counts_sparse = sparse.coo_matrix(
    (vals, (row_idx, col_idx)),
    shape=cell_by_gene_adata.shape,
    dtype=np.int32,
).tocsr()

X_raw_counts_sparse.sum_duplicates()

mol_sum = float(X_raw_counts_sparse.sum())

print(f"X_raw_counts_sparse shape: {X_raw_counts_sparse.shape}")
print(f"X_raw_counts_sparse sum  : {mol_sum:,.0f}")

# ------------------------------------------------------------
# 8. Verify molecule-derived counts against official expression matrix
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VERIFY MOLECULE-DERIVED COUNTS VS OFFICIAL EXPRESSION MATRIX")
print("=" * 80)

X_expr = (
    cell_by_gene_adata.X.tocsr()
    if sparse.issparse(cell_by_gene_adata.X)
    else sparse.csr_matrix(cell_by_gene_adata.X)
)

diff = X_expr.astype(np.float64) - X_raw_counts_sparse.astype(np.float64)
diff.eliminate_zeros()

abs_diff_data = np.abs(diff.data)

max_abs_diff = float(abs_diff_data.max()) if abs_diff_data.size > 0 else 0.0
n_mismatched_pairs = int(abs_diff_data.size)
ratio = mol_sum / max(expr_sum, 1.0)

print(f"Official expression shared-gene sum : {expr_sum:,.0f}")
print(f"Molecule-derived shared-gene sum    : {mol_sum:,.0f}")
print(f"Difference expr - molecule          : {expr_sum - mol_sum:,.0f}")
print(f"Ratio molecule / expression         : {ratio:.6f}")
print(f"Max abs diff per cell-gene pair     : {max_abs_diff:.6f}")
print(f"Mismatched cell-gene pairs          : {n_mismatched_pairs:,}")

# Strong safety check.
# The ratio should be close to 1.0. It may not be exactly 1.0 due to export rules,
# but it must not be extremely low.
if ratio < 0.90:
    raise ValueError(
        "\nCRITICAL ERROR: Molecule-derived counts are far below expression matrix counts.\n"
        f"Official expression shared sum : {expr_sum:,.0f}\n"
        f"Molecule-derived shared sum    : {mol_sum:,.0f}\n"
        f"Ratio molecule/expression      : {ratio:.4f}\n\n"
        "Do NOT continue. This means molecule filtering or gene/cell matching is still wrong."
    )

COUNTS_MATCH = bool(max_abs_diff < 1e-6 and abs(expr_sum - mol_sum) < 1e-6)

if COUNTS_MATCH:
    print("\nVERDICT: Official expression matrix exactly matches molecule-derived counts.")
else:
    print("\nVERDICT: Counts are not exactly identical, but molecule-derived counts are close enough to continue.")
    print("We will keep both matrices:")
    print("  raw_expr_original_shared   = official CosMx expression matrix")
    print("  raw_molecule_counts_clean  = molecule-derived matrix")
    print("The main .X and layers['raw'] will use molecule-derived counts for consistency with Step 5.")

# ------------------------------------------------------------
# 9. Store both raw matrices safely
# ------------------------------------------------------------

cell_by_gene_adata.layers["raw_expr_original_shared"] = X_expr.astype(np.float32).copy()
cell_by_gene_adata.layers["raw_molecule_counts_clean"] = X_raw_counts_sparse.astype(np.float32).copy()

# Use molecule-derived counts as main raw matrix.
# This keeps counts consistent with the molecule table used later for Step 5.
cell_by_gene_adata.X = X_raw_counts_sparse.astype(np.float32).copy()
cell_by_gene_adata.layers["raw"] = cell_by_gene_adata.X.copy()

# ------------------------------------------------------------
# 10. Final filtering: remove cells with zero molecule-derived shared counts
# ------------------------------------------------------------

clean_cell_totals = np.asarray(cell_by_gene_adata.X.sum(axis=1)).ravel()
keep_cells_clean = clean_cell_totals > 0

n_before_clean_filter = cell_by_gene_adata.n_obs
n_zero_clean_cells = int((~keep_cells_clean).sum())

print("\nPost-clean cell filtering:")
print(f"Cells before final zero-count filtering : {n_before_clean_filter:,}")
print(f"Cells with zero clean shared counts      : {n_zero_clean_cells:,}")

if n_zero_clean_cells > 0:
    frac_zero = n_zero_clean_cells / max(n_before_clean_filter, 1)

    # This should be small. If huge, something is still wrong.
    if frac_zero > 0.05:
        raise ValueError(
            "\nCRITICAL ERROR: Too many cells have zero molecule-derived counts.\n"
            f"Zero-count cells: {n_zero_clean_cells:,}/{n_before_clean_filter:,} "
            f"({100 * frac_zero:.2f}%).\n"
            "This indicates molecule matching is still wrong. Do NOT continue."
        )

    cell_by_gene_adata = cell_by_gene_adata[keep_cells_clean, :].copy()

    # Keep layers synchronized after subsetting.
    cell_by_gene_adata.layers["raw"] = cell_by_gene_adata.X.copy()
    cell_by_gene_adata.layers["raw_molecule_counts_clean"] = cell_by_gene_adata.X.copy()

    if "raw_expr_original_shared" in cell_by_gene_adata.layers:
        cell_by_gene_adata.layers["raw_expr_original_shared"] = (
            cell_by_gene_adata.layers["raw_expr_original_shared"].copy()
        )

    print(f"Cells after final filtering             : {cell_by_gene_adata.n_obs:,}")
else:
    print("No zero-count cells removed.")

# ------------------------------------------------------------
# 11. Recompute QC fields after final clean matrix
# ------------------------------------------------------------

cell_by_gene_adata.obs["total_counts_shared_clean"] = np.asarray(
    cell_by_gene_adata.X.sum(axis=1)
).ravel()

cell_by_gene_adata.obs["detected_genes_shared_clean"] = np.asarray(
    (cell_by_gene_adata.X > 0).sum(axis=1)
).ravel()

cell_by_gene_adata.var["total_counts_shared_clean"] = np.asarray(
    cell_by_gene_adata.X.sum(axis=0)
).ravel()

cell_by_gene_adata.var["detected_cells_shared_clean"] = np.asarray(
    (cell_by_gene_adata.X > 0).sum(axis=0)
).ravel()

# ------------------------------------------------------------
# 12. Save clean outputs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVE CLEAN SHARED-GENE PREPROCESSING OUTPUTS")
print("=" * 80)

clean_adata_path = WORK_DIR / "cosmx_cell5_spatial_adata_clean_shared_raw_SAFE.h5ad"
xraw_npz_path = WORK_DIR / "cosmx_cell5_X_raw_counts_shared_sparse_SAFE.npz"
clean_chunks_manifest_path = WORK_DIR / "cosmx_cell5_clean_molecule_chunks_manifest_SAFE.json"
clean_summary_path = WORK_DIR / "cosmx_cell5_clean_shared_preprocessing_summary_SAFE.json"

cell_by_gene_adata.write_h5ad(clean_adata_path)
sparse.save_npz(xraw_npz_path, cell_by_gene_adata.X.tocsr())

clean_manifest = {
    "clean_chunk_dir": str(CLEAN_CHUNK_DIR),
    "clean_chunk_paths": clean_chunk_paths,
    "n_clean_chunks": len(clean_chunk_paths),
}

with open(clean_chunks_manifest_path, "w") as f:
    json.dump(clean_manifest, f, indent=2)

cell5_summary = {
    "sample_name": SAMPLE_NAME,
    "n_shared_genes": int(cell_by_gene_adata.n_vars),
    "shared_genes": list(map(str, cell_by_gene_adata.var_names)),
    "n_cells_final": int(cell_by_gene_adata.n_obs),
    "input_molecule_rows": int(summary["input_rows"]),
    "kept_clean_molecule_rows": int(summary["kept_rows"]),
    "removed_unassigned_rows": int(summary["removed_unassigned"]),
    "removed_nonshared_gene_rows": int(summary["removed_nonshared_gene"]),
    "removed_invalid_cell_rows": int(summary["removed_invalid_cell"]),
    "none_gene_rows_seen": int(summary["none_gene_rows_seen"]),
    "unknown_gene_rows_seen": int(summary["unknown_gene_rows_seen"]),
    "nuclear_rows_clean": int(summary["nuclear_rows"]),
    "membrane_rows_clean": int(summary["membrane_rows"]),
    "cytoplasm_rows_clean": int(summary["cytoplasm_rows"]),
    "extracellular_rows_clean": int(summary["extracellular_rows"]),
    "expression_shared_sum": float(expr_sum),
    "molecule_derived_shared_sum": float(mol_sum),
    "ratio_molecule_to_expression": float(ratio),
    "total_count_difference_expr_minus_molecule": float(expr_sum - mol_sum),
    "max_abs_diff_expr_vs_molecule": float(max_abs_diff),
    "n_mismatched_cell_gene_pairs": int(n_mismatched_pairs),
    "counts_match_exactly": bool(COUNTS_MATCH),
    "n_zero_clean_cells_removed": int(n_zero_clean_cells),
    "clean_adata_path": str(clean_adata_path),
    "xraw_sparse_path": str(xraw_npz_path),
    "clean_chunks_manifest_path": str(clean_chunks_manifest_path),
}

with open(clean_summary_path, "w") as f:
    json.dump(cell5_summary, f, indent=2)

print("Saved clean shared-gene AnnData:")
print(f"  {clean_adata_path}")

print("Saved sparse clean raw count matrix:")
print(f"  {xraw_npz_path}")

print("Saved clean molecule chunks manifest:")
print(f"  {clean_chunks_manifest_path}")

print("Saved Cell 5 summary:")
print(f"  {clean_summary_path}")

# ------------------------------------------------------------
# 13. Final report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("REPLACEMENT COSMX CELL 5 FINAL SUMMARY")
print("=" * 80)

print(cell_by_gene_adata)

print(f"\nFinal cells                    : {cell_by_gene_adata.n_obs:,}")
print(f"Final shared genes             : {cell_by_gene_adata.n_vars:,}")
print(f"Final clean molecule rows      : {summary['kept_rows']:,}")
print(f"Final clean raw counts         : {cell_by_gene_adata.X.sum():,.0f}")

print("\nRemoved rows:")
print(f"  Unassigned transcripts        : {summary['removed_unassigned']:,}")
print(f"  Non-shared-gene transcripts   : {summary['removed_nonshared_gene']:,}")
print(f"  Invalid-cell transcripts      : {summary['removed_invalid_cell']:,}")
print(f"  gene_id='None' rows seen      : {summary['none_gene_rows_seen']:,}")
print(f"  gene_id='Unknown' rows seen   : {summary['unknown_gene_rows_seen']:,}")

print("\nClean compartment counts:")
print(f"  Nuclear                       : {summary['nuclear_rows']:,}")
print(f"  Membrane                      : {summary['membrane_rows']:,}")
print(f"  Cytoplasm                     : {summary['cytoplasm_rows']:,}")
print(f"  Extracellular/0               : {summary['extracellular_rows']:,}")

print("\nCount verification:")
print(f"  Official expression shared sum: {expr_sum:,.0f}")
print(f"  Molecule-derived shared sum   : {mol_sum:,.0f}")
print(f"  Ratio molecule/expression     : {ratio:.6f}")
print(f"  Max abs diff                  : {max_abs_diff:.6f}")
print(f"  Mismatched pairs              : {n_mismatched_pairs:,}")
print(f"  Counts match exactly?         : {COUNTS_MATCH}")

print("\nREPLACEMENT COSMX CELL 5 finished successfully.")
print("Next step: build cell geometry / compartment geometry from CellLabels and CompartmentLabels.")

In [ ]:
# ============================================================
# COSMX CELL 6 — BUILD CELL GEOMETRY + COMPARTMENT GEOMETRY
# Purpose:
#   Use CellLabels_FXXX.tif and CompartmentLabels_FXXX.tif to compute
#   per-cell geometry and compartment statistics.
#
# Inputs:
#   - cell_by_gene_adata from Replacement Cell 5
#   - COSMX_CELL_LABEL_PATHS from Cell 1
#   - COSMX_COMPARTMENT_LABEL_PATHS from Cell 1
#
# Outputs:
#   - geometry table per cell
#   - added geometry/compartment columns in cell_by_gene_adata.obs
#   - saved checkpoint AnnData
# ============================================================

import os
import re
import gc
import json
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path
from PIL import Image

print("=" * 80)
print("COSMX CELL 6 — BUILD CELL GEOMETRY + COMPARTMENT GEOMETRY")
print("=" * 80)

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Check required variables
# ------------------------------------------------------------

required_vars = [
    "cell_by_gene_adata",
    "COSMX_CELL_LABEL_PATHS",
    "COSMX_COMPARTMENT_LABEL_PATHS",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Required variable missing: {v}")

print("Current AnnData:")
print(cell_by_gene_adata)

if cell_by_gene_adata.n_obs < 90_000:
    raise ValueError(
        "cell_by_gene_adata has too few cells. "
        "Run the successful Replacement Cell 5 first."
    )

if cell_by_gene_adata.n_vars != 951:
    raise ValueError(
        f"Expected 951 shared genes, found {cell_by_gene_adata.n_vars}. "
        "Run Replacement Cell 5 first."
    )

# ------------------------------------------------------------
# 2. Helper function to extract FOV number from file name
# ------------------------------------------------------------

def extract_fov_number(path):
    """
    Extract FOV number from names like:
      CellLabels_F001.tif
      CompartmentLabels_F030.tif
    """
    path = Path(path)
    match = re.search(r"_F(\d+)", path.stem)
    if match is None:
        raise ValueError(f"Could not extract FOV number from: {path.name}")
    return int(match.group(1))

cell_label_paths = [Path(p) for p in COSMX_CELL_LABEL_PATHS]
comp_label_paths = [Path(p) for p in COSMX_COMPARTMENT_LABEL_PATHS]

cell_label_by_fov = {extract_fov_number(p): p for p in cell_label_paths}
comp_label_by_fov = {extract_fov_number(p): p for p in comp_label_paths}

print(f"CellLabels files found       : {len(cell_label_by_fov)}")
print(f"CompartmentLabels files found: {len(comp_label_by_fov)}")

# FOVs present in the cleaned AnnData
adata_fovs = sorted(cell_by_gene_adata.obs["fov"].astype(int).unique().tolist())

print(f"FOVs in cleaned AnnData: {adata_fovs}")

missing_cell_label_fovs = [f for f in adata_fovs if f not in cell_label_by_fov]
missing_comp_label_fovs = [f for f in adata_fovs if f not in comp_label_by_fov]

if missing_cell_label_fovs:
    raise FileNotFoundError(f"Missing CellLabels files for FOVs: {missing_cell_label_fovs}")

if missing_comp_label_fovs:
    raise FileNotFoundError(f"Missing CompartmentLabels files for FOVs: {missing_comp_label_fovs}")

# ------------------------------------------------------------
# 3. Build per-cell geometry from label images
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPUTING PER-CELL GEOMETRY FROM LABEL IMAGES")
print("=" * 80)

# Pixel size from CosMx README: 0.18 µm per pixel
PIXEL_SIZE_UM = 0.18

valid_cell_ids = set(cell_by_gene_adata.obs_names.astype(str))

geometry_rows = []

for fov in adata_fovs:
    print(f"\nProcessing FOV {fov}...")

    cell_label_path = cell_label_by_fov[fov]
    comp_label_path = comp_label_by_fov[fov]

    print(f"  CellLabels       : {cell_label_path.name}")
    print(f"  CompartmentLabels: {comp_label_path.name}")

    # Load label images
    cell_img = np.array(Image.open(cell_label_path))
    comp_img = np.array(Image.open(comp_label_path))

    if cell_img.shape != comp_img.shape:
        raise ValueError(
            f"Shape mismatch for FOV {fov}: "
            f"CellLabels shape={cell_img.shape}, CompartmentLabels shape={comp_img.shape}"
        )

    print(f"  Image shape: {cell_img.shape}")

    # Flatten arrays
    cell_flat = cell_img.ravel().astype(np.int64)
    comp_flat = comp_img.ravel().astype(np.int64)

    # Only pixels that belong to a segmented cell
    in_cell = cell_flat > 0

    cell_ids_flat = cell_flat[in_cell]
    comp_in_cell = comp_flat[in_cell]

    max_cell_id = int(cell_ids_flat.max()) if len(cell_ids_flat) > 0 else 0

    # Count total cell pixels per cell ID
    total_area = np.bincount(cell_ids_flat, minlength=max_cell_id + 1)

    # Count compartment-specific pixels per cell ID
    nuclear_area = np.bincount(
        cell_ids_flat,
        weights=(comp_in_cell == 1).astype(np.int64),
        minlength=max_cell_id + 1,
    )

    membrane_area = np.bincount(
        cell_ids_flat,
        weights=(comp_in_cell == 2).astype(np.int64),
        minlength=max_cell_id + 1,
    )

    cytoplasm_area = np.bincount(
        cell_ids_flat,
        weights=(comp_in_cell == 3).astype(np.int64),
        minlength=max_cell_id + 1,
    )

    extracellular_area = np.bincount(
        cell_ids_flat,
        weights=(comp_in_cell == 0).astype(np.int64),
        minlength=max_cell_id + 1,
    )

    # Existing cell IDs in this FOV
    cell_ids_in_fov = np.where(total_area > 0)[0]
    cell_ids_in_fov = cell_ids_in_fov[cell_ids_in_fov > 0]

    print(f"  Segmented cells in CellLabels: {len(cell_ids_in_fov):,}")

    # Keep only cells that are in our cleaned AnnData
    kept_count = 0

    for cid in cell_ids_in_fov:
        global_cell_id = f"{fov}_{int(cid)}"

        if global_cell_id not in valid_cell_ids:
            continue

        area_px = float(total_area[cid])
        nuc_px = float(nuclear_area[cid])
        mem_px = float(membrane_area[cid])
        cyto_px = float(cytoplasm_area[cid])
        extra_px = float(extracellular_area[cid])

        denom = area_px if area_px > 0 else np.nan

        geometry_rows.append({
            "cell_id": global_cell_id,
            "fov": int(fov),
            "cell_ID_original": int(cid),

            # Areas from CellLabels/CompartmentLabels images
            "label_cell_area_px": area_px,
            "label_cell_area_um2": area_px * (PIXEL_SIZE_UM ** 2),

            "label_nuclear_area_px": nuc_px,
            "label_membrane_area_px": mem_px,
            "label_cytoplasm_area_px": cyto_px,
            "label_extracellular_area_px": extra_px,

            "label_nuclear_area_um2": nuc_px * (PIXEL_SIZE_UM ** 2),
            "label_membrane_area_um2": mem_px * (PIXEL_SIZE_UM ** 2),
            "label_cytoplasm_area_um2": cyto_px * (PIXEL_SIZE_UM ** 2),
            "label_extracellular_area_um2": extra_px * (PIXEL_SIZE_UM ** 2),

            # Fractions within the cell mask
            "label_nuclear_frac": nuc_px / denom,
            "label_membrane_frac": mem_px / denom,
            "label_cytoplasm_frac": cyto_px / denom,
            "label_extracellular_frac": extra_px / denom,

            # Sanity check: these should add up to total cell area
            "label_compartment_area_sum_px": nuc_px + mem_px + cyto_px + extra_px,
        })

        kept_count += 1

    print(f"  Cells kept after matching AnnData: {kept_count:,}")

    # Free memory for this FOV
    del cell_img, comp_img, cell_flat, comp_flat
    del in_cell, cell_ids_flat, comp_in_cell
    del total_area, nuclear_area, membrane_area, cytoplasm_area, extracellular_area
    gc.collect()

# ------------------------------------------------------------
# 4. Create geometry DataFrame
# ------------------------------------------------------------

geometry_df = pd.DataFrame(geometry_rows)

print("\n" + "=" * 80)
print("GEOMETRY TABLE SUMMARY")
print("=" * 80)

print(f"geometry_df shape: {geometry_df.shape}")
display(geometry_df.head())

print("\nGeometry coverage:")
print(f"Cells in AnnData          : {cell_by_gene_adata.n_obs:,}")
print(f"Cells with geometry rows  : {geometry_df['cell_id'].nunique():,}")

missing_geometry_cells = set(cell_by_gene_adata.obs_names.astype(str)) - set(geometry_df["cell_id"].astype(str))
extra_geometry_cells = set(geometry_df["cell_id"].astype(str)) - set(cell_by_gene_adata.obs_names.astype(str))

print(f"AnnData cells missing geometry: {len(missing_geometry_cells):,}")
print(f"Extra geometry cells          : {len(extra_geometry_cells):,}")

if len(missing_geometry_cells) > 0:
    print("First 20 missing geometry cells:")
    print(sorted(list(missing_geometry_cells))[:20])

# ------------------------------------------------------------
# 5. Validate compartment areas
# ------------------------------------------------------------

geometry_df["label_area_difference_px"] = (
    geometry_df["label_cell_area_px"]
    - geometry_df["label_compartment_area_sum_px"]
)

max_area_diff = float(np.abs(geometry_df["label_area_difference_px"]).max())

print("\nCompartment area sanity check:")
print(f"Max |cell area - compartment sum|: {max_area_diff:.6f} px")

if max_area_diff > 0:
    print("WARNING: Some compartment areas do not exactly sum to cell area.")
    print("This may happen if compartment labels contain unexpected values.")
else:
    print("Compartment areas perfectly sum to cell area.")

print("\nCompartment fraction summary:")
frac_cols = [
    "label_nuclear_frac",
    "label_membrane_frac",
    "label_cytoplasm_frac",
    "label_extracellular_frac",
]
display(geometry_df[frac_cols].describe().T)

print("\nArea summary:")
area_cols = [
    "label_cell_area_px",
    "label_nuclear_area_px",
    "label_membrane_area_px",
    "label_cytoplasm_area_px",
    "label_extracellular_area_px",
]
display(geometry_df[area_cols].describe().T)

# ------------------------------------------------------------
# 6. Merge geometry into AnnData.obs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MERGING GEOMETRY INTO AnnData.obs")
print("=" * 80)

geometry_join = geometry_df.set_index("cell_id")

common_cells = cell_by_gene_adata.obs_names.intersection(geometry_join.index)

print(f"Cells in AnnData      : {cell_by_gene_adata.n_obs:,}")
print(f"Cells matched geometry: {len(common_cells):,}")

if len(common_cells) < 0.95 * cell_by_gene_adata.n_obs:
    raise ValueError(
        "Too many cells are missing geometry. "
        "Check CellLabels/CompartmentLabels matching."
    )

# Add geometry columns
for col in geometry_join.columns:
    if col in ["fov", "cell_ID_original"]:
        continue

    cell_by_gene_adata.obs[col] = np.nan
    cell_by_gene_adata.obs.loc[common_cells, col] = geometry_join.loc[common_cells, col].values

# ------------------------------------------------------------
# 7. Compare metadata area vs label-image area
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPARE METADATA AREA VS LABEL-IMAGE AREA")
print("=" * 80)

if "cell_area_px" in cell_by_gene_adata.obs.columns and "label_cell_area_px" in cell_by_gene_adata.obs.columns:
    area_compare = cell_by_gene_adata.obs[["cell_area_px", "label_cell_area_px"]].copy()
    area_compare["area_diff_px"] = area_compare["cell_area_px"].astype(float) - area_compare["label_cell_area_px"].astype(float)

    print("Area comparison summary:")
    display(area_compare.describe().T)

    print("\nLargest absolute area differences:")
    display(
        area_compare.assign(abs_diff=lambda x: np.abs(x["area_diff_px"]))
        .sort_values("abs_diff", ascending=False)
        .head(20)
    )
else:
    print("Could not compare metadata area and label-image area.")

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVE CELL 6 OUTPUTS")
print("=" * 80)

geometry_path = WORK_DIR / "cosmx_cell6_cell_compartment_geometry.csv"
geometry_parquet_path = WORK_DIR / "cosmx_cell6_cell_compartment_geometry.parquet"
adata_path = WORK_DIR / "cosmx_cell6_adata_with_geometry.h5ad"
summary_path = WORK_DIR / "cosmx_cell6_geometry_summary.json"

geometry_df.to_csv(geometry_path, index=False)
geometry_df.to_parquet(geometry_parquet_path, index=False)
cell_by_gene_adata.write_h5ad(adata_path)

cell6_summary = {
    "sample_name": SAMPLE_NAME,
    "n_cells_adata": int(cell_by_gene_adata.n_obs),
    "n_cells_geometry": int(geometry_df["cell_id"].nunique()),
    "n_missing_geometry_cells": int(len(missing_geometry_cells)),
    "n_extra_geometry_cells": int(len(extra_geometry_cells)),
    "pixel_size_um": float(PIXEL_SIZE_UM),
    "max_area_difference_px": float(max_area_diff),
    "geometry_csv_path": str(geometry_path),
    "geometry_parquet_path": str(geometry_parquet_path),
    "adata_with_geometry_path": str(adata_path),
    "mean_nuclear_frac": float(geometry_df["label_nuclear_frac"].mean()),
    "mean_membrane_frac": float(geometry_df["label_membrane_frac"].mean()),
    "mean_cytoplasm_frac": float(geometry_df["label_cytoplasm_frac"].mean()),
    "mean_extracellular_frac": float(geometry_df["label_extracellular_frac"].mean()),
}

with open(summary_path, "w") as f:
    json.dump(cell6_summary, f, indent=2)

print("Saved geometry CSV:")
print(f"  {geometry_path}")

print("Saved geometry parquet:")
print(f"  {geometry_parquet_path}")

print("Saved AnnData with geometry:")
print(f"  {adata_path}")

print("Saved summary:")
print(f"  {summary_path}")

# Keep variable for later cells
COSMX_CELL_GEOMETRY_PATH = geometry_parquet_path

print("\nCOSMX CELL 6 finished successfully.")
print("Next step: load GSE131907 expression matrix for the 951 shared genes.")

In [ ]:
# ============================================================
# COSMX CELL 7 — LOAD GSE131907 REFERENCE EXPRESSION FOR 951 SHARED GENES
# Purpose:
#   Build the scRNA-seq reference AnnData using only the 951 genes
#   shared between CosMx and GSE131907.
#
# Inputs:
#   - GSE131907 raw UMI matrix from Cell 4.5
#   - GSE131907 cell annotation from Cell 4.5
#   - shared gene list from Cell 4.5
#
# Output:
#   - annotated_ref_adata: scRNA cells × 951 shared genes
# ============================================================

import os
import gc
import gzip
import json
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from pathlib import Path

print("=" * 80)
print("COSMX CELL 7 — LOAD GSE131907 REFERENCE EXPRESSION FOR 951 SHARED GENES")
print("=" * 80)

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Check/load required paths
# ------------------------------------------------------------

if "SCRNA_RAW_UMI_PATH" not in globals():
    possible_path = Path("/content/gse131907/GSE131907/GSE131907_Lung_Cancer_raw_UMI_matrix.txt.gz")
    if not possible_path.exists():
        raise FileNotFoundError("SCRNA_RAW_UMI_PATH not found. Run Cell 4.5 first.")
    SCRNA_RAW_UMI_PATH = possible_path

if "SCRNA_ANNOT_PATH" not in globals():
    possible_path = Path("/content/gse131907/GSE131907/GSE131907_Lung_Cancer_cell_annotation.txt.gz")
    if not possible_path.exists():
        raise FileNotFoundError("SCRNA_ANNOT_PATH not found. Run Cell 4.5 first.")
    SCRNA_ANNOT_PATH = possible_path

SCRNA_RAW_UMI_PATH = Path(SCRNA_RAW_UMI_PATH)
SCRNA_ANNOT_PATH = Path(SCRNA_ANNOT_PATH)

print(f"scRNA raw UMI matrix : {SCRNA_RAW_UMI_PATH}")
print(f"scRNA annotation file: {SCRNA_ANNOT_PATH}")

if not SCRNA_RAW_UMI_PATH.exists():
    raise FileNotFoundError(SCRNA_RAW_UMI_PATH)

if not SCRNA_ANNOT_PATH.exists():
    raise FileNotFoundError(SCRNA_ANNOT_PATH)

# ------------------------------------------------------------
# 2. Load shared genes
# ------------------------------------------------------------

if "SHARED_GENES_COSMX_GSE131907" not in globals():
    shared_genes_path = WORK_DIR / "cosmx_gse131907_shared_genes.txt"
    if not shared_genes_path.exists():
        raise FileNotFoundError("Shared genes file not found. Run Cell 4.5 first.")
    SHARED_GENES_COSMX_GSE131907 = [
        line.strip() for line in open(shared_genes_path) if line.strip()
    ]

shared_genes = [str(g).strip() for g in SHARED_GENES_COSMX_GSE131907]
shared_gene_set = set(shared_genes)

print(f"Shared genes to extract from reference: {len(shared_genes):,}")

if len(shared_genes) < 900:
    raise ValueError("Too few shared genes. Expected about 951.")

# ------------------------------------------------------------
# 3. Load annotation table
# ------------------------------------------------------------

print("\nLoading GSE131907 annotation...")
scrna_annot = pd.read_csv(SCRNA_ANNOT_PATH, sep="\t", compression="gzip")

print(f"scrna_annot shape: {scrna_annot.shape}")
print("Annotation columns:")
print(list(scrna_annot.columns))
display(scrna_annot.head())

required_annot_cols = ["Index", "Cell_type", "Cell_type.refined", "Cell_subtype"]
missing = [c for c in required_annot_cols if c not in scrna_annot.columns]
if missing:
    raise KeyError(f"Missing expected annotation columns: {missing}")

# Use broad cell type for first-pass assignment.
REFERENCE_CELL_TYPE_COL = "Cell_type"

print(f"\nUsing reference label column: {REFERENCE_CELL_TYPE_COL}")
display(scrna_annot[REFERENCE_CELL_TYPE_COL].value_counts(dropna=False))

# ------------------------------------------------------------
# 4. Stream raw UMI matrix and extract only shared genes
# ------------------------------------------------------------
# The raw matrix is genes × cells.
# We do not load all 29k genes. We only read rows for the 951 shared genes.

print("\n" + "=" * 80)
print("STREAMING RAW UMI MATRIX AND EXTRACTING SHARED GENES")
print("=" * 80)

selected_gene_rows = []
selected_gene_names = []
header_cells = None

n_total_gene_rows = 0
n_selected_gene_rows = 0

with gzip.open(SCRNA_RAW_UMI_PATH, "rt") as f:
    header = f.readline().rstrip("\n")
    header_parts = header.split("\t")

    if header_parts[0] != "Index":
        print(f"WARNING: first header field is {header_parts[0]}, expected 'Index'.")

    header_cells = header_parts[1:]

    print(f"Reference cells in raw matrix header: {len(header_cells):,}")
    print("First 5 cell IDs:")
    print(header_cells[:5])

    for line in f:
        n_total_gene_rows += 1

        if not line:
            continue

        first_tab = line.find("\t")
        if first_tab == -1:
            continue

        gene = line[:first_tab].strip().strip('"')

        if gene not in shared_gene_set:
            continue

        values_str = line[first_tab + 1:].rstrip("\n")

        # Convert tab-separated counts into numeric array.
        counts = np.fromstring(values_str, sep="\t", dtype=np.float32)

        if counts.shape[0] != len(header_cells):
            raise ValueError(
                f"Gene {gene} has {counts.shape[0]} values, "
                f"expected {len(header_cells)}."
            )

        selected_gene_names.append(gene)

        # Store as sparse row to save memory.
        selected_gene_rows.append(sparse.csr_matrix(counts.reshape(1, -1)))

        n_selected_gene_rows += 1

        if n_selected_gene_rows % 100 == 0:
            print(f"Selected {n_selected_gene_rows:,} shared genes so far...")

print("\nFinished reading reference matrix.")
print(f"Total reference gene rows scanned : {n_total_gene_rows:,}")
print(f"Shared gene rows selected         : {n_selected_gene_rows:,}")

missing_shared_genes = sorted(list(shared_gene_set - set(selected_gene_names)))
print(f"Shared genes missing from raw UMI matrix: {len(missing_shared_genes):,}")
print(missing_shared_genes[:50])

if len(selected_gene_names) < 900:
    raise ValueError("Too few shared genes were extracted from the reference matrix.")

# ------------------------------------------------------------
# 5. Reorder reference genes to CosMx shared-gene order
# ------------------------------------------------------------

print("\nReordering reference matrix to match CosMx shared-gene order...")

# Stack selected gene rows: genes × cells
X_ref_gene_by_cell = sparse.vstack(selected_gene_rows).tocsr()
selected_gene_names = np.array(selected_gene_names, dtype=str)

gene_to_row = {g: i for i, g in enumerate(selected_gene_names)}

ordered_genes = [g for g in shared_genes if g in gene_to_row]
ordered_row_indices = [gene_to_row[g] for g in ordered_genes]

X_ref_gene_by_cell = X_ref_gene_by_cell[ordered_row_indices, :]

# Transpose to AnnData format: cells × genes
X_ref_cell_by_gene = X_ref_gene_by_cell.T.tocsr()

annotated_ref_adata = ad.AnnData(X=X_ref_cell_by_gene)
annotated_ref_adata.obs_names = pd.Index(header_cells, name="cell_id")
annotated_ref_adata.var_names = pd.Index(ordered_genes, name="gene_id")
annotated_ref_adata.var_names_make_unique()

print(annotated_ref_adata)

# ------------------------------------------------------------
# 6. Attach annotation labels to reference cells
# ------------------------------------------------------------

print("\nAttaching GSE131907 cell annotations...")

annot_indexed = scrna_annot.copy()
annot_indexed["Index"] = annot_indexed["Index"].astype(str)
annot_indexed = annot_indexed.drop_duplicates(subset=["Index"]).set_index("Index")

common_cells = annotated_ref_adata.obs_names.intersection(annot_indexed.index)

print(f"Reference cells in matrix    : {annotated_ref_adata.n_obs:,}")
print(f"Cells in annotation table    : {annot_indexed.shape[0]:,}")
print(f"Matched cells                : {len(common_cells):,}")

if len(common_cells) < 0.95 * annotated_ref_adata.n_obs:
    raise ValueError("Too many scRNA cells failed to match annotation.")

for col in scrna_annot.columns:
    if col == "Index":
        continue

    annotated_ref_adata.obs[col] = pd.NA
    annotated_ref_adata.obs.loc[common_cells, col] = annot_indexed.loc[common_cells, col].values

# Clean broad label
annotated_ref_adata.obs["Reference_Cell_Type"] = (
    annotated_ref_adata.obs[REFERENCE_CELL_TYPE_COL]
    .astype("string")
    .fillna("Unknown")
    .astype(str)
)

print("\nReference cell-type distribution:")
display(annotated_ref_adata.obs["Reference_Cell_Type"].value_counts(dropna=False))

# ------------------------------------------------------------
# 7. Basic QC
# ------------------------------------------------------------

ref_total_counts = np.asarray(annotated_ref_adata.X.sum(axis=1)).ravel()
ref_detected_genes = np.asarray((annotated_ref_adata.X > 0).sum(axis=1)).ravel()

annotated_ref_adata.obs["ref_total_counts_shared"] = ref_total_counts
annotated_ref_adata.obs["ref_detected_genes_shared"] = ref_detected_genes

annotated_ref_adata.var["ref_total_counts"] = np.asarray(
    annotated_ref_adata.X.sum(axis=0)
).ravel()

annotated_ref_adata.var["ref_detected_cells"] = np.asarray(
    (annotated_ref_adata.X > 0).sum(axis=0)
).ravel()

print("\nReference QC summary:")
print(f"Reference cells                  : {annotated_ref_adata.n_obs:,}")
print(f"Shared genes                     : {annotated_ref_adata.n_vars:,}")
print(f"Total shared-gene UMI counts     : {annotated_ref_adata.X.sum():,.0f}")
print(f"Mean shared-gene counts per cell : {ref_total_counts.mean():.2f}")
print(f"Median shared-gene counts/cell   : {np.median(ref_total_counts):.2f}")
print(f"Mean detected shared genes/cell  : {ref_detected_genes.mean():.2f}")

# ------------------------------------------------------------
# 8. Save reference object
# ------------------------------------------------------------

ref_adata_path = WORK_DIR / "cosmx_cell7_gse131907_reference_shared_951.h5ad"
ref_summary_path = WORK_DIR / "cosmx_cell7_gse131907_reference_summary.json"

annotated_ref_adata.write_h5ad(ref_adata_path)

cell7_summary = {
    "reference": "GSE131907",
    "raw_umi_path": str(SCRNA_RAW_UMI_PATH),
    "annotation_path": str(SCRNA_ANNOT_PATH),
    "n_reference_cells": int(annotated_ref_adata.n_obs),
    "n_shared_genes": int(annotated_ref_adata.n_vars),
    "shared_genes": list(map(str, annotated_ref_adata.var_names)),
    "label_column_used": REFERENCE_CELL_TYPE_COL,
    "reference_cell_type_counts": {
        str(k): int(v)
        for k, v in annotated_ref_adata.obs["Reference_Cell_Type"].value_counts().items()
    },
    "ref_adata_path": str(ref_adata_path),
}

with open(ref_summary_path, "w") as f:
    json.dump(cell7_summary, f, indent=2)

print("\nSaved reference AnnData:")
print(f"  {ref_adata_path}")

print("Saved Cell 7 summary:")
print(f"  {ref_summary_path}")

print("\nCOSMX CELL 7 finished successfully.")
print("Next step: compute reference marker genes and assign marker-based CosMx cell types.")

In [ ]:
# ============================================================
# COSMX CELL 8 — REFERENCE MARKERS + MARKER-BASED COSMX CELL-TYPE ASSIGNMENT
# Purpose:
#   Similar to the Xenium cell-type labeling logic:
#     1. Use scRNA-seq reference labels.
#     2. Find marker genes for each reference cell type.
#     3. Score each CosMx cell using those marker genes.
#     4. Assign the CosMx cell to the reference cell type whose markers are most active.
#
# Inputs:
#   - cell_by_gene_adata from Cell 6
#   - reference AnnData from Cell 7:
#       /content/cosmx_pipeline_work/cosmx_cell7_gse131907_reference_shared_951.h5ad
#
# Outputs:
#   - Reference marker gene table
#   - Reference cell-type profiles
#   - CosMx cells annotated with Assigned_CosMx_Cell_Type
#   - Saved AnnData checkpoint
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from pathlib import Path

print("=" * 80)
print("COSMX CELL 8 — REFERENCE MARKERS + MARKER-BASED COSMX CELL-TYPE ASSIGNMENT")
print("=" * 80)

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load CosMx AnnData if needed
# ------------------------------------------------------------

if "cell_by_gene_adata" not in globals():
    cosmx_adata_path = WORK_DIR / "cosmx_cell6_adata_with_geometry.h5ad"
    if not cosmx_adata_path.exists():
        raise FileNotFoundError(
            "cell_by_gene_adata not found and Cell 6 checkpoint is missing. "
            "Run Cell 6 first."
        )
    cell_by_gene_adata = sc.read_h5ad(cosmx_adata_path)

print("\nCosMx AnnData:")
print(cell_by_gene_adata)

if cell_by_gene_adata.n_vars != 951:
    raise ValueError(
        f"Expected CosMx AnnData to have 951 shared genes. Found {cell_by_gene_adata.n_vars}."
    )

# ------------------------------------------------------------
# 2. Load GSE131907 reference AnnData
# ------------------------------------------------------------

ref_adata_path = WORK_DIR / "cosmx_cell7_gse131907_reference_shared_951.h5ad"

if "adata_ref_shared" in globals():
    ref_adata = adata_ref_shared
else:
    if not ref_adata_path.exists():
        raise FileNotFoundError(
            f"Reference AnnData not found: {ref_adata_path}. Run Cell 7 first."
        )
    ref_adata = sc.read_h5ad(ref_adata_path)

print("\nReference AnnData:")
print(ref_adata)

if ref_adata.n_vars != 951:
    raise ValueError(
        f"Expected reference AnnData to have 951 shared genes. Found {ref_adata.n_vars}."
    )

if "Reference_Cell_Type" not in ref_adata.obs.columns:
    raise KeyError(
        "Reference_Cell_Type column not found in reference AnnData. "
        "Check Cell 7 output."
    )

# ------------------------------------------------------------
# 3. Make sure CosMx and reference genes are in the same order
# ------------------------------------------------------------

cosmx_genes = pd.Index(cell_by_gene_adata.var_names.astype(str))
ref_genes = pd.Index(ref_adata.var_names.astype(str))

if not cosmx_genes.equals(ref_genes):
    print("\nGene order differs between CosMx and reference. Reordering reference to match CosMx.")
    missing_in_ref = [g for g in cosmx_genes if g not in set(ref_genes)]
    if missing_in_ref:
        raise ValueError(f"Genes missing in reference: {missing_in_ref[:20]}")
    ref_adata = ref_adata[:, cosmx_genes].copy()

print("\nGene order check passed.")
print(f"Shared genes: {len(cosmx_genes):,}")

# ------------------------------------------------------------
# 4. Normalize and log-transform both datasets for marker scoring
# ------------------------------------------------------------
# We do NOT overwrite raw count layers. We create temporary normalized matrices.

def normalize_log1p_sparse(X, target_sum=1e4):
    """
    Normalize each cell to target_sum counts and apply log1p.
    Works for sparse or dense matrices and returns CSR sparse matrix.
    """
    if sparse.issparse(X):
        X = X.tocsr().astype(np.float32)
    else:
        X = sparse.csr_matrix(np.asarray(X, dtype=np.float32))

    cell_sums = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    scale = np.zeros_like(cell_sums, dtype=np.float32)
    nonzero = cell_sums > 0
    scale[nonzero] = target_sum / cell_sums[nonzero]

    X_norm = sparse.diags(scale).dot(X).tocsr()
    X_norm.data = np.log1p(X_norm.data)
    return X_norm

print("\nNormalizing/log-transforming reference and CosMx matrices...")

X_ref_log = normalize_log1p_sparse(ref_adata.X, target_sum=1e4)
X_cosmx_log = normalize_log1p_sparse(cell_by_gene_adata.X, target_sum=1e4)

print(f"X_ref_log shape  : {X_ref_log.shape}")
print(f"X_cosmx_log shape: {X_cosmx_log.shape}")

# ------------------------------------------------------------
# 5. Compute reference marker genes by cell type
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPUTE REFERENCE MARKER GENES")
print("=" * 80)

label_col = "Reference_Cell_Type"

# Keep broad labels, but remove very unhelpful label if desired.
# We will keep "Undetermined" in outputs, but it is not preferred for final assignment.
cell_types_all = ref_adata.obs[label_col].astype(str).values
cell_type_counts = pd.Series(cell_types_all).value_counts()

print("\nReference cell-type counts:")
display(cell_type_counts)

MIN_REF_CELLS_PER_TYPE = 100
TOP_MARKERS_PER_TYPE = 30

valid_cell_types = [
    ct for ct, n in cell_type_counts.items()
    if n >= MIN_REF_CELLS_PER_TYPE
]

print(f"\nCell types used for marker discovery: {len(valid_cell_types)}")
print(valid_cell_types)

n_ref, n_genes = X_ref_log.shape
genes = np.array(ref_adata.var_names.astype(str))

# Precompute total sums and detection counts for outside-vs-inside comparison
X_ref_binary = X_ref_log.copy()
X_ref_binary.data = np.ones_like(X_ref_binary.data)

total_sum = np.asarray(X_ref_log.sum(axis=0)).ravel()
total_detect = np.asarray(X_ref_binary.sum(axis=0)).ravel()

marker_rows = []
profile_rows = []

for ct in valid_cell_types:
    print(f"\nFinding markers for: {ct}")

    in_mask = cell_types_all == ct
    n_in = int(in_mask.sum())
    n_out = int(n_ref - n_in)

    X_in = X_ref_log[in_mask, :]
    X_bin_in = X_ref_binary[in_mask, :]

    sum_in = np.asarray(X_in.sum(axis=0)).ravel()
    detect_in = np.asarray(X_bin_in.sum(axis=0)).ravel()

    mean_in = sum_in / max(n_in, 1)
    mean_out = (total_sum - sum_in) / max(n_out, 1)

    pct_in = detect_in / max(n_in, 1)
    pct_out = (total_detect - detect_in) / max(n_out, 1)

    # Marker score:
    # high if gene is strongly expressed in this type and less expressed outside.
    mean_diff = mean_in - mean_out
    pct_diff = pct_in - pct_out
    marker_score = mean_diff * np.maximum(pct_diff, 0)

    # Build marker table for this cell type
    ct_df = pd.DataFrame({
        "reference_cell_type": ct,
        "gene": genes,
        "mean_in": mean_in,
        "mean_out": mean_out,
        "mean_diff": mean_diff,
        "pct_in": pct_in,
        "pct_out": pct_out,
        "pct_diff": pct_diff,
        "marker_score": marker_score,
        "n_cells_in_type": n_in,
    })

    # Prefer genes enriched in this cell type.
    candidate_df = ct_df[
        (ct_df["mean_diff"] > 0)
        & (ct_df["pct_in"] >= 0.03)
        & (ct_df["marker_score"] > 0)
    ].copy()

    if len(candidate_df) == 0:
        print("  WARNING: no marker candidates under thresholds; using top mean_diff genes.")
        candidate_df = ct_df.sort_values("mean_diff", ascending=False).head(TOP_MARKERS_PER_TYPE).copy()
    else:
        candidate_df = candidate_df.sort_values("marker_score", ascending=False).head(TOP_MARKERS_PER_TYPE)

    candidate_df["marker_rank"] = np.arange(1, len(candidate_df) + 1)

    marker_rows.append(candidate_df)

    # Reference profile = mean log-normalized expression of this cell type
    profile_rows.append(pd.Series(mean_in, index=genes, name=ct))

    print("  Top markers:")
    print(candidate_df["gene"].head(15).tolist())

ref_marker_df = pd.concat(marker_rows, ignore_index=True)
ref_profile_df = pd.DataFrame(profile_rows)

print("\nReference marker table shape:", ref_marker_df.shape)
display(ref_marker_df.head(30))

print("\nReference profile matrix shape:", ref_profile_df.shape)
display(ref_profile_df.iloc[:5, :10])

# ------------------------------------------------------------
# 6. Build marker gene sets for scoring CosMx cells
# ------------------------------------------------------------

marker_sets = {}

for ct in valid_cell_types:
    markers = (
        ref_marker_df[ref_marker_df["reference_cell_type"] == ct]
        .sort_values("marker_rank")["gene"]
        .astype(str)
        .tolist()
    )
    markers = [g for g in markers if g in set(cosmx_genes)]
    marker_sets[ct] = markers

print("\nMarker sets used for CosMx scoring:")
for ct, markers in marker_sets.items():
    print(f"{ct:25s}: {len(markers):2d} markers | {markers[:12]}")

# ------------------------------------------------------------
# 7. Score each CosMx cell against each reference cell type
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SCORE COSMX CELLS USING REFERENCE MARKER GENES")
print("=" * 80)

gene_to_idx = {g: i for i, g in enumerate(cosmx_genes)}

score_mat = np.zeros((cell_by_gene_adata.n_obs, len(valid_cell_types)), dtype=np.float32)

for j, ct in enumerate(valid_cell_types):
    markers = marker_sets[ct]

    if len(markers) == 0:
        score_mat[:, j] = 0
        continue

    marker_idx = [gene_to_idx[g] for g in markers]

    # Average log-normalized expression of marker genes
    scores = np.asarray(X_cosmx_log[:, marker_idx].mean(axis=1)).ravel()
    score_mat[:, j] = scores.astype(np.float32)

score_df = pd.DataFrame(
    score_mat,
    index=cell_by_gene_adata.obs_names.astype(str),
    columns=valid_cell_types,
)

print("Score matrix shape:", score_df.shape)
display(score_df.head())

# ------------------------------------------------------------
# 8. Assign best cell type and confidence metrics
# ------------------------------------------------------------

print("\nAssigning CosMx cell types...")

# Best and second-best marker scores
score_values = score_df.to_numpy(dtype=np.float32)

best_idx = np.argmax(score_values, axis=1)
best_scores = score_values[np.arange(score_values.shape[0]), best_idx]

# second-best score
score_values_copy = score_values.copy()
score_values_copy[np.arange(score_values.shape[0]), best_idx] = -np.inf
second_idx = np.argmax(score_values_copy, axis=1)
second_scores = score_values_copy[np.arange(score_values_copy.shape[0]), second_idx]

assigned_types = np.array(valid_cell_types, dtype=object)[best_idx]
second_types = np.array(valid_cell_types, dtype=object)[second_idx]

score_margin = best_scores - second_scores

# Avoid assigning "Undetermined" as a confident biological label unless it really wins.
# We keep the raw assignment but also create a final cleaned assignment.
assigned_types_final = assigned_types.copy()

LOW_SCORE_THRESHOLD = 0.03
LOW_MARGIN_THRESHOLD = 0.01

low_confidence = (
    (best_scores < LOW_SCORE_THRESHOLD)
    | (score_margin < LOW_MARGIN_THRESHOLD)
)

# If assignment is weak, mark it as Low_confidence instead of forcing a cell type.
assigned_types_final[low_confidence] = "Low_confidence"

# Add to AnnData.obs
cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type_raw"] = assigned_types
cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"] = assigned_types_final
cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type_second"] = second_types
cell_by_gene_adata.obs["celltype_marker_score_best"] = best_scores
cell_by_gene_adata.obs["celltype_marker_score_second"] = second_scores
cell_by_gene_adata.obs["celltype_marker_score_margin"] = score_margin
cell_by_gene_adata.obs["celltype_low_confidence"] = low_confidence

print("\nRaw assigned CosMx cell-type distribution:")
display(cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type_raw"].value_counts())

print("\nFinal assigned CosMx cell-type distribution:")
display(cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].value_counts())

print("\nConfidence summary:")
display(
    cell_by_gene_adata.obs[
        [
            "celltype_marker_score_best",
            "celltype_marker_score_second",
            "celltype_marker_score_margin",
        ]
    ].describe().T
)

# ------------------------------------------------------------
# 9. Summarize marker scores by assigned cell type
# ------------------------------------------------------------

print("\nMean marker score by final assigned cell type:")
score_summary = (
    cell_by_gene_adata.obs
    .groupby("Assigned_CosMx_Cell_Type")
    [
        [
            "celltype_marker_score_best",
            "celltype_marker_score_second",
            "celltype_marker_score_margin",
        ]
    ]
    .mean()
    .sort_values("celltype_marker_score_best", ascending=False)
)

display(score_summary)

# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVE CELL 8 OUTPUTS")
print("=" * 80)

markers_path = WORK_DIR / "cosmx_cell8_gse131907_reference_marker_genes.csv"
profiles_path = WORK_DIR / "cosmx_cell8_gse131907_reference_profiles.csv"
score_path = WORK_DIR / "cosmx_cell8_cosmx_marker_score_matrix.parquet"
adata_path = WORK_DIR / "cosmx_cell8_adata_with_marker_celltypes.h5ad"
summary_path = WORK_DIR / "cosmx_cell8_marker_assignment_summary.json"

ref_marker_df.to_csv(markers_path, index=False)
ref_profile_df.to_csv(profiles_path)
score_df.to_parquet(score_path)
cell_by_gene_adata.write_h5ad(adata_path)

cell8_summary = {
    "sample_name": SAMPLE_NAME,
    "reference": "GSE131907",
    "label_column": label_col,
    "n_reference_cells": int(ref_adata.n_obs),
    "n_cosmx_cells": int(cell_by_gene_adata.n_obs),
    "n_shared_genes": int(cell_by_gene_adata.n_vars),
    "n_reference_cell_types_used": int(len(valid_cell_types)),
    "reference_cell_type_counts": {
        str(k): int(v) for k, v in cell_type_counts.items()
    },
    "top_markers_per_type": int(TOP_MARKERS_PER_TYPE),
    "low_score_threshold": float(LOW_SCORE_THRESHOLD),
    "low_margin_threshold": float(LOW_MARGIN_THRESHOLD),
    "final_cosmx_cell_type_counts": {
        str(k): int(v)
        for k, v in cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].value_counts().items()
    },
    "raw_cosmx_cell_type_counts": {
        str(k): int(v)
        for k, v in cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type_raw"].value_counts().items()
    },
    "markers_path": str(markers_path),
    "profiles_path": str(profiles_path),
    "score_matrix_path": str(score_path),
    "adata_with_celltypes_path": str(adata_path),
}

with open(summary_path, "w") as f:
    json.dump(cell8_summary, f, indent=2)

print("Saved marker gene table:")
print(f"  {markers_path}")

print("Saved reference profiles:")
print(f"  {profiles_path}")

print("Saved CosMx marker score matrix:")
print(f"  {score_path}")

print("Saved AnnData with marker-based cell types:")
print(f"  {adata_path}")

print("Saved Cell 8 summary:")
print(f"  {summary_path}")

# Keep useful variables for later cells
COSMX_CELLTYPE_MARKERS_PATH = markers_path
COSMX_CELLTYPE_PROFILES_PATH = profiles_path
COSMX_MARKER_SCORE_PATH = score_path

print("\nCOSMX CELL 8 finished successfully.")
print("Next step: inspect marker assignments, then attach cell types to molecule table.")

In [ ]:
# ============================================================
# COSMX CELL 9 — VALIDATE MARKER-BASED COSMX CELL-TYPE ASSIGNMENTS
# Purpose:
#   Inspect whether Cell 8's assigned CosMx cell types are biologically reasonable.
#
# Checks:
#   1. Assigned cell-type distribution
#   2. Canonical marker expression by assigned cell type
#   3. Morphology marker agreement:
#        - PanCK should be high in epithelial/tumor-like cells
#        - CD45 should be high in immune cells
#        - CD3 should be high in T lymphocytes
#   4. Suspicious labels:
#        - Oligodendrocytes
#        - Undetermined
#        - Low_confidence
#
# Output:
#   Saves validation tables for deciding whether to refine labels before denoising.
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from pathlib import Path

print("=" * 80)
print("COSMX CELL 9 — VALIDATE MARKER-BASED COSMX CELL-TYPE ASSIGNMENTS")
print("=" * 80)

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load Cell 8 AnnData if needed
# ------------------------------------------------------------

if "cell_by_gene_adata" not in globals():
    adata_path = WORK_DIR / "cosmx_cell8_adata_with_marker_celltypes.h5ad"
    if not adata_path.exists():
        raise FileNotFoundError(
            "cell_by_gene_adata not found and Cell 8 checkpoint is missing. "
            "Run Cell 8 first."
        )
    cell_by_gene_adata = sc.read_h5ad(adata_path)

print("Current CosMx AnnData:")
print(cell_by_gene_adata)

required_cols = [
    "Assigned_CosMx_Cell_Type",
    "Assigned_CosMx_Cell_Type_raw",
    "celltype_marker_score_best",
    "celltype_marker_score_second",
    "celltype_marker_score_margin",
]

missing_cols = [c for c in required_cols if c not in cell_by_gene_adata.obs.columns]
if missing_cols:
    raise KeyError(
        f"Missing required cell-type assignment columns: {missing_cols}. "
        "Run Cell 8 first."
    )

# ------------------------------------------------------------
# 2. Basic assignment distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ASSIGNED COSMX CELL-TYPE DISTRIBUTION")
print("=" * 80)

celltype_counts = cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].value_counts()
celltype_counts_df = celltype_counts.reset_index()
celltype_counts_df.columns = ["Assigned_CosMx_Cell_Type", "n_cells"]
celltype_counts_df["fraction"] = celltype_counts_df["n_cells"] / cell_by_gene_adata.n_obs

display(celltype_counts_df)

print("\nRaw assignment distribution:")
display(cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type_raw"].value_counts().reset_index())

print("\nLow-confidence count:")
if "celltype_low_confidence" in cell_by_gene_adata.obs.columns:
    display(cell_by_gene_adata.obs["celltype_low_confidence"].value_counts(dropna=False).reset_index())

# ------------------------------------------------------------
# 3. Define canonical marker genes for sanity checking
# ------------------------------------------------------------
# These are broad marker sets for NSCLC/lung cancer tissue.
# Only genes present in the 951-gene CosMx panel will be used.

marker_dict = {
    "Epithelial cells": [
        "EPCAM", "KRT8", "KRT18", "KRT19", "KRT7", "CLDN4", "TACSTD2", "AGR2", "CEACAM6"
    ],
    "T lymphocytes": [
        "CD3D", "CD3E", "CD3G", "CD2", "TRAC", "IL7R", "CD69", "CCL5", "PTPRC"
    ],
    "B lymphocytes": [
        "MS4A1", "CD79A", "CD79B", "CD37", "IGKC", "IGHM", "IGHG1", "HLA-DRA"
    ],
    "Myeloid cells": [
        "LYZ", "TYROBP", "CD68", "C1QA", "C1QB", "C1QC", "FCER1G", "S100A9", "LST1"
    ],
    "NK cells": [
        "NKG7", "GNLY", "GZMB", "PRF1", "CST7", "GZMA", "GZMH", "CCL5", "KLRB1"
    ],
    "Fibroblasts": [
        "DCN", "LUM", "COL1A1", "COL1A2", "COL3A1", "COL6A1", "COL6A2", "BGN", "MMP2"
    ],
    "Endothelial cells": [
        "PECAM1", "VWF", "RAMP2", "RAMP3", "CAV1", "CLEC14A", "SPARCL1", "IFI27"
    ],
    "MAST cells": [
        "TPSB2", "TPSAB1", "CPA3", "KIT", "HPGDS", "CLU", "FCER1G"
    ],
    "Oligodendrocytes": [
        "S100B", "APOD", "PTGDS", "CRYAB", "CLU", "SPP1"
    ],
    "Immune_general": [
        "PTPRC", "CD52", "HLA-DRA", "HLA-DPA1", "HLA-DPB1", "B2M"
    ],
}

available_genes = set(cell_by_gene_adata.var_names.astype(str))

marker_dict_present = {}
for label, genes in marker_dict.items():
    present = [g for g in genes if g in available_genes]
    marker_dict_present[label] = present

print("\nCanonical marker genes present in CosMx panel:")
for label, genes in marker_dict_present.items():
    print(f"{label:22s}: {len(genes):2d} / {len(marker_dict[label]):2d} present | {genes}")

# ------------------------------------------------------------
# 4. Normalize/log-transform CosMx matrix for marker score validation
# ------------------------------------------------------------

def normalize_log1p_sparse(X, target_sum=1e4):
    if sparse.issparse(X):
        X = X.tocsr().astype(np.float32)
    else:
        X = sparse.csr_matrix(np.asarray(X, dtype=np.float32))

    cell_sums = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    scale = np.zeros_like(cell_sums, dtype=np.float32)
    nonzero = cell_sums > 0
    scale[nonzero] = target_sum / cell_sums[nonzero]

    X_norm = sparse.diags(scale).dot(X).tocsr()
    X_norm.data = np.log1p(X_norm.data)
    return X_norm

print("\nNormalizing/log-transforming CosMx matrix for marker validation...")
X_cosmx_log = normalize_log1p_sparse(cell_by_gene_adata.X, target_sum=1e4)

gene_to_idx = {g: i for i, g in enumerate(cell_by_gene_adata.var_names.astype(str))}

# ------------------------------------------------------------
# 5. Compute canonical marker scores for every CosMx cell
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPUTE CANONICAL MARKER SCORES")
print("=" * 80)

marker_score_cols = []

for label, genes in marker_dict_present.items():
    score_col = f"marker_score__{label.replace(' ', '_')}"

    if len(genes) == 0:
        cell_by_gene_adata.obs[score_col] = 0.0
    else:
        idx = [gene_to_idx[g] for g in genes]
        scores = np.asarray(X_cosmx_log[:, idx].mean(axis=1)).ravel()
        cell_by_gene_adata.obs[score_col] = scores.astype(np.float32)

    marker_score_cols.append(score_col)

print("Marker score columns added:")
print(marker_score_cols)

# ------------------------------------------------------------
# 6. Summarize marker scores by assigned cell type
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MARKER SCORE SUMMARY BY ASSIGNED CELL TYPE")
print("=" * 80)

marker_summary = (
    cell_by_gene_adata.obs
    .groupby("Assigned_CosMx_Cell_Type")[marker_score_cols]
    .mean()
)

display(marker_summary)

# For each assigned cell type, show which canonical marker score is highest.
top_marker_match_rows = []

for assigned_type, row in marker_summary.iterrows():
    sorted_scores = row.sort_values(ascending=False)
    top_marker_match_rows.append({
        "Assigned_CosMx_Cell_Type": assigned_type,
        "top_canonical_marker_program": sorted_scores.index[0].replace("marker_score__", "").replace("_", " "),
        "top_marker_score": float(sorted_scores.iloc[0]),
        "second_canonical_marker_program": sorted_scores.index[1].replace("marker_score__", "").replace("_", " "),
        "second_marker_score": float(sorted_scores.iloc[1]),
        "margin": float(sorted_scores.iloc[0] - sorted_scores.iloc[1]),
    })

top_marker_match_df = pd.DataFrame(top_marker_match_rows)

print("\nTop canonical marker program per assigned cell type:")
display(top_marker_match_df)

# ------------------------------------------------------------
# 7. Morphology marker validation
# ------------------------------------------------------------
# CosMx metadata includes Mean.PanCK, Mean.CD45, Mean.CD3.
# These can sanity-check broad cell types:
#   epithelial/tumor-like cells -> higher PanCK
#   immune cells -> higher CD45
#   T cells -> higher CD3

print("\n" + "=" * 80)
print("MORPHOLOGY MARKER VALIDATION")
print("=" * 80)

morph_cols = [
    "Mean.PanCK", "Max.PanCK",
    "Mean.CD45", "Max.CD45",
    "Mean.CD3", "Max.CD3",
    "Mean.DAPI", "Max.DAPI",
    "Mean.MembraneStain", "Max.MembraneStain",
]

available_morph_cols = [c for c in morph_cols if c in cell_by_gene_adata.obs.columns]

if len(available_morph_cols) == 0:
    print("No morphology marker columns found.")
    morph_summary = pd.DataFrame()
else:
    # Convert to numeric just in case
    for c in available_morph_cols:
        cell_by_gene_adata.obs[c] = pd.to_numeric(cell_by_gene_adata.obs[c], errors="coerce")

    morph_summary = (
        cell_by_gene_adata.obs
        .groupby("Assigned_CosMx_Cell_Type")[available_morph_cols]
        .mean()
    )

    display(morph_summary)

    print("\nMedian morphology marker values by assigned cell type:")
    morph_median_summary = (
        cell_by_gene_adata.obs
        .groupby("Assigned_CosMx_Cell_Type")[available_morph_cols]
        .median()
    )
    display(morph_median_summary)

# ------------------------------------------------------------
# 8. Expected morphology checks
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPECTED MORPHOLOGY SANITY CHECKS")
print("=" * 80)

sanity_rows = []

if "Mean.PanCK" in cell_by_gene_adata.obs.columns:
    panck_by_type = cell_by_gene_adata.obs.groupby("Assigned_CosMx_Cell_Type")["Mean.PanCK"].mean()
    top_panck = panck_by_type.sort_values(ascending=False).head(5)

    sanity_rows.append({
        "check": "Highest Mean.PanCK should include Epithelial cells",
        "top_cell_types": list(top_panck.index),
        "top_values": [float(x) for x in top_panck.values],
        "passes_simple_check": "Epithelial cells" in list(top_panck.index),
    })

if "Mean.CD45" in cell_by_gene_adata.obs.columns:
    cd45_by_type = cell_by_gene_adata.obs.groupby("Assigned_CosMx_Cell_Type")["Mean.CD45"].mean()
    top_cd45 = cd45_by_type.sort_values(ascending=False).head(6)

    immune_labels = {"T lymphocytes", "B lymphocytes", "Myeloid cells", "NK cells", "MAST cells"}
    sanity_rows.append({
        "check": "Highest Mean.CD45 should be immune-like cell types",
        "top_cell_types": list(top_cd45.index),
        "top_values": [float(x) for x in top_cd45.values],
        "passes_simple_check": len(set(top_cd45.index).intersection(immune_labels)) >= 3,
    })

if "Mean.CD3" in cell_by_gene_adata.obs.columns:
    cd3_by_type = cell_by_gene_adata.obs.groupby("Assigned_CosMx_Cell_Type")["Mean.CD3"].mean()
    top_cd3 = cd3_by_type.sort_values(ascending=False).head(5)

    sanity_rows.append({
        "check": "Highest Mean.CD3 should include T lymphocytes",
        "top_cell_types": list(top_cd3.index),
        "top_values": [float(x) for x in top_cd3.values],
        "passes_simple_check": "T lymphocytes" in list(top_cd3.index),
    })

sanity_df = pd.DataFrame(sanity_rows)
display(sanity_df)

# ------------------------------------------------------------
# 9. Inspect suspicious groups
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SUSPICIOUS / AMBIGUOUS GROUP INSPECTION")
print("=" * 80)

suspicious_labels = [
    "Oligodendrocytes",
    "Undetermined",
    "Low_confidence",
]

available_suspicious = [
    x for x in suspicious_labels
    if x in set(cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].astype(str))
]

if len(available_suspicious) == 0:
    print("No suspicious labels found.")
    suspicious_summary = pd.DataFrame()
else:
    suspicious_cols = [
        "celltype_marker_score_best",
        "celltype_marker_score_second",
        "celltype_marker_score_margin",
    ] + marker_score_cols + available_morph_cols

    suspicious_summary = (
        cell_by_gene_adata.obs[
            cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].isin(available_suspicious)
        ]
        .groupby("Assigned_CosMx_Cell_Type")[suspicious_cols]
        .mean()
    )

    display(suspicious_summary)

# ------------------------------------------------------------
# 10. Suggested label refinements, but do not apply automatically
# ------------------------------------------------------------
# This creates a suggested mapping table that we can inspect before applying.

print("\n" + "=" * 80)
print("SUGGESTED LABEL REFINEMENT TABLE")
print("=" * 80)

all_labels = sorted(cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].astype(str).unique())

suggested_rows = []

for label in all_labels:
    suggested = label
    reason = "keep"

    if label == "Undetermined":
        suggested = "Low_confidence"
        reason = "reference label is not a biological cell type"

    if label == "Oligodendrocytes":
        suggested = "Low_confidence"
        reason = "unexpected in lung tissue; inspect before keeping"

    suggested_rows.append({
        "current_label": label,
        "suggested_label": suggested,
        "reason": reason,
    })

suggested_refinement_df = pd.DataFrame(suggested_rows)
display(suggested_refinement_df)

print(
    "\nNote: This cell does NOT apply these refinements automatically. "
    "We will decide in the next cell whether to merge labels."
)

# ------------------------------------------------------------
# 11. Save validation outputs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVE CELL 9 VALIDATION OUTPUTS")
print("=" * 80)

marker_summary_path = WORK_DIR / "cosmx_cell9_marker_score_summary_by_celltype.csv"
top_marker_match_path = WORK_DIR / "cosmx_cell9_top_marker_program_by_celltype.csv"
morph_summary_path = WORK_DIR / "cosmx_cell9_morphology_summary_by_celltype.csv"
sanity_path = WORK_DIR / "cosmx_cell9_morphology_sanity_checks.csv"
suggested_refinement_path = WORK_DIR / "cosmx_cell9_suggested_label_refinement.csv"
adata_path = WORK_DIR / "cosmx_cell9_adata_with_validation_scores.h5ad"
summary_path = WORK_DIR / "cosmx_cell9_validation_summary.json"

marker_summary.to_csv(marker_summary_path)
top_marker_match_df.to_csv(top_marker_match_path, index=False)

if len(available_morph_cols) > 0:
    morph_summary.to_csv(morph_summary_path)
else:
    pd.DataFrame().to_csv(morph_summary_path, index=False)

sanity_df.to_csv(sanity_path, index=False)
suggested_refinement_df.to_csv(suggested_refinement_path, index=False)

cell_by_gene_adata.write_h5ad(adata_path)

cell9_summary = {
    "sample_name": SAMPLE_NAME,
    "n_cells": int(cell_by_gene_adata.n_obs),
    "n_genes": int(cell_by_gene_adata.n_vars),
    "assigned_cell_type_counts": {
        str(k): int(v)
        for k, v in cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].value_counts().items()
    },
    "marker_score_columns": marker_score_cols,
    "available_morphology_columns": available_morph_cols,
    "sanity_checks": sanity_df.to_dict(orient="records"),
    "suspicious_labels_present": available_suspicious,
    "marker_summary_path": str(marker_summary_path),
    "top_marker_match_path": str(top_marker_match_path),
    "morph_summary_path": str(morph_summary_path),
    "sanity_checks_path": str(sanity_path),
    "suggested_refinement_path": str(suggested_refinement_path),
    "adata_with_validation_scores_path": str(adata_path),
}

with open(summary_path, "w") as f:
    json.dump(cell9_summary, f, indent=2)

print("Saved marker summary:")
print(f"  {marker_summary_path}")

print("Saved top marker-program table:")
print(f"  {top_marker_match_path}")

print("Saved morphology summary:")
print(f"  {morph_summary_path}")

print("Saved sanity checks:")
print(f"  {sanity_path}")

print("Saved suggested refinement table:")
print(f"  {suggested_refinement_path}")

print("Saved AnnData with validation scores:")
print(f"  {adata_path}")

print("Saved Cell 9 summary:")
print(f"  {summary_path}")

print("\nCOSMX CELL 9 finished successfully.")
print("Next step: decide whether to refine labels, then attach final cell types to molecule table.")

In [ ]:
# ============================================================
# COSMX CELL 10 — REFINE COSMX CELL-TYPE LABELS
# Purpose:
#   Create final CosMx cell-type labels before attaching labels to molecules.
#
# User decision:
#   Low_confidence  -> Unlabeled
#   Undetermined    -> Unlabeled
#   Oligodendrocytes -> keep as Oligodendrocytes
#
# This follows the Xenium-style logic where uncertain cells are kept
# but labeled as Unlabeled instead of being removed.
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path

print("=" * 80)
print("COSMX CELL 10 — REFINE COSMX CELL-TYPE LABELS")
print("=" * 80)

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load Cell 9 AnnData if needed
# ------------------------------------------------------------

if "cell_by_gene_adata" not in globals():
    adata_path = WORK_DIR / "cosmx_cell9_adata_with_validation_scores.h5ad"
    if not adata_path.exists():
        raise FileNotFoundError(
            "cell_by_gene_adata not found and Cell 9 checkpoint is missing. "
            "Run Cell 9 first."
        )
    cell_by_gene_adata = sc.read_h5ad(adata_path)

print("Current AnnData:")
print(cell_by_gene_adata)

if "Assigned_CosMx_Cell_Type" not in cell_by_gene_adata.obs.columns:
    raise KeyError(
        "Assigned_CosMx_Cell_Type column not found. Run Cell 8/9 first."
    )

# ------------------------------------------------------------
# 2. Show current labels before refinement
# ------------------------------------------------------------

print("\nCurrent assigned cell-type distribution:")
before_counts = cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].value_counts()
display(before_counts.reset_index().rename(
    columns={"index": "Assigned_CosMx_Cell_Type", "Assigned_CosMx_Cell_Type": "n_cells"}
))

# ------------------------------------------------------------
# 3. Apply user-specified label refinement
# ------------------------------------------------------------

label_map = {
    "Low_confidence": "Unlabeled",
    "Undetermined": "Unlabeled",

    # Keep Oligodendrocytes as requested
    "Oligodendrocytes": "Oligodendrocytes",
}

original_labels = cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type"].astype(str)

final_labels = original_labels.map(lambda x: label_map.get(x, x))

cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type_before_refinement"] = original_labels.values
cell_by_gene_adata.obs["Final_CosMx_Cell_Type"] = final_labels.values

# Also create a boolean flag for unlabeled/uncertain cells.
cell_by_gene_adata.obs["is_unlabeled_final"] = (
    cell_by_gene_adata.obs["Final_CosMx_Cell_Type"].astype(str) == "Unlabeled"
)

# Keep Oligodendrocytes flag for later inspection.
cell_by_gene_adata.obs["is_oligodendrocyte_label"] = (
    cell_by_gene_adata.obs["Final_CosMx_Cell_Type"].astype(str) == "Oligodendrocytes"
)

# ------------------------------------------------------------
# 4. Show final label distribution
# ------------------------------------------------------------

print("\nFinal refined CosMx cell-type distribution:")
final_counts = cell_by_gene_adata.obs["Final_CosMx_Cell_Type"].value_counts()
final_counts_df = final_counts.reset_index()
final_counts_df.columns = ["Final_CosMx_Cell_Type", "n_cells"]
final_counts_df["fraction"] = final_counts_df["n_cells"] / cell_by_gene_adata.n_obs
display(final_counts_df)

print("\nBefore vs after refinement:")
transition_df = (
    pd.crosstab(
        cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type_before_refinement"],
        cell_by_gene_adata.obs["Final_CosMx_Cell_Type"],
    )
)

display(transition_df)

# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

n_unlabeled = int(cell_by_gene_adata.obs["is_unlabeled_final"].sum())
n_oligo = int(cell_by_gene_adata.obs["is_oligodendrocyte_label"].sum())

print("\nSanity check:")
print(f"Total cells                  : {cell_by_gene_adata.n_obs:,}")
print(f"Final Unlabeled cells         : {n_unlabeled:,}")
print(f"Final Oligodendrocytes cells  : {n_oligo:,}")

expected_unlabeled = int(
    before_counts.get("Low_confidence", 0)
    + before_counts.get("Undetermined", 0)
)

print(f"Expected Unlabeled from Low_confidence + Undetermined: {expected_unlabeled:,}")

if n_unlabeled != expected_unlabeled:
    raise ValueError(
        "Unlabeled count does not match Low_confidence + Undetermined count. "
        "Check label refinement logic."
    )

if n_oligo != int(before_counts.get("Oligodendrocytes", 0)):
    raise ValueError(
        "Oligodendrocytes count changed unexpectedly. "
        "You requested to keep Oligodendrocytes as-is."
    )

# ------------------------------------------------------------
# 6. Save refined AnnData and label table
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVE CELL 10 OUTPUTS")
print("=" * 80)

adata_path = WORK_DIR / "cosmx_cell10_adata_with_final_celltypes.h5ad"
label_table_path = WORK_DIR / "cosmx_cell10_final_celltype_labels.csv"
summary_path = WORK_DIR / "cosmx_cell10_label_refinement_summary.json"

cell_by_gene_adata.write_h5ad(adata_path)

label_table = cell_by_gene_adata.obs[
    [
        "Assigned_CosMx_Cell_Type_raw",
        "Assigned_CosMx_Cell_Type_before_refinement",
        "Final_CosMx_Cell_Type",
        "is_unlabeled_final",
        "is_oligodendrocyte_label",
        "celltype_marker_score_best",
        "celltype_marker_score_second",
        "celltype_marker_score_margin",
    ]
].copy()

label_table.index.name = "cell_id"
label_table.to_csv(label_table_path)

summary = {
    "sample_name": SAMPLE_NAME,
    "n_cells": int(cell_by_gene_adata.n_obs),
    "refinement_rule": {
        "Low_confidence": "Unlabeled",
        "Undetermined": "Unlabeled",
        "Oligodendrocytes": "Oligodendrocytes",
    },
    "before_counts": {str(k): int(v) for k, v in before_counts.items()},
    "final_counts": {str(k): int(v) for k, v in final_counts.items()},
    "n_final_unlabeled": int(n_unlabeled),
    "n_final_oligodendrocytes": int(n_oligo),
    "adata_path": str(adata_path),
    "label_table_path": str(label_table_path),
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved AnnData with final cell types:")
print(f"  {adata_path}")

print("Saved final label table:")
print(f"  {label_table_path}")

print("Saved summary:")
print(f"  {summary_path}")

print("\nCOSMX CELL 10 finished successfully.")
print("Next step: attach Final_CosMx_Cell_Type to the clean molecule table.")

In [ ]:
# ============================================================
# COSMX CELL 11 — ATTACH FINAL CELL-TYPE LABELS TO MOLECULE TABLE
# Purpose:
#   Add Final_CosMx_Cell_Type to every cleaned molecule/transcript row.
#
# This corresponds to the Xenium step:
#   NEW CODE FOR TRANSCRIPTS FILE'S CELL TYPE LABELING
#
# Input:
#   - Clean molecule parquet chunks from Replacement Cell 5
#   - Final cell labels from Cell 10
#
# Output:
#   - Clean molecule parquet chunks with Final_CosMx_Cell_Type
#   - Updated AnnData checkpoint
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path

print("=" * 80)
print("COSMX CELL 11 — ATTACH FINAL CELL-TYPE LABELS TO MOLECULE TABLE")
print("=" * 80)

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load Cell 10 AnnData if needed
# ------------------------------------------------------------

if "cell_by_gene_adata" not in globals():
    adata_path = WORK_DIR / "cosmx_cell10_adata_with_final_celltypes.h5ad"
    if not adata_path.exists():
        raise FileNotFoundError(
            "cell_by_gene_adata not found and Cell 10 checkpoint is missing. "
            "Run Cell 10 first."
        )
    cell_by_gene_adata = sc.read_h5ad(adata_path)

print("Current AnnData:")
print(cell_by_gene_adata)

if "Final_CosMx_Cell_Type" not in cell_by_gene_adata.obs.columns:
    raise KeyError(
        "Final_CosMx_Cell_Type column not found. Run Cell 10 first."
    )

# ------------------------------------------------------------
# 2. Build cell_id -> final cell type mapping
# ------------------------------------------------------------

celltype_map = (
    cell_by_gene_adata.obs["Final_CosMx_Cell_Type"]
    .astype(str)
    .to_dict()
)

raw_celltype_map = (
    cell_by_gene_adata.obs["Assigned_CosMx_Cell_Type_raw"]
    .astype(str)
    .to_dict()
    if "Assigned_CosMx_Cell_Type_raw" in cell_by_gene_adata.obs.columns
    else {}
)

print("\nCell-type mapping summary:")
print(f"Cells in AnnData label map: {len(celltype_map):,}")

print("\nFinal cell-type distribution:")
display(cell_by_gene_adata.obs["Final_CosMx_Cell_Type"].value_counts().reset_index())

# ------------------------------------------------------------
# 3. Find clean molecule chunks from Replacement Cell 5
# ------------------------------------------------------------

manifest_path = WORK_DIR / "cosmx_cell5_clean_molecule_chunks_manifest_SAFE.json"

if not manifest_path.exists():
    raise FileNotFoundError(
        f"Clean molecule chunk manifest not found: {manifest_path}. "
        "Run Replacement Cell 5 first."
    )

with open(manifest_path, "r") as f:
    manifest = json.load(f)

clean_chunk_paths = [Path(p) for p in manifest["clean_chunk_paths"]]

if not clean_chunk_paths:
    raise FileNotFoundError("No clean molecule chunks listed in manifest.")

for p in clean_chunk_paths:
    if not p.exists():
        raise FileNotFoundError(f"Clean molecule chunk missing: {p}")

print("\nClean molecule chunks found:")
for p in clean_chunk_paths:
    print(f"  {p} ({p.stat().st_size / 1e9:.2f} GB)")

# ------------------------------------------------------------
# 4. Prepare output folder
# ------------------------------------------------------------

LABELED_MOLECULE_DIR = WORK_DIR / "clean_shared_molecule_parquet_chunks_with_celltypes"
LABELED_MOLECULE_DIR.mkdir(parents=True, exist_ok=True)

# Remove old labeled chunks if rerunning.
for old in LABELED_MOLECULE_DIR.glob("clean_molecules_with_celltypes_chunk_*.parquet"):
    old.unlink()

# ------------------------------------------------------------
# 5. Attach final cell type to each molecule
# ------------------------------------------------------------

summary = {
    "input_molecule_rows": 0,
    "output_molecule_rows": 0,
    "missing_celltype_rows": 0,
    "unlabeled_molecule_rows": 0,
    "oligodendrocyte_molecule_rows": 0,
}

labeled_chunk_paths = []

print("\nAttaching final cell types to molecule table...")

for k, chunk_path in enumerate(clean_chunk_paths):
    print(f"\nProcessing clean molecule chunk {k + 1}/{len(clean_chunk_paths)}:")
    print(f"  {chunk_path}")

    mol = pd.read_parquet(chunk_path)

    summary["input_molecule_rows"] += len(mol)

    mol["cell_id"] = mol["cell_id"].astype(str)

    # Attach final cell type.
    mol["Final_CosMx_Cell_Type"] = mol["cell_id"].map(celltype_map)

    # Attach raw/reference-transfer label for traceability.
    if raw_celltype_map:
        mol["Assigned_CosMx_Cell_Type_raw"] = mol["cell_id"].map(raw_celltype_map)

    missing_mask = mol["Final_CosMx_Cell_Type"].isna()
    n_missing = int(missing_mask.sum())

    summary["missing_celltype_rows"] += n_missing

    if n_missing > 0:
        print(f"  WARNING: {n_missing:,} molecules missing final cell type.")
        print("  These rows will be removed because every clean molecule should belong to a labeled cell.")
        mol = mol.loc[~missing_mask].copy()

    mol["Final_CosMx_Cell_Type"] = mol["Final_CosMx_Cell_Type"].astype(str)

    # For convenience, create Xenium-style generic column name too.
    mol["cell_type"] = mol["Final_CosMx_Cell_Type"]

    # Flags
    mol["is_unlabeled_final"] = mol["Final_CosMx_Cell_Type"].eq("Unlabeled")
    mol["is_oligodendrocyte_label"] = mol["Final_CosMx_Cell_Type"].eq("Oligodendrocytes")

    summary["unlabeled_molecule_rows"] += int(mol["is_unlabeled_final"].sum())
    summary["oligodendrocyte_molecule_rows"] += int(mol["is_oligodendrocyte_label"].sum())
    summary["output_molecule_rows"] += len(mol)

    print(f"  Input molecules             : {len(pd.read_parquet(chunk_path, columns=['cell_id'])):,}")
    print(f"  Output molecules            : {len(mol):,}")
    print(f"  Missing cell-type molecules : {n_missing:,}")
    print(f"  Unlabeled molecules         : {mol['is_unlabeled_final'].sum():,}")

    # Save labeled molecule chunk
    out_path = LABELED_MOLECULE_DIR / f"clean_molecules_with_celltypes_chunk_{k:04d}.parquet"
    mol.to_parquet(out_path, index=False)
    labeled_chunk_paths.append(str(out_path))

    del mol
    gc.collect()

# ------------------------------------------------------------
# 6. Summarize molecule counts by final cell type
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MOLECULE CELL-TYPE SUMMARY")
print("=" * 80)

molecule_type_counts = {}

for p in labeled_chunk_paths:
    tmp = pd.read_parquet(p, columns=["Final_CosMx_Cell_Type"])
    vc = tmp["Final_CosMx_Cell_Type"].value_counts()
    for label, count in vc.items():
        molecule_type_counts[str(label)] = molecule_type_counts.get(str(label), 0) + int(count)
    del tmp
    gc.collect()

molecule_type_counts_df = pd.DataFrame({
    "Final_CosMx_Cell_Type": list(molecule_type_counts.keys()),
    "n_molecules": list(molecule_type_counts.values()),
}).sort_values("n_molecules", ascending=False)

molecule_type_counts_df["fraction"] = (
    molecule_type_counts_df["n_molecules"] / molecule_type_counts_df["n_molecules"].sum()
)

display(molecule_type_counts_df)

# ------------------------------------------------------------
# 7. Sanity checks
# ------------------------------------------------------------

print("\nSanity checks:")
print(f"Input clean molecule rows      : {summary['input_molecule_rows']:,}")
print(f"Output labeled molecule rows   : {summary['output_molecule_rows']:,}")
print(f"Missing cell-type rows removed : {summary['missing_celltype_rows']:,}")
print(f"Unlabeled molecule rows        : {summary['unlabeled_molecule_rows']:,}")
print(f"Oligodendrocyte molecule rows  : {summary['oligodendrocyte_molecule_rows']:,}")

if summary["missing_celltype_rows"] > 0:
    frac_missing = summary["missing_celltype_rows"] / max(summary["input_molecule_rows"], 1)
    if frac_missing > 0.001:
        raise ValueError(
            "Too many clean molecules were missing cell-type labels. "
            "Check cell_id matching between AnnData and molecule table."
        )

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVE CELL 11 OUTPUTS")
print("=" * 80)

labeled_manifest_path = WORK_DIR / "cosmx_cell11_labeled_molecule_chunks_manifest.json"
molecule_type_counts_path = WORK_DIR / "cosmx_cell11_molecule_counts_by_final_celltype.csv"
adata_path = WORK_DIR / "cosmx_cell11_adata_final_celltypes.h5ad"
summary_path = WORK_DIR / "cosmx_cell11_attach_celltypes_summary.json"

labeled_manifest = {
    "labeled_molecule_dir": str(LABELED_MOLECULE_DIR),
    "labeled_chunk_paths": labeled_chunk_paths,
    "n_labeled_chunks": len(labeled_chunk_paths),
}

with open(labeled_manifest_path, "w") as f:
    json.dump(labeled_manifest, f, indent=2)

molecule_type_counts_df.to_csv(molecule_type_counts_path, index=False)

cell_by_gene_adata.write_h5ad(adata_path)

cell11_summary = {
    "sample_name": SAMPLE_NAME,
    "n_cells": int(cell_by_gene_adata.n_obs),
    "n_genes": int(cell_by_gene_adata.n_vars),
    "input_molecule_rows": int(summary["input_molecule_rows"]),
    "output_labeled_molecule_rows": int(summary["output_molecule_rows"]),
    "missing_celltype_rows": int(summary["missing_celltype_rows"]),
    "unlabeled_molecule_rows": int(summary["unlabeled_molecule_rows"]),
    "oligodendrocyte_molecule_rows": int(summary["oligodendrocyte_molecule_rows"]),
    "labeled_manifest_path": str(labeled_manifest_path),
    "molecule_type_counts_path": str(molecule_type_counts_path),
    "adata_path": str(adata_path),
}

with open(summary_path, "w") as f:
    json.dump(cell11_summary, f, indent=2)

print("Saved labeled molecule manifest:")
print(f"  {labeled_manifest_path}")

print("Saved molecule counts by final cell type:")
print(f"  {molecule_type_counts_path}")

print("Saved AnnData:")
print(f"  {adata_path}")

print("Saved summary:")
print(f"  {summary_path}")

print("\nCOSMX CELL 11 finished successfully.")
print("Next step: save final Step-4-ready clean files.")

In [ ]:
# ============================================================
# COSMX CELL 11.5 — ALIGN REFERENCE LABELS WITH FINAL COSMX LABELS
# Purpose:
#   Rename GSE131907 reference label:
#       Undetermined -> Unlabeled
#
# Why:
#   In CosMx Cell 10, we merged:
#       Low_confidence + Undetermined -> Unlabeled
#
#   To avoid mismatch during reference-profile construction,
#   the scRNA reference should also use "Unlabeled" instead of "Undetermined".
#
# This cell does NOT remove any cells.
# It only renames the reference label.
# ============================================================

import os
import gc
import json
import pandas as pd
import numpy as np
import scanpy as sc
from pathlib import Path

print("=" * 80)
print("COSMX CELL 11.5 — ALIGN REFERENCE LABELS WITH FINAL COSMX LABELS")
print("=" * 80)

WORK_DIR = Path("/content/cosmx_pipeline_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load reference AnnData from Cell 7
# ------------------------------------------------------------

ref_adata_path = WORK_DIR / "cosmx_cell7_gse131907_reference_shared_951.h5ad"

if "ref_adata" in globals():
    annotated_ref_adata = ref_adata
elif "adata_ref_shared" in globals():
    annotated_ref_adata = adata_ref_shared
else:
    if not ref_adata_path.exists():
        raise FileNotFoundError(
            f"Reference AnnData not found: {ref_adata_path}. "
            "Run Cell 7 first."
        )
    annotated_ref_adata = sc.read_h5ad(ref_adata_path)

print("Reference AnnData before relabeling:")
print(annotated_ref_adata)

if "Reference_Cell_Type" not in annotated_ref_adata.obs.columns:
    raise KeyError(
        "Reference_Cell_Type column not found in reference AnnData. "
        "Check Cell 7 output."
    )

# ------------------------------------------------------------
# 2. Show original reference cell-type distribution
# ------------------------------------------------------------

print("\nOriginal reference cell-type distribution:")
before_counts = annotated_ref_adata.obs["Reference_Cell_Type"].astype(str).value_counts()
display(before_counts.reset_index().rename(
    columns={"index": "Reference_Cell_Type", "Reference_Cell_Type": "n_cells"}
))

# ------------------------------------------------------------
# 3. Rename Undetermined -> Unlabeled
# ------------------------------------------------------------

annotated_ref_adata.obs["Reference_Cell_Type_original"] = (
    annotated_ref_adata.obs["Reference_Cell_Type"].astype(str).values
)

annotated_ref_adata.obs["Reference_Cell_Type_Final"] = (
    annotated_ref_adata.obs["Reference_Cell_Type"]
    .astype(str)
    .replace({
        "Undetermined": "Unlabeled"
    })
    .values
)

# For convenience, overwrite Reference_Cell_Type too.
# From this point forward, Reference_Cell_Type means the final aligned label.
annotated_ref_adata.obs["Reference_Cell_Type"] = (
    annotated_ref_adata.obs["Reference_Cell_Type_Final"].astype(str).values
)

# ------------------------------------------------------------
# 4. Show final reference cell-type distribution
# ------------------------------------------------------------

print("\nFinal reference cell-type distribution after renaming:")
after_counts = annotated_ref_adata.obs["Reference_Cell_Type"].astype(str).value_counts()
after_counts_df = after_counts.reset_index()
after_counts_df.columns = ["Reference_Cell_Type", "n_cells"]
after_counts_df["fraction"] = after_counts_df["n_cells"] / annotated_ref_adata.n_obs
display(after_counts_df)

print("\nBefore vs after reference relabeling:")
transition_df = pd.crosstab(
    annotated_ref_adata.obs["Reference_Cell_Type_original"],
    annotated_ref_adata.obs["Reference_Cell_Type"],
)
display(transition_df)

# ------------------------------------------------------------
# 5. Check compatibility with final CosMx labels
# ------------------------------------------------------------

cosmx_adata_path = WORK_DIR / "cosmx_cell11_adata_final_celltypes.h5ad"

if "cell_by_gene_adata" not in globals():
    if not cosmx_adata_path.exists():
        raise FileNotFoundError(
            f"CosMx AnnData with final labels not found: {cosmx_adata_path}. "
            "Run Cell 11 first."
        )
    cell_by_gene_adata = sc.read_h5ad(cosmx_adata_path)

if "Final_CosMx_Cell_Type" not in cell_by_gene_adata.obs.columns:
    raise KeyError("Final_CosMx_Cell_Type not found in CosMx AnnData. Run Cell 10/11 first.")

cosmx_labels = sorted(cell_by_gene_adata.obs["Final_CosMx_Cell_Type"].astype(str).unique())
ref_labels = sorted(annotated_ref_adata.obs["Reference_Cell_Type"].astype(str).unique())

print("\nFinal CosMx labels:")
print(cosmx_labels)

print("\nFinal reference labels:")
print(ref_labels)

labels_in_cosmx_not_ref = sorted(set(cosmx_labels) - set(ref_labels))
labels_in_ref_not_cosmx = sorted(set(ref_labels) - set(cosmx_labels))

print("\nLabel compatibility check:")
print(f"Labels in CosMx but not reference: {labels_in_cosmx_not_ref}")
print(f"Labels in reference but not CosMx: {labels_in_ref_not_cosmx}")

# Note:
# Low_confidence is not expected in reference anymore.
# Unlabeled should exist in both.
if "Unlabeled" not in ref_labels:
    raise ValueError("Unlabeled label was not created in reference.")

if "Undetermined" in ref_labels:
    raise ValueError("Undetermined still exists in final reference labels.")

# ------------------------------------------------------------
# 6. Save aligned reference AnnData
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVE CELL 11.5 OUTPUTS")
print("=" * 80)

aligned_ref_path = WORK_DIR / "cosmx_cell11_5_gse131907_reference_shared_951_aligned_labels.h5ad"
summary_path = WORK_DIR / "cosmx_cell11_5_reference_label_alignment_summary.json"

annotated_ref_adata.write_h5ad(aligned_ref_path)

summary = {
    "sample_name": SAMPLE_NAME,
    "reference": "GSE131907",
    "rule_applied": {
        "Undetermined": "Unlabeled"
    },
    "n_reference_cells": int(annotated_ref_adata.n_obs),
    "n_reference_genes": int(annotated_ref_adata.n_vars),
    "before_counts": {str(k): int(v) for k, v in before_counts.items()},
    "after_counts": {str(k): int(v) for k, v in after_counts.items()},
    "cosmx_final_labels": cosmx_labels,
    "reference_final_labels": ref_labels,
    "labels_in_cosmx_not_ref": labels_in_cosmx_not_ref,
    "labels_in_ref_not_cosmx": labels_in_ref_not_cosmx,
    "aligned_reference_path": str(aligned_ref_path),
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved aligned reference AnnData:")
print(f"  {aligned_ref_path}")

print("Saved summary:")
print(f"  {summary_path}")

# Keep useful variable for following cells
ALIGNED_REFERENCE_ADATA_PATH = aligned_ref_path
annotated_ref_adata_aligned = annotated_ref_adata

print("\nCOSMX CELL 11.5 finished successfully.")
print("Next step: save final Step-4-ready clean files.")

In [ ]:
# ============================================================
# COSMX CELL 12 — CLEAN FIXED STEP-4-READY EXPORT TO REAL GOOGLE DRIVE
# ============================================================
# Purpose:
#   Save the final CosMx Step-4-ready files to real Google Drive.
#
# Important fix:
#   This cell mounts Google Drive FIRST, before creating:
#       /content/drive/MyDrive/diffusion/step4_cosmx
#
# Drive folder:
#   /content/drive/MyDrive/diffusion/step4_cosmx
#
# Main Xenium-equivalent filenames:
#   molecules.parquet
#   X_raw_counts.npy
#   annotated_spatial_adata_clean_raw.h5ad
#   clean_preprocessing_summary.json
#
# Additional support files:
#   cell_data.npz
#   scrna_ref.npz
#   scrna_ref.h5ad
#   reference_profiles.csv
#   shared_genes.txt
#   gene_names.npy
#   cell_ids.npy
#   step4_config.json
# ============================================================

import os
import gc
import json
import shutil
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from pathlib import Path

print("=" * 80)
print("COSMX CELL 12 — CLEAN FIXED STEP-4-READY EXPORT TO REAL GOOGLE DRIVE")
print("=" * 80)

# ------------------------------------------------------------
# 0. Define non-Drive local paths first
# ------------------------------------------------------------

WORK_DIR = Path("/content/cosmx_pipeline_work")
LOCAL_EXPORT_DIR = Path("/content/step4_cosmx_exports")

WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working folder      : {WORK_DIR}")
print(f"Local export folder : {LOCAL_EXPORT_DIR}")

# ------------------------------------------------------------
# 1. Fix possible fake /content/drive folder, then mount real Drive
# ------------------------------------------------------------

from google.colab import drive

drive_mount = Path("/content/drive")
fake_backup = Path("/content/drive_local_backup_before_real_mount")

# If /content/drive exists but is NOT a real mount, move it away.
# This fixes the previous issue where Cell 12 created local files under /content/drive
# before Google Drive was actually mounted.
if drive_mount.exists() and not os.path.ismount(str(drive_mount)):
    existing_items = list(drive_mount.glob("*"))

    if len(existing_items) > 0:
        print("\nWARNING: /content/drive exists but is not mounted.")
        print("This is likely a fake local drive folder from the previous failed export.")
        print(f"Moving it to: {fake_backup}")

        if fake_backup.exists():
            shutil.rmtree(fake_backup)

        shutil.move(str(drive_mount), str(fake_backup))

# Recreate clean mount point
drive_mount.mkdir(parents=True, exist_ok=True)

print("\nMounting real Google Drive...")
drive.mount("/content/drive", force_remount=True)

if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive did not mount correctly. Stop here.")

print("Google Drive mounted successfully.")

# NOW it is safe to define and create Drive export folder.
DRIVE_DIR = Path("/content/drive/MyDrive/diffusion/step4_cosmx")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Real Drive export folder: {DRIVE_DIR}")

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def require_file(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path

def ensure_dense(X, dtype=np.float32):
    if sparse.issparse(X):
        return X.toarray().astype(dtype)
    return np.asarray(X, dtype=dtype)

def copy_and_verify(src, dst):
    src = Path(src)
    dst = Path(dst)

    if not src.exists():
        raise FileNotFoundError(f"Source file missing: {src}")

    shutil.copy2(src, dst)

    if not dst.exists():
        raise FileNotFoundError(f"Destination file was not created: {dst}")

    src_size = src.stat().st_size
    dst_size = dst.stat().st_size

    if src_size != dst_size:
        raise ValueError(
            f"File size mismatch after copy:\n"
            f"  src: {src} ({src_size})\n"
            f"  dst: {dst} ({dst_size})"
        )

    print(f"[COPIED] {dst.name:45s} {dst_size / 1e6:,.2f} MB")

# ------------------------------------------------------------
# 3. Load final CosMx AnnData from Cell 11
# ------------------------------------------------------------

adata_path = WORK_DIR / "cosmx_cell11_adata_final_celltypes.h5ad"

if not adata_path.exists():
    fallback = WORK_DIR / "cosmx_cell10_adata_with_final_celltypes.h5ad"
    if fallback.exists():
        adata_path = fallback
    else:
        raise FileNotFoundError(
            "Could not find final CosMx AnnData. Run Cell 10 and Cell 11 first."
        )

print("\nLoading final CosMx AnnData:")
print(f"  {adata_path}")

adata_cosmx = sc.read_h5ad(adata_path)

print(adata_cosmx)

if "Final_CosMx_Cell_Type" not in adata_cosmx.obs.columns:
    raise KeyError("Final_CosMx_Cell_Type missing. Run Cell 10/11 first.")

if adata_cosmx.n_vars != 951:
    raise ValueError(f"Expected 951 shared genes, found {adata_cosmx.n_vars}.")

if adata_cosmx.n_obs < 90_000:
    raise ValueError(f"Expected around 100k cells, found {adata_cosmx.n_obs}.")

# Xenium-compatible generic cell-type column
adata_cosmx.obs["cell_type"] = adata_cosmx.obs["Final_CosMx_Cell_Type"].astype(str)

# Make sure main matrix and raw layer use molecule-derived clean counts
if "raw_molecule_counts_clean" in adata_cosmx.layers:
    adata_cosmx.layers["raw"] = adata_cosmx.layers["raw_molecule_counts_clean"].copy()
    adata_cosmx.X = adata_cosmx.layers["raw"].copy()
elif "raw" in adata_cosmx.layers:
    adata_cosmx.X = adata_cosmx.layers["raw"].copy()
else:
    adata_cosmx.layers["raw"] = adata_cosmx.X.copy()

# ------------------------------------------------------------
# 4. Load labeled molecule chunks from Cell 11
# ------------------------------------------------------------

labeled_manifest_path = require_file(
    WORK_DIR / "cosmx_cell11_labeled_molecule_chunks_manifest.json",
    "Cell 11 labeled molecule manifest"
)

with open(labeled_manifest_path, "r") as f:
    labeled_manifest = json.load(f)

labeled_chunk_paths = [Path(p) for p in labeled_manifest["labeled_chunk_paths"]]

if len(labeled_chunk_paths) == 0:
    raise FileNotFoundError("No labeled molecule chunks found in Cell 11 manifest.")

for p in labeled_chunk_paths:
    require_file(p, "labeled molecule parquet chunk")

print("\nLabeled molecule chunks:")
for p in labeled_chunk_paths:
    print(f"  {p} ({p.stat().st_size / 1e9:.2f} GB)")

# ------------------------------------------------------------
# 5. Export molecules.parquet locally first
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPORTING LOCAL STEP-4 FILES")
print("=" * 80)

local_molecules_path = LOCAL_EXPORT_DIR / "molecules.parquet"

print("\nSaving local molecules.parquet...")

if len(labeled_chunk_paths) == 1:
    shutil.copy2(labeled_chunk_paths[0], local_molecules_path)
else:
    mol_parts = [pd.read_parquet(p) for p in labeled_chunk_paths]
    mol_all = pd.concat(mol_parts, ignore_index=True)
    mol_all.to_parquet(local_molecules_path, index=False)
    del mol_parts, mol_all
    gc.collect()

mol_check = pd.read_parquet(
    local_molecules_path,
    columns=["cell_id", "gene_id", "Final_CosMx_Cell_Type"]
)

print(f"  molecule rows             : {len(mol_check):,}")
print(f"  unique molecule cells      : {mol_check['cell_id'].nunique():,}")
print(f"  unique molecule genes      : {mol_check['gene_id'].nunique():,}")
print(f"  missing final cell type    : {mol_check['Final_CosMx_Cell_Type'].isna().sum():,}")

if len(mol_check) == 0:
    raise ValueError("molecules.parquet is empty.")

if mol_check["Final_CosMx_Cell_Type"].isna().sum() > 0:
    raise ValueError("Some molecules are missing Final_CosMx_Cell_Type.")

if mol_check["gene_id"].nunique() != 951:
    raise ValueError(
        f"Expected 951 genes in molecule table, found {mol_check['gene_id'].nunique()}."
    )

if mol_check["cell_id"].nunique() != adata_cosmx.n_obs:
    raise ValueError(
        f"Molecule unique cells {mol_check['cell_id'].nunique()} does not match AnnData cells {adata_cosmx.n_obs}."
    )

# ------------------------------------------------------------
# 6. Export X_raw_counts.npy
# ------------------------------------------------------------

local_xraw_path = LOCAL_EXPORT_DIR / "X_raw_counts.npy"

print("\nSaving local X_raw_counts.npy...")

X_raw_counts = ensure_dense(adata_cosmx.layers["raw"], dtype=np.float32)

if X_raw_counts.shape != adata_cosmx.shape:
    raise ValueError(
        f"X_raw_counts shape {X_raw_counts.shape} does not match AnnData shape {adata_cosmx.shape}."
    )

np.save(local_xraw_path, X_raw_counts)

raw_sum = float(X_raw_counts.sum(dtype=np.float64))

print(f"  X_raw_counts shape: {X_raw_counts.shape}")
print(f"  X_raw_counts sum  : {raw_sum:,.0f}")

if abs(raw_sum - float(len(mol_check))) > 100:
    raise ValueError(
        "X_raw_counts sum does not match molecule row count closely. "
        "Check raw/molecule consistency."
    )

# ------------------------------------------------------------
# 7. Export annotated_spatial_adata_clean_raw.h5ad
# ------------------------------------------------------------

local_adata_export_path = LOCAL_EXPORT_DIR / "annotated_spatial_adata_clean_raw.h5ad"

print("\nSaving local annotated_spatial_adata_clean_raw.h5ad...")

adata_cosmx.uns["platform"] = "CosMx"
adata_cosmx.uns["sample_name"] = "Lung5_Rep1"
adata_cosmx.uns["reference"] = "GSE131907"
adata_cosmx.uns["step4_ready"] = True
adata_cosmx.uns["n_shared_genes"] = int(adata_cosmx.n_vars)
adata_cosmx.uns["cell_type_column"] = "cell_type"
adata_cosmx.uns["final_cell_type_column"] = "Final_CosMx_Cell_Type"

adata_cosmx.write_h5ad(local_adata_export_path)

print(f"  saved: {local_adata_export_path}")

# ------------------------------------------------------------
# 8. Export cell_data.npz
# ------------------------------------------------------------

local_cell_data_path = LOCAL_EXPORT_DIR / "cell_data.npz"

print("\nSaving local cell_data.npz...")

obs = adata_cosmx.obs.copy()

required_geom_cols = [
    "center_x_global_px",
    "center_y_global_px",
    "cell_area_px",
    "label_cell_area_px",
    "label_nuclear_area_px",
    "label_membrane_area_px",
    "label_cytoplasm_area_px",
    "label_extracellular_area_px",
    "label_nuclear_frac",
    "label_membrane_frac",
    "label_cytoplasm_frac",
    "label_extracellular_frac",
]

missing_geom_cols = [c for c in required_geom_cols if c not in obs.columns]
if missing_geom_cols:
    raise KeyError(f"Missing geometry columns in AnnData.obs: {missing_geom_cols}")

cell_ids_arr = np.asarray(adata_cosmx.obs_names.astype(str), dtype="U64")
centroids_arr = obs[["center_x_global_px", "center_y_global_px"]].to_numpy(dtype=np.float32)

np.savez_compressed(
    local_cell_data_path,
    cell_ids=cell_ids_arr,
    centroids=centroids_arr,

    center_x_global_px=obs["center_x_global_px"].to_numpy(dtype=np.float32),
    center_y_global_px=obs["center_y_global_px"].to_numpy(dtype=np.float32),
    center_x_um=obs["center_x_um"].to_numpy(dtype=np.float32) if "center_x_um" in obs.columns else np.full(adata_cosmx.n_obs, np.nan, dtype=np.float32),
    center_y_um=obs["center_y_um"].to_numpy(dtype=np.float32) if "center_y_um" in obs.columns else np.full(adata_cosmx.n_obs, np.nan, dtype=np.float32),

    cell_area_px=obs["cell_area_px"].to_numpy(dtype=np.float32),
    cell_area_um2=obs["cell_area_um2"].to_numpy(dtype=np.float32) if "cell_area_um2" in obs.columns else np.full(adata_cosmx.n_obs, np.nan, dtype=np.float32),

    label_cell_area_px=obs["label_cell_area_px"].to_numpy(dtype=np.float32),
    label_nuclear_area_px=obs["label_nuclear_area_px"].to_numpy(dtype=np.float32),
    label_membrane_area_px=obs["label_membrane_area_px"].to_numpy(dtype=np.float32),
    label_cytoplasm_area_px=obs["label_cytoplasm_area_px"].to_numpy(dtype=np.float32),
    label_extracellular_area_px=obs["label_extracellular_area_px"].to_numpy(dtype=np.float32),

    label_nuclear_frac=obs["label_nuclear_frac"].to_numpy(dtype=np.float32),
    label_membrane_frac=obs["label_membrane_frac"].to_numpy(dtype=np.float32),
    label_cytoplasm_frac=obs["label_cytoplasm_frac"].to_numpy(dtype=np.float32),
    label_extracellular_frac=obs["label_extracellular_frac"].to_numpy(dtype=np.float32),

    fov=obs["fov"].astype(int).to_numpy(dtype=np.int32),
    cell_ID_original=obs["cell_ID_original"].astype(str).to_numpy(dtype="U64"),
    cell_type=obs["cell_type"].astype(str).to_numpy(dtype="U64"),
    Final_CosMx_Cell_Type=obs["Final_CosMx_Cell_Type"].astype(str).to_numpy(dtype="U64"),
)

cell_data_check = np.load(local_cell_data_path, allow_pickle=False)
print(f"  cell_data.npz cells: {len(cell_data_check['cell_ids']):,}")
print(f"  fields: {list(cell_data_check.files)}")

# ------------------------------------------------------------
# 9. Export aligned scRNA reference: scrna_ref.npz and scrna_ref.h5ad
# ------------------------------------------------------------

print("\nSaving local scrna_ref.npz and scrna_ref.h5ad...")

aligned_ref_path = WORK_DIR / "cosmx_cell11_5_gse131907_reference_shared_951_aligned_labels.h5ad"
fallback_ref_path = WORK_DIR / "cosmx_cell7_gse131907_reference_shared_951.h5ad"

if aligned_ref_path.exists():
    ref_path = aligned_ref_path
else:
    ref_path = fallback_ref_path

require_file(ref_path, "aligned/reference AnnData")

ref_adata = sc.read_h5ad(ref_path)

if "Reference_Cell_Type" not in ref_adata.obs.columns:
    raise KeyError("Reference_Cell_Type missing from reference AnnData.")

# Safety alignment: rename Undetermined to Unlabeled
ref_adata.obs["Reference_Cell_Type"] = (
    ref_adata.obs["Reference_Cell_Type"]
    .astype(str)
    .replace({"Undetermined": "Unlabeled"})
)

if "Undetermined" in set(ref_adata.obs["Reference_Cell_Type"].astype(str)):
    raise ValueError("Reference still contains Undetermined. Run Cell 11.5 first.")

# Reorder reference genes to match CosMx genes
if not pd.Index(ref_adata.var_names.astype(str)).equals(pd.Index(adata_cosmx.var_names.astype(str))):
    ref_adata = ref_adata[:, adata_cosmx.var_names].copy()

local_scrna_npz_path = LOCAL_EXPORT_DIR / "scrna_ref.npz"
local_scrna_h5ad_path = LOCAL_EXPORT_DIR / "scrna_ref.h5ad"

ref_expression = ensure_dense(ref_adata.X, dtype=np.float32)
ref_cell_types = ref_adata.obs["Reference_Cell_Type"].astype(str).to_numpy(dtype="U64")
ref_gene_names = np.asarray(ref_adata.var_names.astype(str), dtype="U64")
ref_cell_type_names = np.asarray(sorted(pd.unique(ref_cell_types)), dtype="U64")

np.savez_compressed(
    local_scrna_npz_path,
    expression=ref_expression,
    cell_types=ref_cell_types,
    gene_names=ref_gene_names,
    cell_type_names=ref_cell_type_names,
)

ref_adata.write_h5ad(local_scrna_h5ad_path)

print(f"  reference cells     : {ref_adata.n_obs:,}")
print(f"  reference genes     : {ref_adata.n_vars:,}")
print(f"  reference cell types: {list(ref_cell_type_names)}")

# ------------------------------------------------------------
# 10. Export reference_profiles.csv
# ------------------------------------------------------------

local_ref_profiles_path = LOCAL_EXPORT_DIR / "reference_profiles.csv"

print("\nSaving local reference_profiles.csv...")

profile_rows = []

X_ref_csr = ref_adata.X.tocsr() if sparse.issparse(ref_adata.X) else sparse.csr_matrix(ref_adata.X)

for ct in ref_cell_type_names:
    mask = ref_adata.obs["Reference_Cell_Type"].astype(str).values == str(ct)
    if int(mask.sum()) == 0:
        continue
    mean_expr = np.asarray(X_ref_csr[mask, :].mean(axis=0)).ravel()
    profile_rows.append(pd.Series(mean_expr, index=ref_adata.var_names.astype(str), name=str(ct)))

reference_profiles_df = pd.DataFrame(profile_rows)
reference_profiles_df.to_csv(local_ref_profiles_path)

print(f"  profiles shape: {reference_profiles_df.shape}")

# ------------------------------------------------------------
# 11. Export summary/config/helper files
# ------------------------------------------------------------

print("\nSaving local summary/config/helper files...")

local_summary_path = LOCAL_EXPORT_DIR / "clean_preprocessing_summary.json"
local_shared_genes_path = LOCAL_EXPORT_DIR / "shared_genes.txt"
local_gene_names_path = LOCAL_EXPORT_DIR / "gene_names.npy"
local_cell_ids_path = LOCAL_EXPORT_DIR / "cell_ids.npy"
local_config_path = LOCAL_EXPORT_DIR / "step4_config.json"

final_celltype_counts = (
    adata_cosmx.obs["Final_CosMx_Cell_Type"].astype(str).value_counts().to_dict()
)

molecule_celltype_counts = (
    mol_check["Final_CosMx_Cell_Type"].astype(str).value_counts().to_dict()
)

clean_preprocessing_summary = {
    "platform": "CosMx",
    "sample_name": "Lung5_Rep1",
    "reference": "GSE131907",
    "drive_export_folder": str(DRIVE_DIR),
    "local_export_folder": str(LOCAL_EXPORT_DIR),

    "n_cells": int(adata_cosmx.n_obs),
    "n_genes": int(adata_cosmx.n_vars),
    "n_molecules": int(len(mol_check)),
    "n_unique_molecule_cells": int(mol_check["cell_id"].nunique()),
    "n_unique_molecule_genes": int(mol_check["gene_id"].nunique()),

    "cell_type_column": "cell_type",
    "final_cell_type_column": "Final_CosMx_Cell_Type",
    "final_celltype_counts": {str(k): int(v) for k, v in final_celltype_counts.items()},
    "molecule_celltype_counts": {str(k): int(v) for k, v in molecule_celltype_counts.items()},

    "raw_count_matrix_shape": list(X_raw_counts.shape),
    "raw_count_matrix_sum": float(raw_sum),
    "molecule_rows": int(len(mol_check)),
    "raw_count_vs_molecule_row_difference": float(raw_sum - len(mol_check)),

    "genes_are_shared_with_reference": True,
    "n_shared_genes": int(adata_cosmx.n_vars),
    "gene_names": list(map(str, adata_cosmx.var_names)),

    "cell_metadata_present": True,
    "geometry_present": True,
    "compartment_geometry_present": True,
    "reference_labels_aligned": True,
    "undetermined_reference_renamed_to_unlabeled": True,
}

with open(local_summary_path, "w") as f:
    json.dump(clean_preprocessing_summary, f, indent=2)

with open(local_shared_genes_path, "w") as f:
    for g in adata_cosmx.var_names.astype(str):
        f.write(str(g) + "\n")

np.save(local_gene_names_path, np.asarray(adata_cosmx.var_names.astype(str), dtype="U64"))
np.save(local_cell_ids_path, np.asarray(adata_cosmx.obs_names.astype(str), dtype="U64"))

step4_config = {
    "platform": "CosMx",
    "sample_name": "Lung5_Rep1",
    "reference_dataset": "GSE131907",
    "export_folder": str(DRIVE_DIR),

    "files": {
        "molecules": "molecules.parquet",
        "X_raw_counts": "X_raw_counts.npy",
        "annotated_spatial_adata": "annotated_spatial_adata_clean_raw.h5ad",
        "cell_data": "cell_data.npz",
        "scrna_ref_npz": "scrna_ref.npz",
        "scrna_ref_h5ad": "scrna_ref.h5ad",
        "reference_profiles": "reference_profiles.csv",
        "summary": "clean_preprocessing_summary.json",
        "shared_genes": "shared_genes.txt",
        "gene_names": "gene_names.npy",
        "cell_ids": "cell_ids.npy",
    },

    "cell_type_columns": {
        "spatial_cell_type": "cell_type",
        "final_cosmx_cell_type": "Final_CosMx_Cell_Type",
        "reference_cell_type": "Reference_Cell_Type",
    },

    "counts": {
        "n_cells": int(adata_cosmx.n_obs),
        "n_genes": int(adata_cosmx.n_vars),
        "n_molecules": int(len(mol_check)),
        "raw_count_sum": float(raw_sum),
    },

    "preprocessing_choices": {
        "kept_only_shared_genes_with_reference": True,
        "n_shared_genes": int(adata_cosmx.n_vars),
        "negative_probes_removed_from_biological_matrix": True,
        "unassigned_cell_ID_0_transcripts_removed": True,
        "cell_ID_0_pseudocells_removed": True,
        "molecule_derived_counts_used_as_raw": True,
        "cell_type_labels_attached_to_molecules": True,
        "undetermined_and_low_confidence_merged_to_unlabeled": True,
        "oligodendrocytes_kept_as_requested": True,
    },

    "geometry": {
        "coordinate_units": "pixels",
        "pixel_size_um": 0.18,
        "centroid_columns": ["center_x_global_px", "center_y_global_px"],
        "compartment_columns": [
            "label_nuclear_frac",
            "label_membrane_frac",
            "label_cytoplasm_frac",
            "label_extracellular_frac",
        ],
    },
}

with open(local_config_path, "w") as f:
    json.dump(step4_config, f, indent=2)

# ------------------------------------------------------------
# 12. Copy local exports to REAL Google Drive
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COPYING LOCAL EXPORTS TO REAL GOOGLE DRIVE")
print("=" * 80)

files_to_copy = [
    "molecules.parquet",
    "X_raw_counts.npy",
    "annotated_spatial_adata_clean_raw.h5ad",
    "cell_data.npz",
    "scrna_ref.npz",
    "scrna_ref.h5ad",
    "reference_profiles.csv",
    "clean_preprocessing_summary.json",
    "shared_genes.txt",
    "gene_names.npy",
    "cell_ids.npy",
    "step4_config.json",
]

for fname in files_to_copy:
    copy_and_verify(LOCAL_EXPORT_DIR / fname, DRIVE_DIR / fname)

# Add a tiny visible marker file
marker_path = DRIVE_DIR / "_EXPORT_COMPLETE_VISIBLE_TEST.txt"
with open(marker_path, "w") as f:
    f.write("CosMx Step-4-ready files were exported to real Google Drive successfully.\n")
    f.write(f"Folder: {DRIVE_DIR}\n")

print(f"[WRITTEN] {marker_path.name}")

# ------------------------------------------------------------
# 13. Final verification directly from REAL Google Drive
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL REAL DRIVE VERIFICATION")
print("=" * 80)

expected_drive_files = files_to_copy + ["_EXPORT_COMPLETE_VISIBLE_TEST.txt"]

missing_drive_files = []

for fname in expected_drive_files:
    p = DRIVE_DIR / fname
    if not p.exists():
        missing_drive_files.append(fname)
        print(f"[MISSING] {fname}")
    else:
        print(f"[OK] {fname:45s} {p.stat().st_size / 1e6:,.2f} MB")

if missing_drive_files:
    raise FileNotFoundError(f"Missing files in real Drive export: {missing_drive_files}")

print("\nLoading quick checks from real Drive...")

drive_xraw = np.load(DRIVE_DIR / "X_raw_counts.npy")
drive_mol_check = pd.read_parquet(
    DRIVE_DIR / "molecules.parquet",
    columns=["cell_id", "gene_id", "Final_CosMx_Cell_Type"]
)
drive_adata_check = sc.read_h5ad(DRIVE_DIR / "annotated_spatial_adata_clean_raw.h5ad")

print(f"Drive X_raw_counts shape      : {drive_xraw.shape}")
print(f"Drive X_raw_counts sum        : {drive_xraw.sum(dtype=np.float64):,.0f}")
print(f"Drive molecules rows          : {len(drive_mol_check):,}")
print(f"Drive molecule unique cells   : {drive_mol_check['cell_id'].nunique():,}")
print(f"Drive molecule unique genes   : {drive_mol_check['gene_id'].nunique():,}")
print(f"Drive missing cell types      : {drive_mol_check['Final_CosMx_Cell_Type'].isna().sum():,}")
print(f"Drive AnnData shape           : {drive_adata_check.shape}")

if drive_xraw.shape != drive_adata_check.shape:
    raise ValueError("Drive X_raw_counts shape does not match Drive AnnData shape.")

if drive_mol_check["gene_id"].nunique() != drive_adata_check.n_vars:
    raise ValueError("Number of molecule genes does not match AnnData genes.")

if drive_mol_check["cell_id"].nunique() != drive_adata_check.n_obs:
    raise ValueError("Number of molecule cells does not match AnnData cells.")

if abs(float(drive_xraw.sum(dtype=np.float64)) - float(len(drive_mol_check))) > 100:
    raise ValueError("Drive raw count sum does not match molecule row count closely.")

# ------------------------------------------------------------
# 14. Flush writes to Drive
# ------------------------------------------------------------

print("\nFlushing Google Drive writes...")
drive.flush_and_unmount()

print("\n" + "=" * 80)
print("COSMX STEP-4-READY EXPORT COMPLETE")
print("=" * 80)

print("Files were copied to REAL Google Drive folder:")
print("  My Drive → diffusion → step4_cosmx")
print()
print("Expected visible marker file:")
print("  _EXPORT_COMPLETE_VISIBLE_TEST.txt")
print()
print("Final dataset summary:")
print(f"  cells      : {drive_adata_check.n_obs:,}")
print(f"  genes      : {drive_adata_check.n_vars:,}")
print(f"  molecules  : {len(drive_mol_check):,}")
print(f"  raw counts : {drive_xraw.sum(dtype=np.float64):,.0f}")

print("\nCOSMX CELL 12 finished successfully.")

In [ ]:
# ============================================================
# COSMX STEP-4-READY FILE LOADING + SANITY CHECK
# Purpose:
#   Load final preprocessed CosMx files from Google Drive,
#   preview the contents, and verify that the dataset is ready
#   for cell-level denoising.
#
# Expected Drive folder:
#   /content/drive/MyDrive/diffusion/step4_cosmx
#
# Main files:
#   molecules.parquet
#   X_raw_counts.npy
#   annotated_spatial_adata_clean_raw.h5ad
#   cell_data.npz
#   scrna_ref.npz
#   scrna_ref.h5ad
#   reference_profiles.csv
#   clean_preprocessing_summary.json
#   step4_config.json
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

from scipy import sparse

print("=" * 80)
print("COSMX STEP-4-READY FILE LOADING + SANITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

STEP4_DIR = Path("/content/drive/MyDrive/diffusion/step4_cosmx")

print("\nStep-4-ready folder:")
print(STEP4_DIR)
print("Exists:", STEP4_DIR.exists())

if not STEP4_DIR.exists():
    raise FileNotFoundError(
        f"Folder not found: {STEP4_DIR}\n"
        "Check that the files were saved under My Drive/diffusion/step4_cosmx."
    )

# ------------------------------------------------------------
# 2. Check expected files
# ------------------------------------------------------------

expected_files = {
    "molecules": STEP4_DIR / "molecules.parquet",
    "X_raw_counts": STEP4_DIR / "X_raw_counts.npy",
    "adata": STEP4_DIR / "annotated_spatial_adata_clean_raw.h5ad",
    "cell_data": STEP4_DIR / "cell_data.npz",
    "scrna_ref_npz": STEP4_DIR / "scrna_ref.npz",
    "scrna_ref_h5ad": STEP4_DIR / "scrna_ref.h5ad",
    "reference_profiles": STEP4_DIR / "reference_profiles.csv",
    "summary": STEP4_DIR / "clean_preprocessing_summary.json",
    "shared_genes": STEP4_DIR / "shared_genes.txt",
    "gene_names": STEP4_DIR / "gene_names.npy",
    "cell_ids": STEP4_DIR / "cell_ids.npy",
    "config": STEP4_DIR / "step4_config.json",
}

print("\n" + "=" * 80)
print("FILE PRESENCE CHECK")
print("=" * 80)

missing_files = []

for name, path in expected_files.items():
    if path.exists():
        print(f"[OK] {path.name:45s} {path.stat().st_size / 1e6:,.2f} MB")
    else:
        print(f"[MISSING] {path.name}")
        missing_files.append(path.name)

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

print("\nAll expected files are present.")

# ------------------------------------------------------------
# 3. Load summary/config first
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD SUMMARY + CONFIG")
print("=" * 80)

with open(expected_files["summary"], "r") as f:
    clean_summary = json.load(f)

with open(expected_files["config"], "r") as f:
    step4_config = json.load(f)

print("\nclean_preprocessing_summary.json:")
print(json.dumps({
    "platform": clean_summary.get("platform"),
    "sample_name": clean_summary.get("sample_name"),
    "reference": clean_summary.get("reference"),
    "n_cells": clean_summary.get("n_cells"),
    "n_genes": clean_summary.get("n_genes"),
    "n_molecules": clean_summary.get("n_molecules"),
    "raw_count_matrix_shape": clean_summary.get("raw_count_matrix_shape"),
    "raw_count_matrix_sum": clean_summary.get("raw_count_matrix_sum"),
    "cell_type_column": clean_summary.get("cell_type_column"),
}, indent=2))

print("\nstep4_config.json:")
print(json.dumps({
    "platform": step4_config.get("platform"),
    "sample_name": step4_config.get("sample_name"),
    "reference_dataset": step4_config.get("reference_dataset"),
    "counts": step4_config.get("counts"),
    "cell_type_columns": step4_config.get("cell_type_columns"),
}, indent=2))

# ------------------------------------------------------------
# 4. Load AnnData
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD SPATIAL AnnData")
print("=" * 80)

try:
    import scanpy as sc
except ImportError:
    print("scanpy not found. Installing scanpy...")
    !pip install -q scanpy anndata
    import scanpy as sc

adata_spatial = sc.read_h5ad(expected_files["adata"])

print("\nSpatial AnnData:")
print(adata_spatial)

print("\nobs columns:")
print(list(adata_spatial.obs.columns))

print("\nvar columns:")
print(list(adata_spatial.var.columns))

print("\nLayers:")
print(list(adata_spatial.layers.keys()))

print("\nFirst 5 cells:")
display(adata_spatial.obs.head())

print("\nFirst 10 genes:")
print(adata_spatial.var_names[:10].tolist())

# ------------------------------------------------------------
# 5. Load raw count matrix
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD X_raw_counts.npy")
print("=" * 80)

X_raw_counts = np.load(expected_files["X_raw_counts"])

print(f"X_raw_counts shape: {X_raw_counts.shape}")
print(f"X_raw_counts dtype : {X_raw_counts.dtype}")
print(f"X_raw_counts sum   : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Min count          : {X_raw_counts.min():.0f}")
print(f"Max count          : {X_raw_counts.max():.0f}")

# ------------------------------------------------------------
# 6. Load molecule table preview and selected columns
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD MOLECULE TABLE PREVIEW")
print("=" * 80)

molecule_path = expected_files["molecules"]

# Preview first rows
mol_preview = pd.read_parquet(molecule_path).head(10)

print("\nMolecule table preview:")
display(mol_preview)

print("\nMolecule columns:")
print(list(mol_preview.columns))

# Load only key columns for sanity checks.
key_molecule_cols = [
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "CellComp",
    "overlaps_nucleus",
    "Final_CosMx_Cell_Type",
    "cell_type",
]

available_cols = pd.read_parquet(molecule_path).head(1).columns.tolist()
key_molecule_cols = [c for c in key_molecule_cols if c in available_cols]

print("\nLoading key molecule columns for checks:")
print(key_molecule_cols)

mol_key = pd.read_parquet(molecule_path, columns=key_molecule_cols)

print(f"\nMolecule rows       : {len(mol_key):,}")
print(f"Unique molecule cells: {mol_key['cell_id'].nunique():,}")
print(f"Unique molecule genes: {mol_key['gene_id'].nunique():,}")

if "Final_CosMx_Cell_Type" in mol_key.columns:
    print(f"Missing final cell type: {mol_key['Final_CosMx_Cell_Type'].isna().sum():,}")

print("\nMolecule cell-type distribution:")
display(mol_key["Final_CosMx_Cell_Type"].value_counts().reset_index())

print("\nMolecule compartment distribution:")
if "CellComp" in mol_key.columns:
    display(mol_key["CellComp"].value_counts(dropna=False).reset_index())

# ------------------------------------------------------------
# 7. Load cell_data.npz
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD CELL GEOMETRY: cell_data.npz")
print("=" * 80)

cell_data = np.load(expected_files["cell_data"], allow_pickle=False)

print("cell_data.npz fields:")
print(list(cell_data.files))

cell_data_cell_ids = cell_data["cell_ids"]

print(f"\ncell_data cells: {len(cell_data_cell_ids):,}")

if "centroids" in cell_data.files:
    print(f"centroids shape: {cell_data['centroids'].shape}")
    print("First 5 centroids:")
    print(cell_data["centroids"][:5])

if "label_nuclear_frac" in cell_data.files:
    print("\nNuclear fraction summary:")
    nuc_frac = cell_data["label_nuclear_frac"]
    print(pd.Series(nuc_frac).describe())

# ------------------------------------------------------------
# 8. Load reference data
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD scRNA REFERENCE")
print("=" * 80)

scrna_ref = np.load(expected_files["scrna_ref_npz"], allow_pickle=True)

print("scrna_ref.npz fields:")
print(list(scrna_ref.files))

ref_expression = scrna_ref["expression"]
ref_cell_types = scrna_ref["cell_types"]
ref_gene_names = scrna_ref["gene_names"]
ref_cell_type_names = scrna_ref["cell_type_names"]

print(f"\nReference expression shape: {ref_expression.shape}")
print(f"Reference cell types      : {len(ref_cell_types):,}")
print(f"Reference genes           : {len(ref_gene_names):,}")
print(f"Reference cell type names : {list(ref_cell_type_names)}")

print("\nReference cell-type distribution:")
display(pd.Series(ref_cell_types).value_counts().reset_index())

adata_ref = sc.read_h5ad(expected_files["scrna_ref_h5ad"])

print("\nReference AnnData:")
print(adata_ref)

if "Reference_Cell_Type" in adata_ref.obs.columns:
    print("\nReference AnnData cell-type distribution:")
    display(adata_ref.obs["Reference_Cell_Type"].value_counts().reset_index())

# ------------------------------------------------------------
# 9. Load reference profiles
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD REFERENCE PROFILES")
print("=" * 80)

reference_profiles = pd.read_csv(expected_files["reference_profiles"], index_col=0)

print(f"reference_profiles shape: {reference_profiles.shape}")
print("\nReference profile rows/cell types:")
print(reference_profiles.index.tolist())

print("\nFirst 5 genes in profiles:")
display(reference_profiles.iloc[:, :5])

# ------------------------------------------------------------
# 10. Load helper gene/cell files
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOAD HELPER GENE/CELL FILES")
print("=" * 80)

with open(expected_files["shared_genes"], "r") as f:
    shared_genes_txt = [line.strip() for line in f if line.strip()]

gene_names_npy = np.load(expected_files["gene_names"])
cell_ids_npy = np.load(expected_files["cell_ids"])

print(f"shared_genes.txt genes: {len(shared_genes_txt):,}")
print(f"gene_names.npy genes  : {len(gene_names_npy):,}")
print(f"cell_ids.npy cells    : {len(cell_ids_npy):,}")

print("\nFirst 10 shared genes:")
print(shared_genes_txt[:10])

print("\nFirst 10 cell IDs:")
print(cell_ids_npy[:10].tolist())

# ------------------------------------------------------------
# 11. Sanity checks
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SANITY CHECKS")
print("=" * 80)

sanity_results = []

def add_check(name, condition, details=""):
    status = "PASS" if bool(condition) else "FAIL"
    sanity_results.append({
        "check": name,
        "status": status,
        "details": details,
    })
    print(f"[{status}] {name}")
    if details:
        print(f"       {details}")

# Shape checks
add_check(
    "AnnData shape matches X_raw_counts shape",
    adata_spatial.shape == X_raw_counts.shape,
    f"AnnData={adata_spatial.shape}, X_raw_counts={X_raw_counts.shape}"
)

add_check(
    "AnnData has expected 951 shared genes",
    adata_spatial.n_vars == 951,
    f"n_vars={adata_spatial.n_vars}"
)

add_check(
    "AnnData has expected ~100k cells",
    adata_spatial.n_obs > 90000,
    f"n_obs={adata_spatial.n_obs}"
)

# Molecule consistency
add_check(
    "Molecule unique cells match AnnData cells",
    mol_key["cell_id"].nunique() == adata_spatial.n_obs,
    f"molecule cells={mol_key['cell_id'].nunique()}, AnnData cells={adata_spatial.n_obs}"
)

add_check(
    "Molecule unique genes match AnnData genes",
    mol_key["gene_id"].nunique() == adata_spatial.n_vars,
    f"molecule genes={mol_key['gene_id'].nunique()}, AnnData genes={adata_spatial.n_vars}"
)

add_check(
    "Raw count sum matches number of molecule rows",
    abs(float(X_raw_counts.sum(dtype=np.float64)) - float(len(mol_key))) <= 100,
    f"raw_sum={X_raw_counts.sum(dtype=np.float64):,.0f}, molecule_rows={len(mol_key):,}"
)

if "Final_CosMx_Cell_Type" in mol_key.columns:
    add_check(
        "No molecules missing final cell type",
        mol_key["Final_CosMx_Cell_Type"].isna().sum() == 0,
        f"missing={mol_key['Final_CosMx_Cell_Type'].isna().sum():,}"
    )

# Gene set checks
adata_gene_set = set(adata_spatial.var_names.astype(str))
mol_gene_set = set(mol_key["gene_id"].astype(str))
shared_gene_set = set(shared_genes_txt)
ref_gene_set = set(map(str, ref_gene_names))

add_check(
    "Molecule genes equal AnnData genes",
    mol_gene_set == adata_gene_set,
    f"mol_only={len(mol_gene_set - adata_gene_set)}, adata_only={len(adata_gene_set - mol_gene_set)}"
)

add_check(
    "shared_genes.txt equals AnnData genes",
    shared_gene_set == adata_gene_set,
    f"shared_only={len(shared_gene_set - adata_gene_set)}, adata_only={len(adata_gene_set - shared_gene_set)}"
)

add_check(
    "Reference genes equal AnnData genes",
    ref_gene_set == adata_gene_set,
    f"ref_only={len(ref_gene_set - adata_gene_set)}, adata_only={len(adata_gene_set - ref_gene_set)}"
)

# Cell ID checks
adata_cell_set = set(adata_spatial.obs_names.astype(str))
mol_cell_set = set(mol_key["cell_id"].astype(str))
cell_data_cell_set = set(map(str, cell_data_cell_ids))
cell_ids_npy_set = set(map(str, cell_ids_npy))

add_check(
    "Molecule cells equal AnnData cells",
    mol_cell_set == adata_cell_set,
    f"mol_only={len(mol_cell_set - adata_cell_set)}, adata_only={len(adata_cell_set - mol_cell_set)}"
)

add_check(
    "cell_data.npz cells equal AnnData cells",
    cell_data_cell_set == adata_cell_set,
    f"cell_data_only={len(cell_data_cell_set - adata_cell_set)}, adata_only={len(adata_cell_set - cell_data_cell_set)}"
)

add_check(
    "cell_ids.npy cells equal AnnData cells",
    cell_ids_npy_set == adata_cell_set,
    f"cell_ids_only={len(cell_ids_npy_set - adata_cell_set)}, adata_only={len(adata_cell_set - cell_ids_npy_set)}"
)

# Label checks
required_obs_cols = [
    "cell_type",
    "Final_CosMx_Cell_Type",
    "center_x_global_px",
    "center_y_global_px",
    "label_nuclear_frac",
    "label_cytoplasm_frac",
    "label_membrane_frac",
]

missing_obs_cols = [c for c in required_obs_cols if c not in adata_spatial.obs.columns]

add_check(
    "Required AnnData.obs columns are present",
    len(missing_obs_cols) == 0,
    f"missing={missing_obs_cols}"
)

add_check(
    "Unlabeled exists as final uncertainty category",
    "Unlabeled" in set(adata_spatial.obs["cell_type"].astype(str)),
    f"cell types={sorted(adata_spatial.obs['cell_type'].astype(str).unique().tolist())}"
)

add_check(
    "Reference does not contain Undetermined",
    "Undetermined" not in set(map(str, ref_cell_types)),
    f"reference cell types={list(ref_cell_type_names)}"
)

add_check(
    "Reference contains Unlabeled",
    "Unlabeled" in set(map(str, ref_cell_types)),
    f"reference cell types={list(ref_cell_type_names)}"
)

# Raw layer check
add_check(
    "AnnData contains raw layer",
    "raw" in adata_spatial.layers.keys(),
    f"layers={list(adata_spatial.layers.keys())}"
)

if "raw" in adata_spatial.layers.keys():
    raw_layer_sum = (
        adata_spatial.layers["raw"].sum()
        if not sparse.issparse(adata_spatial.layers["raw"])
        else adata_spatial.layers["raw"].sum()
    )
    raw_layer_sum = float(raw_layer_sum)

    add_check(
        "AnnData raw layer sum matches X_raw_counts sum",
        abs(raw_layer_sum - float(X_raw_counts.sum(dtype=np.float64))) <= 100,
        f"raw_layer_sum={raw_layer_sum:,.0f}, X_raw_sum={X_raw_counts.sum(dtype=np.float64):,.0f}"
    )

# Reference profile check
add_check(
    "Reference profiles have same gene count as AnnData",
    reference_profiles.shape[1] == adata_spatial.n_vars,
    f"profiles genes={reference_profiles.shape[1]}, AnnData genes={adata_spatial.n_vars}"
)

spatial_celltypes = set(adata_spatial.obs["cell_type"].astype(str).unique())
profile_celltypes = set(reference_profiles.index.astype(str))

add_check(
    "All spatial cell types have reference profiles",
    spatial_celltypes.issubset(profile_celltypes),
    f"missing profiles for={sorted(spatial_celltypes - profile_celltypes)}"
)

# ------------------------------------------------------------
# 12. Final report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL SANITY CHECK REPORT")
print("=" * 80)

sanity_df = pd.DataFrame(sanity_results)
display(sanity_df)

n_failed = int((sanity_df["status"] == "FAIL").sum())

print("\nFinal status:")
if n_failed == 0:
    print("✅ CosMx Step-4-ready files passed all sanity checks.")
    print("✅ The dataset is ready for the cell-level denoising stage.")
else:
    print(f"⚠️ {n_failed} sanity checks failed.")
    print("Please inspect the failed checks before starting denoising.")

print("\nMost important files for denoising:")
print("1. annotated_spatial_adata_clean_raw.h5ad")
print("   - main spatial AnnData object")
print("   - contains cells × genes matrix, metadata, geometry, and cell types")
print("2. X_raw_counts.npy")
print("   - molecule-derived raw count matrix")
print("3. molecules.parquet")
print("   - cleaned molecule/transcript table with coordinates and cell types")
print("4. cell_data.npz")
print("   - geometry/centroid/compartment data")
print("5. scrna_ref.h5ad or scrna_ref.npz")
print("   - paired GSE131907 scRNA-seq reference restricted to shared genes")
print("6. reference_profiles.csv")
print("   - average reference expression profile per cell type")
print("7. step4_config.json")
print("   - reproducibility/configuration metadata")

print("\nDataset summary:")
print(f"Spatial cells     : {adata_spatial.n_obs:,}")
print(f"Shared genes      : {adata_spatial.n_vars:,}")
print(f"Molecule rows     : {len(mol_key):,}")
print(f"Raw count sum     : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Reference cells   : {ref_expression.shape[0]:,}")
print(f"Reference genes   : {ref_expression.shape[1]:,}")

print("\nSpatial cell-type distribution:")
display(adata_spatial.obs["cell_type"].value_counts().reset_index())

print("\nReference cell-type distribution:")
display(pd.Series(ref_cell_types).value_counts().reset_index())

In [ ]:
# ============================================================
# COMPACT COSMX STEP-4 FILE CONSISTENCY CHECK
# Checks:
#   molecules.parquet vs annotated_spatial_adata_clean_raw.h5ad
#   X_raw_counts.npy vs AnnData
#   cell_data.npz vs AnnData
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

try:
    import scanpy as sc
except ImportError:
    !pip install -q scanpy anndata
    import scanpy as sc

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

STEP4_DIR = Path("/content/drive/MyDrive/diffusion/step4_cosmx")

mol_path = STEP4_DIR / "molecules.parquet"
adata_path = STEP4_DIR / "annotated_spatial_adata_clean_raw.h5ad"
xraw_path = STEP4_DIR / "X_raw_counts.npy"
cell_data_path = STEP4_DIR / "cell_data.npz"

print("=" * 80)
print("COSMX STEP-4 FILE CONSISTENCY CHECK")
print("=" * 80)

for p in [mol_path, adata_path, xraw_path, cell_data_path]:
    print(f"{p.name:45s} exists={p.exists()} size={p.stat().st_size / 1e6 if p.exists() else 0:.2f} MB")

# Load files
adata = sc.read_h5ad(adata_path)
X_raw = np.load(xraw_path)
cell_data = np.load(cell_data_path, allow_pickle=False)

# Load only needed molecule columns
mol = pd.read_parquet(mol_path, columns=["cell_id", "gene_id"])

# Extract IDs
mol_cells = set(mol["cell_id"].astype(str).unique())
mol_genes = set(mol["gene_id"].astype(str).unique())

adata_cells = set(adata.obs_names.astype(str))
adata_genes = set(adata.var_names.astype(str))

cell_data_cells = set(map(str, cell_data["cell_ids"]))

# Summary
print("\n" + "=" * 80)
print("COUNTS")
print("=" * 80)

print(f"Molecule table rows        : {len(mol):,}")
print(f"Molecule unique cells      : {len(mol_cells):,}")
print(f"Molecule unique genes      : {len(mol_genes):,}")
print(f"AnnData shape              : {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
print(f"X_raw_counts shape         : {X_raw.shape[0]:,} cells × {X_raw.shape[1]:,} genes")
print(f"cell_data.npz cells        : {len(cell_data_cells):,}")
print(f"X_raw_counts sum           : {X_raw.sum(dtype=np.float64):,.0f}")

# Checks
checks = []

def check(name, condition, detail):
    checks.append((name, bool(condition), detail))

check(
    "Molecule unique cells == AnnData cells",
    mol_cells == adata_cells,
    f"mol={len(mol_cells):,}, adata={len(adata_cells):,}, mol_only={len(mol_cells-adata_cells):,}, adata_only={len(adata_cells-mol_cells):,}"
)

check(
    "Molecule unique genes == AnnData genes",
    mol_genes == adata_genes,
    f"mol={len(mol_genes):,}, adata={len(adata_genes):,}, mol_only={len(mol_genes-adata_genes):,}, adata_only={len(adata_genes-mol_genes):,}"
)

check(
    "X_raw_counts shape == AnnData shape",
    X_raw.shape == adata.shape,
    f"X_raw={X_raw.shape}, adata={adata.shape}"
)

check(
    "cell_data.npz cells == AnnData cells",
    cell_data_cells == adata_cells,
    f"cell_data={len(cell_data_cells):,}, adata={len(adata_cells):,}, cell_data_only={len(cell_data_cells-adata_cells):,}, adata_only={len(adata_cells-cell_data_cells):,}"
)

check(
    "X_raw_counts sum == molecule table rows",
    abs(float(X_raw.sum(dtype=np.float64)) - float(len(mol))) <= 100,
    f"X_raw_sum={X_raw.sum(dtype=np.float64):,.0f}, molecule_rows={len(mol):,}"
)

print("\n" + "=" * 80)
print("CHECK RESULTS")
print("=" * 80)

all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"[{status}] {name}")
    print(f"       {detail}")

print("\n" + "=" * 80)
if all_pass:
    print("✅ All core consistency checks passed. CosMx files are aligned.")
else:
    print("⚠️ Some checks failed. Inspect the FAIL rows above before denoising.")
print("=" * 80)

=============================CELL LEVEL DENOISING==================================

In [ ]:
# ============================================================
# COSMX DENOISING SETUP — Load Step-4-ready files using Xenium-compatible names
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

STEP4_DIR = Path("/content/drive/MyDrive/diffusion/step4_cosmx")

annotated_spatial_adata = sc.read_h5ad(STEP4_DIR / "annotated_spatial_adata_clean_raw.h5ad")
annotated_ref_adata = sc.read_h5ad(STEP4_DIR / "scrna_ref.h5ad")
X_raw_counts = np.load(STEP4_DIR / "X_raw_counts.npy")

# Make spatial cell-type column compatible
if "cell_type" not in annotated_spatial_adata.obs.columns:
    if "Final_CosMx_Cell_Type" in annotated_spatial_adata.obs.columns:
        annotated_spatial_adata.obs["cell_type"] = annotated_spatial_adata.obs["Final_CosMx_Cell_Type"].astype(str)
    else:
        raise KeyError("No cell_type or Final_CosMx_Cell_Type found in spatial AnnData.")

# Make reference cell-type column compatible
if "cell_type" not in annotated_ref_adata.obs.columns:
    if "Reference_Cell_Type" in annotated_ref_adata.obs.columns:
        annotated_ref_adata.obs["cell_type"] = annotated_ref_adata.obs["Reference_Cell_Type"].astype(str)
    else:
        raise KeyError("No cell_type or Reference_Cell_Type found in reference AnnData.")

# Make CosMx coordinates compatible with Xenium denoising code
if "x_centroid" not in annotated_spatial_adata.obs.columns:
    if "center_x_global_px" in annotated_spatial_adata.obs.columns:
        annotated_spatial_adata.obs["x_centroid"] = annotated_spatial_adata.obs["center_x_global_px"].astype(float)
    else:
        raise KeyError("No x_centroid or center_x_global_px found.")

if "y_centroid" not in annotated_spatial_adata.obs.columns:
    if "center_y_global_px" in annotated_spatial_adata.obs.columns:
        annotated_spatial_adata.obs["y_centroid"] = annotated_spatial_adata.obs["center_y_global_px"].astype(float)
    else:
        raise KeyError("No y_centroid or center_y_global_px found.")

# Ensure matrix is molecule-derived raw counts
if "raw" in annotated_spatial_adata.layers:
    annotated_spatial_adata.X = annotated_spatial_adata.layers["raw"].copy()

print("CosMx denoising setup complete.")
print("Spatial:", annotated_spatial_adata.shape)
print("Reference:", annotated_ref_adata.shape)
print("X_raw_counts:", X_raw_counts.shape)
print("Spatial cell types:", annotated_spatial_adata.obs["cell_type"].value_counts())
print("Reference cell types:", annotated_ref_adata.obs["cell_type"].value_counts())
print("Coordinate columns ready:", "x_centroid" in annotated_spatial_adata.obs, "y_centroid" in annotated_spatial_adata.obs)

In [ ]:
# ==============================================================================
# CELL 1: Install dependencies
# ==============================================================================
!pip install scanpy scikit-learn tqdm --quiet

In [ ]:
# ==============================================================================
# CELL 2: Verify datasets are ready
# ==============================================================================

print("Checking datasets...")
print(f"\nXenium: {annotated_spatial_adata.n_obs:,} cells × {annotated_spatial_adata.n_vars} genes")
print(f"scRNA-seq: {annotated_ref_adata.n_obs:,} cells × {annotated_ref_adata.n_vars} genes")

shared = set(annotated_spatial_adata.var_names) & set(annotated_ref_adata.var_names)
print(f"Shared genes: {len(shared)}")

In [ ]:
# ==============================================================================
# CELL 3: Define all helper functions and classes
# ==============================================================================

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import nbinom
from scipy.special import gammaln
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree, NearestNeighbors
from tqdm import tqdm

def ensure_dense(X):
    """Convert sparse to dense if needed."""
    if sparse.issparse(X):
        return X.toarray()
    return X

def log_normalize(X):
    """Library size normalization + log transform."""
    if X.max() > 100:
        lib_sizes = X.sum(axis=1, keepdims=True)
        X_norm = X / (lib_sizes + 1e-8) * 1e4
        return np.log1p(X_norm)
    return X

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import nbinom
from scipy.special import gammaln
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree, NearestNeighbors
from tqdm import tqdm

def ensure_dense(X):
    """Convert sparse to dense if needed."""
    if sparse.issparse(X):
        return X.toarray()
    return X

def log_normalize(X):
    """Library size normalization + log transform."""
    if X.max() > 100:
        lib_sizes = X.sum(axis=1, keepdims=True)
        X_norm = X / (lib_sizes + 1e-8) * 1e4
        return np.log1p(X_norm)
    return X

# -----------------------------------------------------------------------------
# IMPROVEMENT 1: Adaptive Radius Graph
# -----------------------------------------------------------------------------

class AdaptiveRadiusGraph:
    """
    Adaptive neighbor selection based on local density.

    Dense regions → more neighbors
    Sparse regions → fewer neighbors (avoid noise from distant cells)
    """

    def __init__(self, radius=100.0, k_min=5, k_max=30, adaptive_radius=True):
        self.radius = radius
        self.k_min = k_min
        self.k_max = k_max
        self.adaptive_radius = adaptive_radius
        self.tree_ = None
        self.coords_ = None

    def fit(self, spatial_coords):
        """Build spatial index."""
        self.coords_ = spatial_coords
        self.tree_ = BallTree(spatial_coords)

        if self.adaptive_radius:
            counts = self.tree_.query_radius(spatial_coords, r=self.radius,
                                              count_only=True)
            self.local_density_ = counts / (np.pi * self.radius ** 2)
            self.median_density_ = np.median(self.local_density_)
        return self

    def get_neighbors(self, idx):
        """Get adaptive neighbors for one cell."""
        query_point = self.coords_[idx:idx+1]

        # Adapt radius to local density
        if self.adaptive_radius:
            density_ratio = self.local_density_[idx] / (self.median_density_ + 1e-8)
            adaptive_r = self.radius / np.sqrt(density_ratio + 0.1)
            adaptive_r = np.clip(adaptive_r, self.radius * 0.5, self.radius * 2.0)
        else:
            adaptive_r = self.radius

        # Query neighbors
        indices, distances = self.tree_.query_radius(
            query_point, r=adaptive_r, return_distance=True
        )
        indices = indices[0]
        distances = distances[0]

        # Remove self
        mask = indices != idx
        indices = indices[mask]
        distances = distances[mask]

        # Sort and clip to [k_min, k_max]
        sort_idx = np.argsort(distances)
        indices = indices[sort_idx]
        distances = distances[sort_idx]

        if len(indices) < self.k_min:
            # Fall back to k nearest
            all_dist, all_idx = self.tree_.query(query_point, k=self.k_min + 1)
            indices = all_idx[0, 1:]
            distances = all_dist[0, 1:]
        elif len(indices) > self.k_max:
            indices = indices[:self.k_max]
            distances = distances[:self.k_max]

        # Gaussian weights
        sigma = adaptive_r / 2
        weights = np.exp(-distances ** 2 / (2 * sigma ** 2))
        weights = weights /(weights.sum() + 1e-8)
        return indices, weights


# -----------------------------------------------------------------------------
# IMPROVEMENT 2: Proper NB Likelihood
# -----------------------------------------------------------------------------

def nb_cdf(observed, mu, alpha):
    """
    CDF of Negative Binomial distribution.

    Used to detect dropouts: if P(X ≤ observed) is very low,
    the observation is surprisingly small → likely a dropout.
    """
    mu = np.maximum(mu, 1e-8)
    alpha = np.maximum(alpha, 1e-8)

    r = 1.0 / alpha
    p = r / (r + mu)

    return nbinom.cdf(observed, r, p)


def detect_dropouts_strict(observed, mu_expected, alpha, p_threshold=0.01):
    """
    STRICT dropout detection - only flag values that are CLEARLY dropouts.

    Criteria:
        1. P(X <= observed) < p_threshold (statistically unlikely)
        2. Expected value > 0.5 (gene should be expressed)
        3. Observed < 70% of expected (clearly too low)
    """
    cdf = nb_cdf(observed, mu_expected, alpha)

    is_dropout = (
        (cdf < p_threshold) &           # Statistically unlikely
        (mu_expected > 0.5) &           # Expected is substantial
        (observed < 0.7 * mu_expected)  # Much lower than expected
    )


    return is_dropout, cdf

# -----------------------------------------------------------------------------
# IMPROVEMENT 3: Calibrated Uncertainty
# -----------------------------------------------------------------------------

class UncertaintyCalibrator:
    """
    Calibrate uncertainty using held-out values.

    1. Hide some values
    2. Predict them + uncertainty
    3. Check: |error| ~ predicted uncertainty?
    4. Learn scale factor to achieve proper coverage
    """
    """FIXED: Uses binary search to find scale that achieves target coverage.
    The old version always made coverage WORSE."""

    def __init__(self, holdout_fraction=0.1, target_coverage=0.68):
        self.holdout_fraction = holdout_fraction
        self.target_coverage = target_coverage
        self.scale_factor_ = 1.0
        self.calibration_stats_ = {}

    def create_holdout_mask(self, n_cells, n_genes, seed=42):
        """Create random holdout mask."""
        rng = np.random.RandomState(seed)
        return rng.random((n_cells, n_genes)) < self.holdout_fraction

    def calibrate(self, predicted, uncertainty, true_values, mask, was_corrected):
        """
        Calibrate uncertainty using held-out values.

        Goal: 68% of true values should fall within ±1σ of prediction."""
        """Only calibrate on corrected values."""
        calibration_mask = mask & was_corrected

        if calibration_mask.sum() < 100:
            print("  Warning: Too few corrected values for calibration")
            self.scale_factor_ = 1.0
            self.calibration_stats_ = {'scale_factor': 1.0, 'n_corrected_holdout': calibration_mask.sum()}
            return uncertainty

        pred_h  = predicted[calibration_mask]
        true_h  = true_values[calibration_mask]
        unc_h  = uncertainty[calibration_mask]

        valid = unc_h > 1e-8
        pred_v, true_v, unc_v = pred_h[valid], true_h[valid], unc_h[valid]
        errors = np.abs(pred_v - true_v)

        # Before
        z_before = errors / unc_v
        cov_1s_before = (z_before < 1.0).mean()

        # FIXED: Binary search for optimal scale
        def coverage(scale):
            return (errors / (unc_v * scale) < 1.0).mean()

        low, high = 0.1, 10.0
        for _ in range(50):
            mid = (low + high) / 2
            if coverage(mid) < self.target_coverage:
                high = mid
            else:
                low = mid

        self.scale_factor_ = np.clip(mid, 0.1, 5.0)

        # After
        z_after = errors / (unc_v * self.scale_factor_)
        cov_1s_after = (z_after < 1.0).mean()

        self.calibration_stats_ = {
            'n_corrected_holdout': calibration_mask.sum(),
            'scale_factor': self.scale_factor_,
            'coverage_1sigma_before': cov_1s_before,
            'coverage_1sigma_after': cov_1s_after
        }

        return uncertainty * self.scale_factor_

    def print_report(self):
        s = self.calibration_stats_
        print(f"\n  Calibration on {s['n_corrected_holdout']:,} corrected held-out values")
        print(f"  Scale factor: {s['scale_factor']:.4f}")
        if 'coverage_1sigma_before' in s:
            print(f"  Coverage: {s['coverage_1sigma_before']:.1%} → {s['coverage_1sigma_after']:.1%}")


In [ ]:

# ==============================================================================
# CELL * : Improved cell type mapping
# ==============================================================================

def map_cell_types_improved(xenium_types, ref_types, verbose=True):
    """Fuzzy matching for cell types."""

    synonyms = {
        'cd4_t': ['cd4', 't_helper', 'cd4+'],
        'cd8_t': ['cd8', 'cytotoxic', 'cd8+'],
        'b_cell': ['b_lymphocyte', 'bcell'],
        'macrophage': ['macro', 'monocyte'],
        'endothelial': ['endo', 'vascular'],
        'fibroblast': ['fibro', 'stromal', 'stroma'],
        'tumor': ['cancer', 'malignant', 'invasive'],
        'dcis': ['ductal', 'carcinoma_in_situ'],
        'prolif': ['proliferating', 'cycling']
    }

    def normalize(s):
        return s.lower().strip().replace('-', '_').replace(' ', '_').replace('+', '_plus')

    def find_match(xt, ref_types):
        xt_norm = normalize(xt)

        # Exact
        for rt in ref_types:
            if normalize(rt) == xt_norm:
                return rt

        # Substring
        for rt in ref_types:
            rt_norm = normalize(rt)
            if xt_norm in rt_norm or rt_norm in xt_norm:
                return rt

        # Synonym
        for key, syns in synonyms.items():
            if key in xt_norm or any(s in xt_norm for s in syns):
                for rt in ref_types:
                    rt_norm = normalize(rt)
                    if key in rt_norm or any(s in rt_norm for s in syns):
                        return rt

        # Word overlap
        xt_words = set(xt_norm.replace('_', ' ').split())
        for rt in ref_types:
            rt_words = set(normalize(rt).replace('_', ' ').split())
            if xt_words & rt_words:
                return rt

        return None

    mapping = {}
    unmapped = []

    for xt in xenium_types:
        match = find_match(xt, ref_types)
        if match:
            mapping[xt] = match
        else:
            unmapped.append(xt)

    if verbose:
        print(f"\nCell type mapping: {len(mapping)}/{len(xenium_types)}")
        if unmapped:
            print(f"  Unmapped: {unmapped}")

    return mapping, unmapped


In [ ]:

# ==============================================================================
# CELL 4: Reference profile computation
# ==============================================================================

def compute_reference_profiles(ref_adata, ct_col='cell_type'):
    """Compute mean expression per cell type from scRNA-seq."""
    print("Computing reference profiles...")
    X = ensure_dense(ref_adata.X)
    X = log_normalize(X)

    cell_types = ref_adata.obs[ct_col].values
    profiles = {}

    for ct in np.unique(cell_types):
        mask = cell_types == ct #Creates a Boolean mask that selects only cells of the current type.
        profiles[ct] = {
            'mean': X[mask].mean(axis=0),
            'std': X[mask].std(axis=0),
            'n_cells': mask.sum()
        }
        print(f"  {ct}: {mask.sum()} cells")

    return profiles



In [ ]:
# ==============================================================================
# CELL 7: Main denoising function (v4 - SELECTIVE)
# ==============================================================================

def denoise_v4_selective(xenium_adata, ref_adata,
                         spatial_weight=0.5,
                         reference_weight=0.3,
                         spatial_radius=100.0,
                         k_min=5,
                         k_max=30,
                         n_pca=50,
                         dropout_p_threshold=0.1,
                         shrinkage=0.8,
                         calibrate=True,
                         holdout_frac=0.1):
    """
    SELECTIVE denoising - only correct detected dropouts.

    KEY DIFFERENCE from v3:
        v3: Changed ALL values → destroyed biological signal
        v4: Only changes dropout values → preserves biology
    """
    print("=" * 60)
    print("SELECTIVE SPATIAL DENOISING (v4)")
    print("Only correcting detected dropouts")
    print("=" * 60)
    print(f"  dropout_p_threshold: {dropout_p_threshold}")
    print(f"  shrinkage: {shrinkage}")

    # Shared genes
    shared_genes = list(set(xenium_adata.var_names) & set(ref_adata.var_names))
    print(f"  Shared genes: {len(shared_genes)}")

    xenium_sub = xenium_adata[:, shared_genes].copy()
    ref_sub = ref_adata[:, shared_genes].copy()

    # Reference profiles
    ref_ct_col = None
    for col in ['cell_type', 'celltype', 'cluster', 'Assigned Cell Type']:
        if col in ref_sub.obs.columns:
            ref_ct_col = col
            break

    ref_profiles = compute_reference_profiles(ref_sub, ref_ct_col)

    # Cell type mapping
    xenium_ct_col = 'cell_type' if 'cell_type' in xenium_sub.obs else 'Assigned_Xenium_Cell_Type'
    xenium_types = xenium_sub.obs[xenium_ct_col].unique()
    type_mapping, unmapped = map_cell_types_improved(xenium_types, list(ref_profiles.keys()))

    # Get RAW data
    X = ensure_dense(xenium_sub.X).astype(float)
    n_cells, n_genes = X.shape

    # Holdout
    if calibrate:
        calibrator = UncertaintyCalibrator(holdout_frac)
        holdout_mask = calibrator.create_holdout_mask(n_cells, n_genes)
        X_for_neighbors = X.copy()
        X_for_neighbors[holdout_mask] = 0
        print(f"  Held out {holdout_mask.sum():,} values")
    else:
        X_for_neighbors = X
        holdout_mask = None

    # PCA
    print("  Running PCA...")
    pca = PCA(n_components=n_pca)
    expr_features = pca.fit_transform(log_normalize(X_for_neighbors))

    # Spatial
    x_col = 'x_centroid' if 'x_centroid' in xenium_sub.obs else 'x'
    y_col = 'y_centroid' if 'y_centroid' in xenium_sub.obs else 'y'
    spatial_coords = np.column_stack([xenium_sub.obs[x_col].values, xenium_sub.obs[y_col].values])
    scaler = StandardScaler()
    spatial_norm = scaler.fit_transform(spatial_coords)

    # Hybrid
    scale = np.sqrt(n_pca / 2)
    hybrid = np.hstack([expr_features * (1 - spatial_weight), spatial_norm * spatial_weight * scale])

    # KEY: Start with ORIGINAL data
    X_denoised = X.copy()
    uncertainty = np.zeros_like(X, dtype=float)
    was_corrected = np.zeros_like(X, dtype=bool)

    # Process by cell type
    print("  Detecting and correcting dropouts...")
    cell_types = xenium_sub.obs[xenium_ct_col].values

    for ct in tqdm(np.unique(cell_types), desc="Cell types"):
        ct_mask = cell_types == ct
        ct_idx = np.where(ct_mask)[0]
        n_ct = len(ct_idx)

        if n_ct < 10:
            continue

        has_ref = ct in type_mapping
        if has_ref:
            ref_mean = ref_profiles[type_mapping[ct]]['mean']

        ct_spatial = spatial_coords[ct_mask]
        ct_hybrid = hybrid[ct_mask]
        ct_X = X_for_neighbors[ct_mask]

        graph = AdaptiveRadiusGraph(radius=spatial_radius, k_min=k_min, k_max=k_max)
        graph.fit(ct_spatial)

        knn = NearestNeighbors(n_neighbors=min(k_max, n_ct - 1), metric='cosine')
        knn.fit(ct_hybrid)

        for i in range(n_ct):
            global_idx = ct_idx[i]

            spatial_idx, _ = graph.get_neighbors(i)
            _, expr_idx = knn.kneighbors(ct_hybrid[i:i+1])
            expr_idx = expr_idx[0, 1:]

            all_nbrs = np.unique(np.concatenate([spatial_idx, expr_idx]))
            if len(all_nbrs) == 0:
                continue

            neighbor_expr = ct_X[all_nbrs]

            s_dist = np.linalg.norm(ct_spatial[all_nbrs] - ct_spatial[i], axis=1)
            e_dist = np.linalg.norm(ct_hybrid[all_nbrs] - ct_hybrid[i], axis=1)
            s_dist_n = s_dist / (s_dist.max() + 1e-8)
            e_dist_n = e_dist / (e_dist.max() + 1e-8)
            h_dist = (1 - spatial_weight) * e_dist_n + spatial_weight * s_dist_n

            weights = np.exp(-h_dist ** 2 / 0.5)
            weights = weights / (weights.sum() + 1e-8)

            mu_neighbors = np.average(neighbor_expr, axis=0, weights=weights)
            var_neighbors = np.average((neighbor_expr - mu_neighbors) ** 2, axis=0, weights=weights)

            if has_ref:
                var_norm = var_neighbors / (var_neighbors.max() + 1e-8)
                lam = np.clip(reference_weight * (1 + var_norm), 0, 0.8)
                mu_expected = (1 - lam) * mu_neighbors + lam * ref_mean
            else:
                mu_expected = mu_neighbors

            alpha = np.maximum((var_neighbors - mu_neighbors) / (mu_neighbors ** 2 + 1e-8), 0.01)
            unc = np.sqrt(mu_expected + alpha * mu_expected ** 2)
            uncertainty[global_idx] = unc

            # STRICT dropout detection
            observed = X[global_idx]
            is_dropout, _ = detect_dropouts_strict(observed, mu_expected, alpha, dropout_p_threshold)

            # ONLY correct dropouts!
            if is_dropout.any():
                corrected = shrinkage * mu_expected + (1 - shrinkage) * observed
                X_denoised[global_idx, is_dropout] = corrected[is_dropout]
                was_corrected[global_idx, is_dropout] = True

    n_corrected = was_corrected.sum()
    pct_corrected = 100 * n_corrected / (n_cells * n_genes)
    print(f"\n  Dropouts corrected: {n_corrected:,} ({pct_corrected:.2f}%)")

    # Calibrate
    if calibrate:
        print("  Calibrating...")
        uncertainty = calibrator.calibrate(X_denoised, uncertainty, X, holdout_mask, was_corrected)
        calibrator.print_report()
        cal_stats = calibrator.calibration_stats_
    else:
        cal_stats = None

    # Diagnostics
    cv_raw = np.std(X, axis=0) / (np.mean(X, axis=0) + 1e-8)
    cv_den = np.std(X_denoised, axis=0) / (np.mean(X_denoised, axis=0) + 1e-8)
    cv_change = (cv_den - cv_raw) / (cv_raw + 1e-8) * 100
    print("Mean CV before:", cv_raw.mean())
    print("Mean CV after :", cv_den.mean())
    print(f"\n  CV change: {cv_change.mean():+.2f}%")
    print(f"  (Negative = variability decreased = GOOD)")

    metadata = {
        'was_corrected': was_corrected,
        'n_corrected': n_corrected,
        'pct_corrected': pct_corrected,
        'shared_genes': shared_genes,
        'calibration_stats': cal_stats,
        'cv_raw': cv_raw,
        'cv_denoised': cv_den,
        'cv_change_pct': cv_change,
        'type_mapping': type_mapping
    }

    print("=" * 60)
    return X_denoised, uncertainty, metadata

In [ ]:
# ==============================================================================
# COSMX CELL: Run the denoising
# ==============================================================================

import numpy as np

# Ensure compatible coordinate columns exist
if "x_centroid" not in annotated_spatial_adata.obs.columns:
    annotated_spatial_adata.obs["x_centroid"] = annotated_spatial_adata.obs["center_x_global_px"].astype(float)

if "y_centroid" not in annotated_spatial_adata.obs.columns:
    annotated_spatial_adata.obs["y_centroid"] = annotated_spatial_adata.obs["center_y_global_px"].astype(float)

# Ensure compatible cell-type columns exist
if "cell_type" not in annotated_spatial_adata.obs.columns:
    annotated_spatial_adata.obs["cell_type"] = annotated_spatial_adata.obs["Final_CosMx_Cell_Type"].astype(str)

if "cell_type" not in annotated_ref_adata.obs.columns:
    annotated_ref_adata.obs["cell_type"] = annotated_ref_adata.obs["Reference_Cell_Type"].astype(str)

annotated_ref_adata.var_names_make_unique()

# Parameters — start with the same conservative Xenium parameters
DROPOUT_P = 0.35
SHRINKAGE = 0.85

X_denoised, uncertainty, metadata = denoise_v4_selective(
    annotated_spatial_adata,
    annotated_ref_adata,
    spatial_weight=0.5,
    reference_weight=0.5,
    spatial_radius=100.0,      # CosMx coordinates are pixels; 100 px ≈ 18 µm
    k_min=5,
    k_max=30,
    dropout_p_threshold=DROPOUT_P,
    shrinkage=SHRINKAGE,
    calibrate=False
)

# Store results
shared_genes = metadata["shared_genes"]
denoised_adata = annotated_spatial_adata[:, shared_genes].copy()

# Use clean raw counts
denoised_adata.layers["raw"] = ensure_dense(denoised_adata.X).astype(np.float32)

denoised_adata.X = X_denoised.astype(np.float32)
denoised_adata.layers["uncertainty"] = uncertainty.astype(np.float32)
denoised_adata.obs["was_corrected_count"] = metadata["was_corrected"].sum(axis=1)

# Ensure cell_type is carried
denoised_adata.obs["cell_type"] = annotated_spatial_adata.obs.loc[
    denoised_adata.obs_names, "cell_type"
].astype(str).values

print("\nStored CosMx denoising result in denoised_adata")
print(denoised_adata)
print("Raw sum:", denoised_adata.layers["raw"].sum())
print("Denoised sum:", denoised_adata.X.sum())
print("Corrected pairs:", metadata["n_corrected"])
print("Correction rate:", metadata["pct_corrected"])

In [ ]:
# ==============================================================================
# REQUIRED OBJECT CHECKS + SAFE ALIGNMENT REBUILD
# ==============================================================================
# Important:
#   The denoising function may return X_denoised in metadata["shared_genes"] order.
#   annotated_spatial_adata may be in alphabetical/original gene order.
#   Therefore, simply assigning X_raw_counts to denoised_adata.layers["raw"] can
#   create a raw-vs-denoised gene-order mismatch.
#
# This block rebuilds denoised_adata in the exact denoising gene order before any
# downstream validation task is run.
# ==============================================================================

if "denoised_adata" not in globals():
    raise NameError("denoised_adata not found. Run denoising first.")

if "cell_type" not in denoised_adata.obs.columns:
    if "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
        denoised_adata.obs["cell_type"] = denoised_adata.obs["Final_CosMx_Cell_Type"].astype(str)
    else:
        raise KeyError("No cell_type or Final_CosMx_Cell_Type found.")


def rebuild_denoised_adata_in_safe_gene_order():
    """
    Rebuild denoised_adata so that:
      - denoised_adata.var_names == metadata["shared_genes"]
      - denoised_adata.X == X_denoised in the same gene order
      - denoised_adata.layers["raw"] is raw counts in that same gene order
      - no negative deltas exist for selective dropout correction
    """

    global denoised_adata

    can_rebuild = (
        "annotated_spatial_adata" in globals()
        and "X_denoised" in globals()
        and "metadata" in globals()
        and isinstance(metadata, dict)
        and "shared_genes" in metadata
    )

    if not can_rebuild:
        print("Safe rebuild inputs not fully available.")
        print("Using existing denoised_adata, but checking raw-vs-denoised alignment.")
        return

    shared_genes_safe = list(map(str, metadata["shared_genes"]))

    missing_genes = [
        g for g in shared_genes_safe
        if g not in annotated_spatial_adata.var_names.astype(str)
    ]

    if missing_genes:
        raise ValueError(
            "Cannot rebuild denoised_adata: some metadata['shared_genes'] are "
            f"missing from annotated_spatial_adata. Examples: {missing_genes[:20]}"
        )

    # Build AnnData in the exact denoising gene order.
    denoised_adata_fixed = annotated_spatial_adata[:, shared_genes_safe].copy()

    X_denoised_arr = ensure_dense(X_denoised).astype(np.float32)

    if X_denoised_arr.shape != denoised_adata_fixed.shape:
        raise ValueError(
            f"X_denoised shape {X_denoised_arr.shape} does not match "
            f"rebuilt AnnData shape {denoised_adata_fixed.shape}."
        )

    denoised_adata_fixed.X = X_denoised_arr.copy()

    # Raw layer in the same gene order.
    if "raw_molecule_counts_clean" in annotated_spatial_adata.layers:
        raw_source = annotated_spatial_adata[:, shared_genes_safe].layers["raw_molecule_counts_clean"]
        print("Using annotated_spatial_adata.layers['raw_molecule_counts_clean'] for raw layer.")
    elif "raw" in annotated_spatial_adata.layers:
        raw_source = annotated_spatial_adata[:, shared_genes_safe].layers["raw"]
        print("Using annotated_spatial_adata.layers['raw'] for raw layer.")
    else:
        raw_source = annotated_spatial_adata[:, shared_genes_safe].X
        print("Using annotated_spatial_adata.X for raw layer.")

    X_raw_safe = ensure_dense(raw_source).astype(np.float32)

    if X_raw_safe.shape != denoised_adata_fixed.shape:
        raise ValueError(
            f"Raw layer shape {X_raw_safe.shape} does not match "
            f"rebuilt AnnData shape {denoised_adata_fixed.shape}."
        )

    denoised_adata_fixed.layers["raw"] = X_raw_safe.copy()
    denoised_adata_fixed.layers["raw_molecule_counts_clean"] = X_raw_safe.copy()

    # Copy uncertainty if available and aligned.
    if "uncertainty" in globals() and uncertainty is not None:
        uncertainty_arr = ensure_dense(uncertainty).astype(np.float32)
        if uncertainty_arr.shape == denoised_adata_fixed.shape:
            denoised_adata_fixed.layers["uncertainty"] = uncertainty_arr.copy()
        else:
            print("WARNING: uncertainty shape does not match rebuilt AnnData; not copied.")

    # Copy was_corrected mask if available and aligned.
    if "was_corrected" in metadata:
        was_corrected_arr = np.asarray(metadata["was_corrected"])
        if was_corrected_arr.shape == denoised_adata_fixed.shape:
            denoised_adata_fixed.layers["was_corrected"] = was_corrected_arr.astype(np.uint8)
            denoised_adata_fixed.obs["was_corrected_count"] = was_corrected_arr.sum(axis=1)
            denoised_adata_fixed.var["was_corrected_count"] = was_corrected_arr.sum(axis=0)
        else:
            print("WARNING: metadata['was_corrected'] shape does not match rebuilt AnnData; not copied.")

    # Preserve cell_type column if needed.
    if "cell_type" not in denoised_adata_fixed.obs.columns:
        if "Final_CosMx_Cell_Type" in denoised_adata_fixed.obs.columns:
            denoised_adata_fixed.obs["cell_type"] = denoised_adata_fixed.obs["Final_CosMx_Cell_Type"].astype(str)
        else:
            raise KeyError("No cell_type or Final_CosMx_Cell_Type found after rebuild.")

    denoised_adata = denoised_adata_fixed

    print("Rebuilt denoised_adata in safe denoising gene order.")
    print(f"  shape: {denoised_adata.shape}")
    print(f"  first 10 genes: {list(denoised_adata.var_names.astype(str))[:10]}")


rebuild_denoised_adata_in_safe_gene_order()

if "raw" not in denoised_adata.layers:
    raise KeyError("denoised_adata.layers['raw'] is missing after alignment check.")

# Final directionality check.
X_raw_check = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
X_den_check = ensure_dense(denoised_adata.X).astype(np.float32)

if X_raw_check.shape != X_den_check.shape:
    raise ValueError(
        f"Raw layer shape {X_raw_check.shape} does not match denoised X shape {X_den_check.shape}."
    )

delta_check = X_den_check - X_raw_check
negative_entries = int((delta_check < -1e-6).sum())

print(f"Validation AnnData: {denoised_adata}")
print(f"Raw sum: {X_raw_check.sum(dtype=np.float64):,.0f}")
print(f"Denoised sum: {X_den_check.sum(dtype=np.float64):,.2f}")
print(f"Net added counts: {delta_check.sum(dtype=np.float64):,.2f}")
print(f"Negative delta entries: {negative_entries:,}")

if negative_entries > 0:
    raise ValueError(
        "Raw-vs-denoised alignment check failed: negative deltas were found. "
        "For selective dropout correction, denoised counts should not be below raw counts."
    )


In [ ]:
# ==============================================================================
# COSMX CELL 85: Visualize results
# Diagnostics only — no final export here
# ==============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import sparse
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------------------------
# Output folders
# ------------------------------------------------------------------------------

OUTPUT_ROOT = Path("/content/cosmx_cell_level_denoising_outputs")
DIAG_DIR = OUTPUT_ROOT / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("COSMX CELL 85 — DIAGNOSTICS ONLY")
print("=" * 80)
print(f"Diagnostics folder: {DIAG_DIR}")

# ------------------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------------------------
# Required checks
# ------------------------------------------------------------------------------

if "denoised_adata" not in globals():
    raise NameError("denoised_adata not found. Run the actual denoising cell first.")

if "metadata" not in globals():
    raise NameError("metadata not found. Run the actual denoising cell first.")

if "raw" not in denoised_adata.layers:
    raise KeyError("denoised_adata.layers['raw'] is missing.")

if "uncertainty" not in denoised_adata.layers:
    raise KeyError("denoised_adata.layers['uncertainty'] is missing.")

if "was_corrected" not in metadata:
    raise KeyError("metadata['was_corrected'] is missing.")

# Ensure cell type exists
if "cell_type" not in denoised_adata.obs.columns:
    if "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
        denoised_adata.obs["cell_type"] = denoised_adata.obs["Final_CosMx_Cell_Type"].astype(str)
    else:
        raise KeyError("No cell_type or Final_CosMx_Cell_Type found in denoised_adata.obs.")

# ------------------------------------------------------------------------------
# CELL 9-style diagnostic figure, matching Xenium structure
# ------------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. CV comparison
ax = axes[0, 0]
cv_raw = metadata["cv_raw"]
cv_den = metadata["cv_denoised"]

ax.hist(
    cv_raw,
    bins=50,
    alpha=0.5,
    label=f"Raw (mean={cv_raw.mean():.2f})",
    density=True
)
ax.hist(
    cv_den,
    bins=50,
    alpha=0.5,
    label=f"Denoised (mean={cv_den.mean():.2f})",
    density=True
)
ax.set_xlabel("Coefficient of Variation")
ax.set_ylabel("Density")
ax.set_title("Gene CV: Before vs After (selective)")
ax.legend()

# 2. CV scatter
ax = axes[0, 1]
ax.scatter(cv_raw, cv_den, alpha=0.3, s=5)
ax.plot([0, cv_raw.max()], [0, cv_raw.max()], "r--", label="No change")
ax.set_xlabel("CV (Raw)")
ax.set_ylabel("CV (Denoised)")
ax.set_title("CV per gene")
ax.legend()

# 3. CV change distribution
ax = axes[1, 0]
cv_change = metadata["cv_change_pct"]

ax.hist(cv_change, bins=50, edgecolor="white")
ax.axvline(
    cv_change.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {cv_change.mean():+.1f}%"
)
ax.axvline(0, color="black", linestyle="-", alpha=0.5)
ax.set_xlabel("CV Change (%)")
ax.set_ylabel("Genes")
ax.set_title("CV change per gene (negative = improvement)")
ax.legend()

# 4. Corrections per cell
ax = axes[1, 1]
corrections = metadata["was_corrected"].sum(axis=1)

ax.hist(corrections, bins=50, edgecolor="white")
ax.axvline(
    corrections.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {corrections.mean():.1f}"
)
ax.set_xlabel("Dropouts corrected per cell")
ax.set_ylabel("Cells")
ax.set_title(f'Dropout corrections ({metadata["pct_corrected"]:.2f}% of values)')
ax.legend()

plt.tight_layout()

local_diag_png = "/content/cosmx_step4_v4_diagnostics.png"

plt.savefig(local_diag_png, dpi=150)
plt.show()

print(f"Saved diagnostics figure:")
print(f"  {local_diag_png}")

# ------------------------------------------------------------------------------
# CELL 10-style specific gene before/after comparison
# ------------------------------------------------------------------------------

# Choose a CosMx-relevant marker but keep the plot style same as Xenium.
candidate_genes = [
    "CD3D",     # T-cell marker
    "EPCAM",    # epithelial marker
    "KRT19",    # epithelial marker
    "CD79A",    # B-cell marker
    "LYZ",      # myeloid marker
    "COL1A1",   # fibroblast marker
    "PECAM1"    # endothelial marker
]

test_gene = None
for g in candidate_genes:
    if g in denoised_adata.var_names:
        test_gene = g
        break

if test_gene is None:
    print("No candidate marker gene found for gene-specific diagnostic plot.")
else:
    print(f"\nUsing test gene for before/after plot: {test_gene}")

    gene_idx = denoised_adata.var_names.tolist().index(test_gene)

    raw = ensure_dense(denoised_adata.layers["raw"])[:, gene_idx]
    den = ensure_dense(denoised_adata.X)[:, gene_idx]
    corrected_mask = metadata["was_corrected"][:, gene_idx]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Distribution
    ax = axes[0]
    ax.hist(raw, bins=50, alpha=0.5, label="Raw", density=True)
    ax.hist(den, bins=50, alpha=0.5, label="Denoised", density=True)
    ax.set_title(f"{test_gene}: Distribution")
    ax.legend()

    # Scatter raw vs denoised
    ax = axes[1]
    ax.scatter(
        raw[~corrected_mask],
        den[~corrected_mask],
        alpha=0.1,
        s=1,
        label="Unchanged",
        c="gray"
    )
    ax.scatter(
        raw[corrected_mask],
        den[corrected_mask],
        alpha=0.5,
        s=3,
        label="Corrected",
        c="red"
    )

    max_val = max(float(raw.max()), float(den.max()), 1.0)
    ax.plot([0, max_val], [0, max_val], "k--", alpha=0.5)
    ax.set_xlabel("Raw")
    ax.set_ylabel("Denoised")
    ax.set_title(f"{test_gene}: Raw vs Denoised")
    ax.legend()

    # Correction amount
    ax = axes[2]
    correction_amount = den - raw
    ax.hist(correction_amount[corrected_mask], bins=50, edgecolor="white")
    ax.set_xlabel("Denoised - Raw")
    ax.set_ylabel("Corrected cells")
    ax.set_title(f"{test_gene}: Correction Amount")

    plt.tight_layout()

    local_gene_png = f"/content/cosmx_step4_v4_gene_example_{test_gene}.png"
    plt.savefig(local_gene_png, dpi=150)
    plt.show()

    print(f"Saved gene example figure:")
    print(f"  {local_gene_png}")

# ------------------------------------------------------------------------------
# Summary printout, matching Xenium style
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 4 DIAGNOSTIC SUMMARY")
print("=" * 80)
print(f"CV change: {metadata['cv_change_pct'].mean():+.2f}%")
print(f"Values corrected: {metadata['n_corrected']:,} ({metadata['pct_corrected']:.2f}%)")
print(f"Values unchanged: {metadata['was_corrected'].size - metadata['n_corrected']:,}")
print(f"Raw sum: {ensure_dense(denoised_adata.layers['raw']).sum(dtype=np.float64):,.0f}")
print(f"Denoised sum: {ensure_dense(denoised_adata.X).sum(dtype=np.float64):,.2f}")

# Save a compact CSV summary but do not export final objects here
diag_summary = pd.DataFrame([{
    "n_cells": denoised_adata.n_obs,
    "n_genes": denoised_adata.n_vars,
    "raw_sum": float(ensure_dense(denoised_adata.layers["raw"]).sum(dtype=np.float64)),
    "denoised_sum": float(ensure_dense(denoised_adata.X).sum(dtype=np.float64)),
    "corrected_pairs": int(metadata["n_corrected"]),
    "total_cell_gene_pairs": int(metadata["was_corrected"].size),
    "correction_rate_percent": float(metadata["pct_corrected"]),
    "mean_cv_raw": float(metadata["cv_raw"].mean()),
    "mean_cv_denoised": float(metadata["cv_denoised"].mean()),
    "mean_cv_change_percent": float(metadata["cv_change_pct"].mean()),
}])

diag_summary_path = DIAG_DIR / "cosmx_cell85_diagnostic_summary.csv"
diag_summary.to_csv(diag_summary_path, index=False)

print(f"\nSaved diagnostic summary:")
print(f"  {diag_summary_path}")

print("\nCOSMX CELL 85 finished successfully.")
print("No final export was performed here. Final export will happen at the very end.")

In [ ]:
"""
================================================================================
COSMX CELL 86 — DOWNSTREAM VALIDATION TASKS FOR CELL-LEVEL DENOISING
================================================================================
This cell is structurally aligned with the Xenium validation cell.

Tasks:
  1. Marker gene validation
  2. Clustering quality: ARI / NMI / silhouette
  3. Differential expression with Xenium-style significant DE gene count
  4. SVG detection using Moran's I
================================================================================
"""

# ==============================================================================
# INSTALL / IMPORTS
# ==============================================================================

!pip install -q scanpy leidenalg igraph

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy import sparse
from scipy.stats import mannwhitneyu

import scanpy as sc
import anndata as ad

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score
)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ==============================================================================
# OUTPUT FOLDER
# ==============================================================================

OUTPUT_ROOT = Path("/content/cosmx_cell_level_denoising_outputs")
VAL_DIR = OUTPUT_ROOT / "validation"
VAL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("COSMX CELL 86 — DOWNSTREAM VALIDATION")
print("=" * 80)
print(f"Validation output folder: {VAL_DIR}")

# ==============================================================================
# HELPERS
# ==============================================================================

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def get_cell_type_column(adata):
    """
    Finds the correct cell-type column.
    """
    if "cell_type" in adata.obs.columns:
        return "cell_type"
    elif "Final_CosMx_Cell_Type" in adata.obs.columns:
        return "Final_CosMx_Cell_Type"
    elif "Assigned_Xenium_Cell_Type" in adata.obs.columns:
        return "Assigned_Xenium_Cell_Type"
    else:
        raise KeyError(
            "No cell-type column found. Expected 'cell_type', "
            "'Final_CosMx_Cell_Type', or 'Assigned_Xenium_Cell_Type'."
        )


# ==============================================================================
# REQUIRED OBJECT CHECKS + SAFE ALIGNMENT REBUILD
# ==============================================================================
# Important:
#   The denoising function may return X_denoised in metadata["shared_genes"] order.
#   annotated_spatial_adata may be in alphabetical/original gene order.
#   Therefore, simply assigning X_raw_counts to denoised_adata.layers["raw"] can
#   create a raw-vs-denoised gene-order mismatch.
#
# This block rebuilds denoised_adata in the exact denoising gene order before any
# downstream validation task is run.
# ==============================================================================

if "denoised_adata" not in globals():
    raise NameError("denoised_adata not found. Run denoising first.")

if "cell_type" not in denoised_adata.obs.columns:
    if "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
        denoised_adata.obs["cell_type"] = denoised_adata.obs["Final_CosMx_Cell_Type"].astype(str)
    else:
        raise KeyError("No cell_type or Final_CosMx_Cell_Type found.")


def rebuild_denoised_adata_in_safe_gene_order():
    """
    Rebuild denoised_adata so that:
      - denoised_adata.var_names == metadata["shared_genes"]
      - denoised_adata.X == X_denoised in the same gene order
      - denoised_adata.layers["raw"] is raw counts in that same gene order
      - no negative deltas exist for selective dropout correction
    """

    global denoised_adata

    can_rebuild = (
        "annotated_spatial_adata" in globals()
        and "X_denoised" in globals()
        and "metadata" in globals()
        and isinstance(metadata, dict)
        and "shared_genes" in metadata
    )

    if not can_rebuild:
        print("Safe rebuild inputs not fully available.")
        print("Using existing denoised_adata, but checking raw-vs-denoised alignment.")
        return

    shared_genes_safe = list(map(str, metadata["shared_genes"]))

    missing_genes = [
        g for g in shared_genes_safe
        if g not in annotated_spatial_adata.var_names.astype(str)
    ]

    if missing_genes:
        raise ValueError(
            "Cannot rebuild denoised_adata: some metadata['shared_genes'] are "
            f"missing from annotated_spatial_adata. Examples: {missing_genes[:20]}"
        )

    # Build AnnData in the exact denoising gene order.
    denoised_adata_fixed = annotated_spatial_adata[:, shared_genes_safe].copy()

    X_denoised_arr = ensure_dense(X_denoised).astype(np.float32)

    if X_denoised_arr.shape != denoised_adata_fixed.shape:
        raise ValueError(
            f"X_denoised shape {X_denoised_arr.shape} does not match "
            f"rebuilt AnnData shape {denoised_adata_fixed.shape}."
        )

    denoised_adata_fixed.X = X_denoised_arr.copy()

    # Raw layer in the same gene order.
    if "raw_molecule_counts_clean" in annotated_spatial_adata.layers:
        raw_source = annotated_spatial_adata[:, shared_genes_safe].layers["raw_molecule_counts_clean"]
        print("Using annotated_spatial_adata.layers['raw_molecule_counts_clean'] for raw layer.")
    elif "raw" in annotated_spatial_adata.layers:
        raw_source = annotated_spatial_adata[:, shared_genes_safe].layers["raw"]
        print("Using annotated_spatial_adata.layers['raw'] for raw layer.")
    else:
        raw_source = annotated_spatial_adata[:, shared_genes_safe].X
        print("Using annotated_spatial_adata.X for raw layer.")

    X_raw_safe = ensure_dense(raw_source).astype(np.float32)

    if X_raw_safe.shape != denoised_adata_fixed.shape:
        raise ValueError(
            f"Raw layer shape {X_raw_safe.shape} does not match "
            f"rebuilt AnnData shape {denoised_adata_fixed.shape}."
        )

    denoised_adata_fixed.layers["raw"] = X_raw_safe.copy()
    denoised_adata_fixed.layers["raw_molecule_counts_clean"] = X_raw_safe.copy()

    # Copy uncertainty if available and aligned.
    if "uncertainty" in globals() and uncertainty is not None:
        uncertainty_arr = ensure_dense(uncertainty).astype(np.float32)
        if uncertainty_arr.shape == denoised_adata_fixed.shape:
            denoised_adata_fixed.layers["uncertainty"] = uncertainty_arr.copy()
        else:
            print("WARNING: uncertainty shape does not match rebuilt AnnData; not copied.")

    # Copy was_corrected mask if available and aligned.
    if "was_corrected" in metadata:
        was_corrected_arr = np.asarray(metadata["was_corrected"])
        if was_corrected_arr.shape == denoised_adata_fixed.shape:
            denoised_adata_fixed.layers["was_corrected"] = was_corrected_arr.astype(np.uint8)
            denoised_adata_fixed.obs["was_corrected_count"] = was_corrected_arr.sum(axis=1)
            denoised_adata_fixed.var["was_corrected_count"] = was_corrected_arr.sum(axis=0)
        else:
            print("WARNING: metadata['was_corrected'] shape does not match rebuilt AnnData; not copied.")

    # Preserve cell_type column if needed.
    if "cell_type" not in denoised_adata_fixed.obs.columns:
        if "Final_CosMx_Cell_Type" in denoised_adata_fixed.obs.columns:
            denoised_adata_fixed.obs["cell_type"] = denoised_adata_fixed.obs["Final_CosMx_Cell_Type"].astype(str)
        else:
            raise KeyError("No cell_type or Final_CosMx_Cell_Type found after rebuild.")

    denoised_adata = denoised_adata_fixed

    print("Rebuilt denoised_adata in safe denoising gene order.")
    print(f"  shape: {denoised_adata.shape}")
    print(f"  first 10 genes: {list(denoised_adata.var_names.astype(str))[:10]}")


rebuild_denoised_adata_in_safe_gene_order()

if "raw" not in denoised_adata.layers:
    raise KeyError("denoised_adata.layers['raw'] is missing after alignment check.")

# Final directionality check.
X_raw_check = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
X_den_check = ensure_dense(denoised_adata.X).astype(np.float32)

if X_raw_check.shape != X_den_check.shape:
    raise ValueError(
        f"Raw layer shape {X_raw_check.shape} does not match denoised X shape {X_den_check.shape}."
    )

delta_check = X_den_check - X_raw_check
negative_entries = int((delta_check < -1e-6).sum())

print(f"Validation AnnData: {denoised_adata}")
print(f"Raw sum: {X_raw_check.sum(dtype=np.float64):,.0f}")
print(f"Denoised sum: {X_den_check.sum(dtype=np.float64):,.2f}")
print(f"Net added counts: {delta_check.sum(dtype=np.float64):,.2f}")
print(f"Negative delta entries: {negative_entries:,}")

if negative_entries > 0:
    raise ValueError(
        "Raw-vs-denoised alignment check failed: negative deltas were found. "
        "For selective dropout correction, denoised counts should not be below raw counts."
    )


# ==============================================================================
# TASK 1: MARKER GENE VALIDATION
# ==============================================================================

# Keep exactly 6 genes so the figure remains 2 x 3, like Xenium.
MARKERS = {
    "EPCAM": ["Epithelial cells"],
    "CD3D": ["T lymphocytes"],
    "CD79A": ["B lymphocytes"],
    "LYZ": ["Myeloid cells"],
    "COL1A1": ["Fibroblasts"],
    "PECAM1": ["Endothelial cells"],
}


def validate_markers(denoised_adata, markers=MARKERS):
    """
    Compare marker gene expression in expected vs unexpected cell types.

    A good denoising should:
      - maintain high expression in expected cell types
      - not artificially inflate expression in unexpected cell types
    """

    ct_col = get_cell_type_column(denoised_adata)

    results = []

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    X_raw = ensure_dense(denoised_adata.layers["raw"])
    X_den = ensure_dense(denoised_adata.X)
    cell_types = denoised_adata.obs[ct_col].astype(str).values

    for i, (gene, expected_types) in enumerate(markers.items()):
        if i >= len(axes):
            break

        ax = axes[i]

        if gene not in denoised_adata.var_names:
            ax.text(0.5, 0.5, f"{gene} not found", ha="center")
            ax.set_title(gene)
            continue

        gene_idx = denoised_adata.var_names.tolist().index(gene)

        raw = X_raw[:, gene_idx]
        den = X_den[:, gene_idx]

        ct_stats = []

        for ct in np.unique(cell_types):
            mask = cell_types == ct

            if mask.sum() < 10:
                continue

            is_expected = any(
                exp.lower() in ct.lower() or ct.lower() in exp.lower()
                for exp in expected_types
            )

            ct_stats.append({
                "gene": gene,
                "cell_type": ct[:20],
                "full_cell_type": ct,
                "expected": is_expected,
                "raw_mean": float(raw[mask].mean()),
                "den_mean": float(den[mask].mean()),
                "raw_nonzero": float((raw[mask] > 0).mean() * 100),
                "den_nonzero": float((den[mask] > 0).mean() * 100),
                "n_cells": int(mask.sum()),
            })

        ct_df = pd.DataFrame(ct_stats).sort_values("den_mean", ascending=False)

        if len(ct_df) == 0:
            ax.text(0.5, 0.5, "No valid cell types", ha="center")
            ax.set_title(gene)
            continue

        x = np.arange(len(ct_df))
        width = 0.35

        ax.barh(
            x - width / 2,
            ct_df["raw_mean"],
            width,
            label="Raw",
            alpha=0.7,
            color="lightblue"
        )

        ax.barh(
            x + width / 2,
            ct_df["den_mean"],
            width,
            label="Denoised",
            alpha=0.7,
            color="coral"
        )

        for j, exp in enumerate(ct_df["expected"]):
            if exp:
                ax.scatter(
                    [ct_df.iloc[j]["den_mean"] + 0.1],
                    [j],
                    marker="*",
                    color="green",
                    s=100,
                    zorder=5
                )

        ax.set_yticks(x)
        ax.set_yticklabels(ct_df["cell_type"])
        ax.set_xlabel("Mean Expression")
        ax.set_title(gene)
        ax.legend(loc="lower right")

        results.append(ct_df)

    plt.tight_layout()

    local_path = "/content/marker_validation.png"
    drive_path = VAL_DIR / "marker_validation.png"

    plt.savefig(local_path, dpi=150)
    plt.savefig(drive_path, dpi=150)
    plt.show()

    print(f"Saved marker validation plot:")
    print(f"  {local_path}")
    print(f"  {drive_path}")

    if len(results) > 0:
        marker_df = pd.concat(results, ignore_index=True)
        marker_df.to_csv(VAL_DIR / "cosmx_marker_validation_table.csv", index=False)
    else:
        marker_df = pd.DataFrame()

    return marker_df


# ==============================================================================
# TASK 2: CLUSTERING QUALITY
# ==============================================================================

def evaluate_clustering_quality(
    denoised_adata,
    raw_matrix=None,
    sample_size=50000,
    random_state=42,
    n_pcs=30,
    leiden_resolution=0.5,
):
    """
    Compare clustering quality between raw and denoised expression.

    Xenium-matched configuration:
      - sample_size = 50,000
      - random_state = 42
      - normalize_total(target_sum=1e4)
      - log1p
      - PCA = 30 PCs
      - Leiden resolution = 0.5
      - ARI / NMI / silhouette vs cell_type labels
    """

    ct_col = get_cell_type_column(denoised_adata)

    if raw_matrix is None:
        if "raw" not in denoised_adata.layers:
            raise KeyError("denoised_adata.layers['raw'] is missing.")
        raw_matrix = denoised_adata.layers["raw"]

    X_raw = ensure_dense(raw_matrix).astype(np.float32)
    X_den = ensure_dense(denoised_adata.X).astype(np.float32)

    if X_raw.shape != X_den.shape:
        raise ValueError(f"Raw shape {X_raw.shape} does not match denoised shape {X_den.shape}.")

    n_cells = denoised_adata.n_obs
    n_sample = min(sample_size, n_cells)

    rng = np.random.default_rng(random_state)
    idx = rng.choice(n_cells, n_sample, replace=False)

    obs_sub = denoised_adata.obs.iloc[idx].copy()
    var_sub = denoised_adata.var.copy()

    adata_raw = ad.AnnData(
        X=X_raw[idx, :].copy(),
        obs=obs_sub.copy(),
        var=var_sub.copy()
    )

    adata_den = ad.AnnData(
        X=X_den[idx, :].copy(),
        obs=obs_sub.copy(),
        var=var_sub.copy()
    )

    adata_raw.var_names = denoised_adata.var_names.astype(str)
    adata_den.var_names = denoised_adata.var_names.astype(str)

    adata_raw.var_names_make_unique()
    adata_den.var_names_make_unique()

    print(f"Sampled cells for clustering: {n_sample:,} / {n_cells:,}")

    # Normalize/log/PCA/neighbors/Leiden
    for label, adata_tmp in [("Raw", adata_raw), ("Denoised", adata_den)]:
        print(f"Processing {label}...")
        sc.pp.normalize_total(adata_tmp, target_sum=1e4)
        sc.pp.log1p(adata_tmp)

        n_pcs_use = min(n_pcs, adata_tmp.n_vars - 1)
        sc.pp.pca(adata_tmp, n_comps=n_pcs_use, random_state=random_state)
        sc.pp.neighbors(adata_tmp, n_pcs=n_pcs_use)
        sc.tl.leiden(
            adata_tmp,
            resolution=leiden_resolution,
            random_state=random_state,
            key_added="leiden"
        )

    true_labels = obs_sub[ct_col].astype(str).values

    clusters_raw = adata_raw.obs["leiden"].astype(str).values
    clusters_den = adata_den.obs["leiden"].astype(str).values

    ari_raw = adjusted_rand_score(true_labels, clusters_raw)
    ari_den = adjusted_rand_score(true_labels, clusters_den)

    nmi_raw = normalized_mutual_info_score(true_labels, clusters_raw)
    nmi_den = normalized_mutual_info_score(true_labels, clusters_den)

    sil_raw = silhouette_score(adata_raw.obsm["X_pca"], true_labels)
    sil_den = silhouette_score(adata_den.obsm["X_pca"], true_labels)

    results = {
        "Raw observed counts": {
            "ARI": ari_raw,
            "NMI": nmi_raw,
            "silhouette": sil_raw,
            "n_clusters": int(pd.Series(clusters_raw).nunique()),
        },
        "Step4 denoised": {
            "ARI": ari_den,
            "NMI": nmi_den,
            "silhouette": sil_den,
            "n_clusters": int(pd.Series(clusters_den).nunique()),
        },
    }

    results_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "dataset"})

    print("\nClustering quality:")
    display(results_df)

    print("\nDetailed comparison:")
    print(f"{'Metric':<20} {'Raw':>12} {'Denoised':>12} {'Change':>12}")
    print("-" * 60)

    for metric in ["ARI", "NMI", "silhouette"]:
        raw_val = results["Raw observed counts"][metric]
        den_val = results["Step4 denoised"][metric]
        change = den_val - raw_val
        symbol = "✓" if change > 0 else "✗"
        print(f"{metric:<20} {raw_val:>12.4f} {den_val:>12.4f} {change:>+12.4f} {symbol}")

    print(
        f"{'n_clusters':<20} "
        f"{results['Raw observed counts']['n_clusters']:>12} "
        f"{results['Step4 denoised']['n_clusters']:>12} "
        f"{results['Step4 denoised']['n_clusters'] - results['Raw observed counts']['n_clusters']:>+12}"
    )

    # Plot in Xenium-style 1x2 panel
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    metrics = ["ARI", "NMI", "silhouette"]
    x = np.arange(len(metrics))
    width = 0.35

    raw_vals = [results["Raw observed counts"][m] for m in metrics]
    den_vals = [results["Step4 denoised"][m] for m in metrics]

    axes[0].bar(x - width / 2, raw_vals, width, label="Raw observed counts")
    axes[0].bar(x + width / 2, den_vals, width, label="Step4 denoised")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metrics)
    axes[0].set_ylabel("Score")
    axes[0].set_title("Cell-type structure metrics")
    axes[0].legend()

    for i, (r, d) in enumerate(zip(raw_vals, den_vals)):
        axes[0].text(i, max(r, d) + 0.02, f"{d-r:+.3f}", ha="center", fontsize=9)

    axes[1].bar(
        ["Raw observed counts", "Step4 denoised"],
        [
            results["Raw observed counts"]["n_clusters"],
            results["Step4 denoised"]["n_clusters"],
        ]
    )
    axes[1].set_ylabel("Number of Leiden clusters")
    axes[1].set_title("Leiden cluster count")

    plt.tight_layout()

    local_plot = "/content/clustering_quality.png"
    drive_plot = VAL_DIR / "clustering_quality.png"

    plt.savefig(local_plot, dpi=150)
    plt.savefig(drive_plot, dpi=150)
    plt.show()

    results_df.to_csv(VAL_DIR / "cosmx_clustering_quality.csv", index=False)

    return results, results_df, adata_raw, adata_den


# ==============================================================================
# TASK 3: DIFFERENTIAL EXPRESSION
# ==============================================================================

def compare_differential_expression(denoised_adata, celltype1, celltype2, top_n=20):
    """
    Compare DE genes between two cell types using raw vs denoised data.

    This matches the Xenium-style logic:
      raw = denoised_adata.layers['raw']
      denoised = denoised_adata.X
      Mann-Whitney U test
      Bonferroni alpha = 0.05 / n_genes
      report significant DE genes in raw and denoised
    """

    ct_col = get_cell_type_column(denoised_adata)

    all_types = denoised_adata.obs[ct_col].astype(str).unique()

    ct1_match = [ct for ct in all_types if celltype1.lower() in ct.lower()]
    ct2_match = [ct for ct in all_types if celltype2.lower() in ct.lower()]

    if not ct1_match or not ct2_match:
        print(f"Could not find cell types matching {celltype1} and {celltype2}")
        print(f"Available: {list(all_types)}")
        return None

    mask1 = denoised_adata.obs[ct_col].astype(str).isin(ct1_match).values
    mask2 = denoised_adata.obs[ct_col].astype(str).isin(ct2_match).values

    print(f"Comparing {ct1_match} (n={mask1.sum()}) vs {ct2_match} (n={mask2.sum()})")

    X_raw = ensure_dense(denoised_adata.layers["raw"])
    X_den = ensure_dense(denoised_adata.X)
    genes = denoised_adata.var_names

    de_results = []

    for i, gene in enumerate(genes):
        raw1, raw2 = X_raw[mask1, i], X_raw[mask2, i]
        den1, den2 = X_den[mask1, i], X_den[mask2, i]

        stat_raw, pval_raw = mannwhitneyu(raw1, raw2, alternative="two-sided")
        stat_den, pval_den = mannwhitneyu(den1, den2, alternative="two-sided")

        fc_raw = np.log2((raw1.mean() + 0.1) / (raw2.mean() + 0.1))
        fc_den = np.log2((den1.mean() + 0.1) / (den2.mean() + 0.1))

        de_results.append({
            "gene": gene,
            "pval_raw": pval_raw,
            "pval_den": pval_den,
            "fc_raw": fc_raw,
            "fc_den": fc_den,
            "more_sig_den": pval_den < pval_raw,
        })

    de_df = pd.DataFrame(de_results)

    alpha = 0.05 / len(genes)

    n_sig_raw = int((de_df["pval_raw"] < alpha).sum())
    n_sig_den = int((de_df["pval_den"] < alpha).sum())

    print(f"\nSignificant DE genes, Bonferroni alpha={alpha:.2e}:")
    print(f"  Raw: {n_sig_raw}")
    print(f"  Denoised: {n_sig_den}")
    print(f"  Improvement: {n_sig_den - n_sig_raw:+d}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, (pval_col, fc_col, title) in zip(
        axes,
        [
            ("pval_raw", "fc_raw", "Raw"),
            ("pval_den", "fc_den", "Denoised"),
        ]
    ):
        sig = de_df[pval_col] < alpha

        ax.scatter(
            de_df.loc[~sig, fc_col],
            -np.log10(de_df.loc[~sig, pval_col] + 1e-300),
            alpha=0.3,
            s=5,
            c="gray",
            label="Not significant"
        )

        ax.scatter(
            de_df.loc[sig, fc_col],
            -np.log10(de_df.loc[sig, pval_col] + 1e-300),
            alpha=0.7,
            s=10,
            c="red",
            label=f"Significant, n={sig.sum()}"
        )

        ax.axhline(-np.log10(alpha), color="black", linestyle="--", alpha=0.5)
        ax.axvline(0, color="black", linestyle="-", alpha=0.3)

        ax.set_xlabel("Log2 Fold Change")
        ax.set_ylabel("-Log10 p-value")
        ax.set_title(f"{title}: {ct1_match[0][:10]} vs {ct2_match[0][:10]}")
        ax.legend()

    plt.tight_layout()

    local_de_plot = "/content/de_comparison.png"
    drive_de_plot = VAL_DIR / "de_comparison.png"

    plt.savefig(local_de_plot, dpi=150)
    plt.savefig(drive_de_plot, dpi=150)
    plt.show()

    de_df["neg_log_p_den"] = -np.log10(de_df["pval_den"] + 1e-300)

    top_genes = de_df.nlargest(top_n, "neg_log_p_den")[
        ["gene", "fc_den", "pval_den", "pval_raw"]
    ]

    print(f"\nTop {top_n} DE genes, denoised:")
    display(top_genes)

    safe_1 = ct1_match[0].replace(" ", "_").replace("/", "_")
    safe_2 = ct2_match[0].replace(" ", "_").replace("/", "_")

    de_df.to_csv(VAL_DIR / f"cosmx_de_{safe_1}_vs_{safe_2}.csv", index=False)

    de_summary = pd.DataFrame([{
        "celltype1": ct1_match[0],
        "celltype2": ct2_match[0],
        "n_celltype1": int(mask1.sum()),
        "n_celltype2": int(mask2.sum()),
        "bonferroni_alpha": float(alpha),
        "n_sig_raw": n_sig_raw,
        "n_sig_denoised": n_sig_den,
        "improvement": n_sig_den - n_sig_raw,
    }])
    de_summary.to_csv(VAL_DIR / f"cosmx_de_summary_{safe_1}_vs_{safe_2}.csv", index=False)

    return de_df



# ==============================================================================
# TASK 4: SVG DETECTION
# ==============================================================================

def compare_svg_detection(denoised_adata, method="moran"):
    """
    Compare spatially variable genes detected in raw vs denoised data.

    This keeps the Xenium-style output format:
      - Moran's I analysis
      - 5,000-cell subsample
      - random seed 42
      - spatial graph using sc.pp.neighbors(use_rep='spatial', n_neighbors=15)
      - normalize_total + log1p before Moran's I
      - threshold Moran's I > 0.1

    Correction relative to the earlier CosMx version:
      - raw and denoised matrices are taken from the same aligned denoised_adata object
      - raw = denoised_adata.layers["raw"]
      - denoised = denoised_adata.X
      - same cells, same genes, same spatial graph configuration
      - reports raw-vs-denoised Moran correlation
    """

    n_sample = min(5000, denoised_adata.n_obs)

    rng = np.random.default_rng(42)
    idx = rng.choice(denoised_adata.n_obs, n_sample, replace=False)

    adata_sub = denoised_adata[idx].copy()

    # Coordinate compatibility
    if "x_centroid" not in adata_sub.obs.columns:
        if "center_x_global_px" in adata_sub.obs.columns:
            adata_sub.obs["x_centroid"] = adata_sub.obs["center_x_global_px"].astype(float)
        elif "CenterX_global_px" in adata_sub.obs.columns:
            adata_sub.obs["x_centroid"] = adata_sub.obs["CenterX_global_px"].astype(float)
        else:
            raise KeyError("No x_centroid or center_x_global_px found for SVG detection.")

    if "y_centroid" not in adata_sub.obs.columns:
        if "center_y_global_px" in adata_sub.obs.columns:
            adata_sub.obs["y_centroid"] = adata_sub.obs["center_y_global_px"].astype(float)
        elif "CenterY_global_px" in adata_sub.obs.columns:
            adata_sub.obs["y_centroid"] = adata_sub.obs["CenterY_global_px"].astype(float)
        else:
            raise KeyError("No y_centroid or center_y_global_px found for SVG detection.")

    x_col = "x_centroid"
    y_col = "y_centroid"

    adata_sub.obsm["spatial"] = np.column_stack([
        adata_sub.obs[x_col].values.astype(float),
        adata_sub.obs[y_col].values.astype(float),
    ])

    if "raw" not in adata_sub.layers:
        raise KeyError("adata_sub.layers['raw'] is missing. Cannot run raw-vs-denoised SVG validation.")

    # Build raw and denoised objects from the same aligned subset.
    adata_raw = adata_sub.copy()
    adata_raw.X = ensure_dense(adata_sub.layers["raw"]).astype(np.float32)

    adata_den = adata_sub.copy()
    adata_den.X = ensure_dense(adata_sub.X).astype(np.float32)

    if adata_raw.shape != adata_den.shape:
        raise ValueError(f"Raw and denoised SVG matrices differ in shape: {adata_raw.shape} vs {adata_den.shape}")

    if list(adata_raw.var_names.astype(str)) != list(adata_den.var_names.astype(str)):
        raise ValueError("Raw and denoised SVG matrices have different gene order.")

    if list(adata_raw.obs_names.astype(str)) != list(adata_den.obs_names.astype(str)):
        raise ValueError("Raw and denoised SVG matrices have different cell order.")

    # Same normalization style as Xenium.
    sc.pp.normalize_total(adata_raw, target_sum=1e4)
    sc.pp.log1p(adata_raw)

    sc.pp.normalize_total(adata_den, target_sum=1e4)
    sc.pp.log1p(adata_den)

    # Same graph configuration as Xenium. Since both use identical spatial coordinates,
    # this creates the same spatial-neighborhood logic for raw and denoised data.
    sc.pp.neighbors(adata_raw, use_rep="spatial", n_neighbors=15)
    sc.pp.neighbors(adata_den, use_rep="spatial", n_neighbors=15)

    def compute_morans_i(adata):
        W = adata.obsp["connectivities"]

        row_sums = np.asarray(W.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        W = W.multiply(1 / row_sums[:, None])

        X = ensure_dense(adata.X)

        n = X.shape[0]
        morans = []

        for j in range(X.shape[1]):
            x = X[:, j]
            x_mean = x.mean()
            x_dev = x - x_mean

            numerator = (W @ x_dev) * x_dev
            denominator = (x_dev ** 2).sum()

            if denominator > 0:
                I = n * numerator.sum() / (W.sum() * denominator)
            else:
                I = 0.0

            morans.append(I)

        return np.array(morans)

    print("Computing Moran's I for raw data...")
    morans_raw = compute_morans_i(adata_raw)

    print("Computing Moran's I for denoised data...")
    morans_den = compute_morans_i(adata_den)

    genes = adata_sub.var_names.astype(str)

    svg_df = pd.DataFrame({
        "gene": genes,
        "moran_raw": morans_raw,
        "moran_den": morans_den,
    })

    svg_df["delta_moran"] = svg_df["moran_den"] - svg_df["moran_raw"]

    threshold = 0.1

    svg_raw = set(svg_df[svg_df["moran_raw"] > threshold]["gene"])
    svg_den = set(svg_df[svg_df["moran_den"] > threshold]["gene"])

    moran_corr = svg_df[["moran_raw", "moran_den"]].corr().iloc[0, 1]
    if not np.isfinite(moran_corr):
        moran_corr = np.nan

    print(f"\nSVGs detected, Moran's I > {threshold}:")
    print(f"  Raw: {len(svg_raw)}")
    print(f"  Denoised: {len(svg_den)}")
    print(f"  Overlap: {len(svg_raw & svg_den)}")
    print(f"  New in denoised: {len(svg_den - svg_raw)}")
    print(f"  Lost in denoised: {len(svg_raw - svg_den)}")
    print(f"  Raw vs denoised Moran correlation: {moran_corr:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(svg_df["moran_raw"], svg_df["moran_den"], alpha=0.5, s=10)

    max_val = max(svg_df["moran_raw"].max(), svg_df["moran_den"].max())
    axes[0].plot([0, max_val], [0, max_val], "r--")

    axes[0].axhline(threshold, color="gray", linestyle="--", alpha=0.5)
    axes[0].axvline(threshold, color="gray", linestyle="--", alpha=0.5)

    axes[0].set_xlabel("Moran's I, Raw")
    axes[0].set_ylabel("Moran's I, Denoised")
    axes[0].set_title("Spatial variability: Raw vs Denoised")

    categories = ["Only Raw", "Both", "Only Denoised"]
    values = [
        len(svg_raw - svg_den),
        len(svg_raw & svg_den),
        len(svg_den - svg_raw),
    ]

    axes[1].bar(categories, values)
    axes[1].set_ylabel("Number of SVGs")
    axes[1].set_title(f"SVG comparison, threshold={threshold}")

    for i, v in enumerate(values):
        axes[1].text(i, v + 1, str(v), ha="center")

    plt.tight_layout()

    local_svg_plot = "/content/svg_comparison.png"
    drive_svg_plot = VAL_DIR / "svg_comparison.png"

    plt.savefig(local_svg_plot, dpi=150)
    plt.savefig(drive_svg_plot, dpi=150)
    plt.show()

    svg_df["is_svg_raw"] = svg_df["moran_raw"] > threshold
    svg_df["is_svg_denoised"] = svg_df["moran_den"] > threshold

    svg_df.to_csv(VAL_DIR / "cosmx_svg_morans_i.csv", index=False)

    svg_summary = {
        "method": "Moran's I",
        "n_sampled_cells": int(n_sample),
        "random_state": 42,
        "n_neighbors": 15,
        "threshold": threshold,
        "n_svg_raw": int(len(svg_raw)),
        "n_svg_denoised": int(len(svg_den)),
        "n_svg_overlap": int(len(svg_raw & svg_den)),
        "n_svg_new_in_denoised": int(len(svg_den - svg_raw)),
        "n_svg_lost_in_denoised": int(len(svg_raw - svg_den)),
        "moran_raw_denoised_correlation": float(moran_corr),
        "raw_svg_genes": sorted(list(svg_raw)),
        "denoised_svg_genes": sorted(list(svg_den)),
        "new_in_denoised_genes": sorted(list(svg_den - svg_raw)),
        "lost_in_denoised_genes": sorted(list(svg_raw - svg_den)),
    }

    with open(VAL_DIR / "cosmx_svg_morans_i_summary.json", "w") as f:
        json.dump(svg_summary, f, indent=2)

    return svg_df


# ==============================================================================
# RUN ALL VALIDATIONS
# ==============================================================================

def run_all_validations(denoised_adata):
    """
    Run all downstream validation tasks.
    """

    print("=" * 70)
    print("DOWNSTREAM VALIDATION FOR CELL-LEVEL DENOISING")
    print("=" * 70)

    print("\n" + "=" * 70)
    print("TASK 1: MARKER GENE VALIDATION")
    print("=" * 70)

    marker_results = validate_markers(denoised_adata)

    print("\n" + "=" * 70)
    print("TASK 2: CLUSTERING QUALITY")
    print("=" * 70)

    clustering_results, clustering_df, adata_raw_clustered, adata_den_clustered = (
        evaluate_clustering_quality(
            denoised_adata,
            raw_matrix=None,
            sample_size=50000,
            random_state=42,
            n_pcs=30,
            leiden_resolution=0.5,
        )
    )

    print("\n" + "=" * 70)
    print("TASK 3: DIFFERENTIAL EXPRESSION")
    print("=" * 70)

    # CosMx equivalent of Xenium's tumor vs T-cell comparison.
    # Epithelial cells are the closest broad tumor/epithelial compartment.
    de_results = compare_differential_expression(
        denoised_adata,
        "Epithelial",
        "T lymphocytes"
    )

    print("\n" + "=" * 70)
    print("TASK 4: SVG DETECTION")
    print("=" * 70)

    svg_results = compare_svg_detection(denoised_adata)

    print("\n" + "=" * 70)
    print("VALIDATION COMPLETE")
    print("=" * 70)

    return {
        "markers": marker_results,
        "clustering": clustering_results,
        "clustering_df": clustering_df,
        "adata_raw_clustered": adata_raw_clustered,
        "adata_den_clustered": adata_den_clustered,
        "de": de_results,
        "svg": svg_results,
    }


# ==============================================================================
# RUN
# ==============================================================================

results = run_all_validations(denoised_adata)

In [ ]:
# ==============================================================================
# COSMX CELL 87 — UMAP: RAW VS DENOISED DATA
# Xenium-matched configuration
# ==============================================================================

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# ============================================================
# UMAP: RAW VS DENOISED DATA
# Coherent with downstream validation:
#   raw      = X_raw_counts / denoised_adata.layers['raw']
#   denoised = denoised_adata.X
#   labels   = cell_type
# ============================================================

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

RANDOM_STATE = 42
N_PCS = 30
N_NEIGHBORS = 15

OUTPUT_ROOT = Path("/content/cosmx_cell_level_denoising_outputs")
UMAP_DIR = OUTPUT_ROOT / "umap"
UMAP_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Ensure cell_type exists
# ------------------------------------------------------------

if "cell_type" not in denoised_adata.obs.columns:
    if "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
        denoised_adata.obs["cell_type"] = denoised_adata.obs["Final_CosMx_Cell_Type"].astype(str)
        print("Copied Final_CosMx_Cell_Type to cell_type.")
    elif "cell_type" in annotated_spatial_adata.obs.columns:
        denoised_adata.obs["cell_type"] = (
            annotated_spatial_adata.obs.loc[denoised_adata.obs_names, "cell_type"]
            .astype(str)
            .values
        )
        print("Copied cell_type from annotated_spatial_adata to denoised_adata.")
    else:
        raise KeyError("cell_type column not found in denoised_adata or annotated_spatial_adata.")

missing_labels = denoised_adata.obs["cell_type"].isna().sum()
print(f"Missing cell_type labels: {missing_labels:,}")

if missing_labels > 0:
    raise ValueError(
        f"{missing_labels:,} cells have missing cell_type labels. "
        "Fix labels before plotting UMAP."
    )

# ------------------------------------------------------------
# 2. Use already aligned raw layer from rebuilt denoised_adata
# ------------------------------------------------------------

if "raw" not in denoised_adata.layers:
    raise KeyError(
        "denoised_adata.layers['raw'] is missing. "
        "Run the safe rebuild cell immediately after denoising."
    )

X_raw_umap = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
X_den_umap = ensure_dense(denoised_adata.X).astype(np.float32)

if X_raw_umap.shape != X_den_umap.shape:
    raise ValueError(
        f"Raw layer shape {X_raw_umap.shape} does not match "
        f"denoised matrix shape {X_den_umap.shape}."
    )

delta_umap = X_den_umap - X_raw_umap
negative_entries = int((delta_umap < -1e-6).sum())

print(f"Raw sum: {X_raw_umap.sum(dtype=np.float64):,.0f}")
print(f"Denoised sum: {X_den_umap.sum(dtype=np.float64):,.2f}")
print(f"Net added counts: {delta_umap.sum(dtype=np.float64):,.2f}")
print(f"Negative delta entries: {negative_entries:,}")

if negative_entries > 0:
    raise ValueError(
        "UMAP raw-vs-denoised alignment failed: negative deltas found. "
        "Do not use X_raw_counts directly here; use rebuilt denoised_adata.layers['raw']."
    )

print("Using aligned denoised_adata.layers['raw'] for raw UMAP.")

# ------------------------------------------------------------
# 3. Create temporary AnnData objects
# ------------------------------------------------------------

adata_raw = denoised_adata.copy()
adata_raw.X = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)

adata_den = denoised_adata.copy()
adata_den.X = ensure_dense(denoised_adata.X).astype(np.float32)

# ------------------------------------------------------------
# 4. Normalize, PCA, neighbors, UMAP
# ------------------------------------------------------------

for label, adata_tmp in [("Raw", adata_raw), ("Denoised", adata_den)]:
    print(f"\nProcessing {label}...")

    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)

    n_pcs_use = min(N_PCS, adata_tmp.n_vars - 1)

    sc.pp.pca(
        adata_tmp,
        n_comps=n_pcs_use,
        random_state=RANDOM_STATE
    )

    sc.pp.neighbors(
        adata_tmp,
        n_neighbors=N_NEIGHBORS,
        n_pcs=n_pcs_use,
        random_state=RANDOM_STATE
    )

    sc.tl.umap(
        adata_tmp,
        random_state=RANDOM_STATE
    )

# ------------------------------------------------------------
# 5. Plot raw vs denoised UMAP
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(
    adata_raw,
    color="cell_type",
    ax=axes[0],
    show=False,
    title="Raw counts",
    frameon=False,
    legend_loc=None,
    size=8,
)

sc.pl.umap(
    adata_den,
    color="cell_type",
    ax=axes[1],
    show=False,
    title="Denoised counts",
    frameon=False,
    legend_loc="right margin",
    size=8,
)

plt.tight_layout()

local_umap_png = "/content/umap_raw_vs_denoised.png"

plt.savefig(local_umap_png, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved UMAP plot:")
print(f"  {local_umap_png}")

# ------------------------------------------------------------
# 6. Save UMAP coordinates and objects
# ------------------------------------------------------------

umap_df = pd.DataFrame({
    "cell_id": denoised_adata.obs_names.astype(str),
    "cell_type": denoised_adata.obs["cell_type"].astype(str).values,
    "raw_umap_1": adata_raw.obsm["X_umap"][:, 0],
    "raw_umap_2": adata_raw.obsm["X_umap"][:, 1],
    "denoised_umap_1": adata_den.obsm["X_umap"][:, 0],
    "denoised_umap_2": adata_den.obsm["X_umap"][:, 1],
})

umap_df.to_csv(UMAP_DIR / "cosmx_umap_raw_vs_denoised_coordinates.csv", index=False)

adata_raw.write_h5ad(UMAP_DIR / "cosmx_raw_umap_adata.h5ad")
adata_den.write_h5ad(UMAP_DIR / "cosmx_denoised_umap_adata.h5ad")

print("\nSaved UMAP outputs:")
print(f"  {UMAP_DIR / 'cosmx_umap_raw_vs_denoised_coordinates.csv'}")
print(f"  {UMAP_DIR / 'cosmx_raw_umap_adata.h5ad'}")
print(f"  {UMAP_DIR / 'cosmx_denoised_umap_adata.h5ad'}")

print("\nCOSMX CELL 87 finished successfully.")

In [ ]:
# ==============================================================================
# COSMX CELL: EXPORT
# ==============================================================================
# This saves everything Step 5 molecule-level imputation needs into one folder.
#
# Main Step 5 files:
#   molecules.parquet
#   cell_data.npz
#   denoised_adata.h5ad
#   was_corrected.npy
#   step4_config.json
#
# Important:
#   denoised_adata.X = denoised counts
#   denoised_adata.layers['raw'] = clean molecule-derived raw counts
# ==============================================================================

import os
import json
import time
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------------------

STEP4_READY_DIR = Path("/content/drive/MyDrive/diffusion/step4_cosmx")

EXPORT_DIR_LOCAL = Path("/content/cosmx_step4_exports")
EXPORT_DIR_DRIVE = STEP4_READY_DIR / "step4_exports"

EXPORT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
EXPORT_DIR_DRIVE.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("EXPORTING FILES FOR STEP 5")
print("=" * 60)
print(f"Local export folder: {EXPORT_DIR_LOCAL}")
print(f"Drive export folder: {EXPORT_DIR_DRIVE}")

# ------------------------------------------------------------------------------
# Required variable checks
# ------------------------------------------------------------------------------

if "denoised_adata" not in globals():
    raise NameError("denoised_adata not found. Run denoising and validation cells first.")

if "metadata" not in globals():
    raise NameError("metadata not found. Run this export in the same runtime after denoising.")

if "was_corrected" not in metadata:
    raise KeyError("metadata['was_corrected'] is missing.")

# ------------------------------------------------------------------------------
# Enforce clean raw layer consistency before saving
# Reorder X_raw_counts to match denoised_adata.obs_names and denoised_adata.var_names
# ------------------------------------------------------------------------------

xraw_path = STEP4_READY_DIR / "X_raw_counts.npy"
gene_names_path = STEP4_READY_DIR / "gene_names.npy"
cell_ids_path = STEP4_READY_DIR / "cell_ids.npy"

if "X_raw_counts" not in globals():
    if not xraw_path.exists():
        raise FileNotFoundError(f"X_raw_counts.npy not found: {xraw_path}")
    X_raw_counts = np.load(xraw_path)
    print(f"Loaded X_raw_counts from: {xraw_path}")

if not gene_names_path.exists():
    raise FileNotFoundError(f"gene_names.npy not found: {gene_names_path}")

if not cell_ids_path.exists():
    raise FileNotFoundError(f"cell_ids.npy not found: {cell_ids_path}")

raw_gene_names = np.load(gene_names_path).astype(str)
raw_cell_ids = np.load(cell_ids_path).astype(str)

target_gene_names = denoised_adata.var_names.astype(str).to_numpy()
target_cell_ids = denoised_adata.obs_names.astype(str).to_numpy()

raw_gene_index = pd.Index(raw_gene_names)
raw_cell_index = pd.Index(raw_cell_ids)

col_idx = raw_gene_index.get_indexer(target_gene_names)
row_idx = raw_cell_index.get_indexer(target_cell_ids)

if (col_idx < 0).any():
    missing_genes = target_gene_names[col_idx < 0].tolist()
    raise ValueError(f"Some denoised_adata genes are missing from gene_names.npy: {missing_genes[:20]}")

if (row_idx < 0).any():
    missing_cells = target_cell_ids[row_idx < 0].tolist()
    raise ValueError(f"Some denoised_adata cells are missing from cell_ids.npy: {missing_cells[:20]}")

X_raw_export = X_raw_counts[np.ix_(row_idx, col_idx)].astype(np.float32)

if X_raw_export.shape != denoised_adata.shape:
    raise ValueError(
        f"Reordered raw matrix shape {X_raw_export.shape} does not match "
        f"denoised_adata shape {denoised_adata.shape}"
    )

denoised_adata.layers["raw"] = X_raw_export.copy()
denoised_adata.layers["raw_molecule_counts_clean"] = X_raw_export.copy()

print("Updated denoised_adata.layers['raw'] with gene/cell-order-safe X_raw_counts.")
print(f"Raw export sum: {X_raw_export.sum(dtype=np.float64):,.0f}")

# Ensure uncertainty exists
if "uncertainty" not in denoised_adata.layers:
    if "uncertainty" in globals():
        denoised_adata.layers["uncertainty"] = uncertainty.astype(np.float32)
        print("Added uncertainty layer from global uncertainty variable.")
    else:
        raise KeyError("denoised_adata.layers['uncertainty'] is missing.")

# Ensure cell_type exists
if "cell_type" not in denoised_adata.obs.columns:
    if "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
        denoised_adata.obs["cell_type"] = denoised_adata.obs["Final_CosMx_Cell_Type"].astype(str)
    else:
        raise KeyError("No cell_type or Final_CosMx_Cell_Type found in denoised_adata.obs.")

# Add correction count column
denoised_adata.obs["was_corrected_count"] = metadata["was_corrected"].sum(axis=1)

# ------------------------------------------------------------------------------
# FILE 1: molecules.parquet
# ------------------------------------------------------------------------------

src_mol = STEP4_READY_DIR / "molecules.parquet"
dst_mol = EXPORT_DIR_LOCAL / "molecules.parquet"

if src_mol.exists():
    shutil.copy2(src_mol, dst_mol)
    mol_check = pd.read_parquet(dst_mol, columns=["cell_id"]).shape[0]
    print(f"1. molecules.parquet: {mol_check:,} rows — COPIED")
else:
    raise FileNotFoundError(f"molecules.parquet not found: {src_mol}")

# ------------------------------------------------------------------------------
# FILE 2: cell_data.npz
# ------------------------------------------------------------------------------

src_geom = STEP4_READY_DIR / "cell_data.npz"
dst_geom = EXPORT_DIR_LOCAL / "cell_data.npz"

if src_geom.exists():
    shutil.copy2(src_geom, dst_geom)
    geom_check = np.load(dst_geom, allow_pickle=False)
    print(f"2. cell_data.npz: {len(geom_check['cell_ids']):,} cells — COPIED")
else:
    raise FileNotFoundError(f"cell_data.npz not found: {src_geom}")

# ------------------------------------------------------------------------------
# FILE 3: denoised_adata.h5ad
# ------------------------------------------------------------------------------

dst_adata = EXPORT_DIR_LOCAL / "denoised_adata.h5ad"

denoised_adata.write(dst_adata)

print(f"3. denoised_adata.h5ad: {denoised_adata.shape} — SAVED")
print(f"   .X = denoised counts")
print(f"   .layers = {list(denoised_adata.layers.keys())}")
print(f"   raw sum = {ensure_dense(denoised_adata.layers['raw']).sum(dtype=np.float64):,.0f}")
print(f"   denoised sum = {ensure_dense(denoised_adata.X).sum(dtype=np.float64):,.2f}")

# ------------------------------------------------------------------------------
# FILE 4: was_corrected.npy
# ------------------------------------------------------------------------------

dst_wc = EXPORT_DIR_LOCAL / "was_corrected.npy"
np.save(dst_wc, metadata["was_corrected"])

print(f"4. was_corrected.npy: {metadata['was_corrected'].shape} — SAVED")
print(f"   corrected pairs: {int(metadata['was_corrected'].sum()):,}")

# ------------------------------------------------------------------------------
# FILE 5: step4_config.json
# ------------------------------------------------------------------------------

raw_sum = float(ensure_dense(denoised_adata.layers["raw"]).sum(dtype=np.float64))
den_sum = float(ensure_dense(denoised_adata.X).sum(dtype=np.float64))

config = {
    "platform": "CosMx",
    "sample_name": "Lung5_Rep1",
    "reference_dataset": "GSE131907",
    "export_time": time.strftime("%Y-%m-%d %H:%M:%S"),

    "n_cells": int(denoised_adata.n_obs),
    "n_genes": int(denoised_adata.n_vars),
    "cell_type_column": "cell_type",
    "cell_types": sorted(denoised_adata.obs["cell_type"].astype(str).unique().tolist()),
    "shared_genes": denoised_adata.var_names.astype(str).tolist(),
    "cell_ids": denoised_adata.obs_names.astype(str).tolist(),

    "raw_sum": raw_sum,
    "denoised_sum": den_sum,
    "added_counts": den_sum - raw_sum,

    "corrected_pairs": int(metadata["was_corrected"].sum()),
    "total_cell_gene_pairs": int(metadata["was_corrected"].size),
    "correction_rate_percent": float(100 * metadata["was_corrected"].sum() / metadata["was_corrected"].size),

    "denoising_parameters": {
        "spatial_weight": 0.5,
        "reference_weight": 0.7,
        "spatial_radius": 100.0,
        "k_min": 5,
        "k_max": 30,
        "dropout_p_threshold": 0.35,
        "shrinkage": 0.87,
        "calibrate": False,
    },

    "files": {
        "molecules": "molecules.parquet",
        "cell_data": "cell_data.npz",
        "denoised_adata": "denoised_adata.h5ad",
        "was_corrected": "was_corrected.npy",
        "config": "step4_config.json",
    },

    "notes": [
        "denoised_adata.X contains denoised cell-gene counts.",
        "denoised_adata.layers['raw'] contains clean molecule-derived raw counts from X_raw_counts.npy.",
        "was_corrected.npy is a boolean cell-gene matrix showing which entries were corrected.",
        "Correction rate is percentage of cell-gene pairs, not percentage of molecules."
    ]
}

dst_config = EXPORT_DIR_LOCAL / "step4_config.json"

with open(dst_config, "w") as f:
    json.dump(config, f, indent=2)

print("5. step4_config.json — SAVED")

# ------------------------------------------------------------------------------
# Optional support files
# ------------------------------------------------------------------------------

optional_files = [
    "X_raw_counts.npy",
    "gene_names.npy",
    "cell_ids.npy",
    "clean_preprocessing_summary.json",
    "reference_profiles.csv",
    "scrna_ref.h5ad",
    "scrna_ref.npz",
    "shared_genes.txt",
]

for fname in optional_files:
    src = STEP4_READY_DIR / fname
    dst = EXPORT_DIR_LOCAL / fname

    if src.exists():
        shutil.copy2(src, dst)
        print(f"Optional: {fname} — COPIED")
    else:
        print(f"Optional: {fname} not found, skipped.")

# ------------------------------------------------------------------------------
# Copy local export to Drive export folder
# ------------------------------------------------------------------------------

print("\n" + "=" * 60)
print("COPYING EXPORT TO GOOGLE DRIVE")
print("=" * 60)

for src in sorted(EXPORT_DIR_LOCAL.glob("*")):
    dst = EXPORT_DIR_DRIVE / src.name
    shutil.copy2(src, dst)

    if not dst.exists():
        raise FileNotFoundError(f"Failed to copy {src.name} to Drive.")

    print(f"[DRIVE] {dst.name:40s} {dst.stat().st_size / 1e6:,.2f} MB")

# ------------------------------------------------------------------------------
# Final verification
# ------------------------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL EXPORT VERIFICATION")
print("=" * 60)

required_final = [
    "molecules.parquet",
    "cell_data.npz",
    "denoised_adata.h5ad",
    "was_corrected.npy",
    "step4_config.json",
]

missing = []

for fname in required_final:
    p = EXPORT_DIR_DRIVE / fname

    if not p.exists():
        missing.append(fname)
        print(f"[MISSING] {fname}")
    else:
        print(f"[OK] {fname:35s} {p.stat().st_size / 1e6:,.2f} MB")

if missing:
    raise FileNotFoundError(f"Missing final export files: {missing}")

# Quick shape checks
import scanpy as sc

adata_check = sc.read_h5ad(EXPORT_DIR_DRIVE / "denoised_adata.h5ad")
wc_check = np.load(EXPORT_DIR_DRIVE / "was_corrected.npy")
geom_check = np.load(EXPORT_DIR_DRIVE / "cell_data.npz", allow_pickle=False)
mol_check_rows = pd.read_parquet(EXPORT_DIR_DRIVE / "molecules.parquet", columns=["cell_id"]).shape[0]

print("\nQuick checks:")
print(f"  denoised_adata shape : {adata_check.shape}")
print(f"  was_corrected shape  : {wc_check.shape}")
print(f"  cell_data cells      : {len(geom_check['cell_ids']):,}")
print(f"  molecules rows       : {mol_check_rows:,}")
print(f"  raw sum              : {ensure_dense(adata_check.layers['raw']).sum(dtype=np.float64):,.0f}")
print(f"  denoised sum         : {ensure_dense(adata_check.X).sum(dtype=np.float64):,.2f}")

if adata_check.shape != wc_check.shape:
    raise ValueError("denoised_adata shape and was_corrected shape do not match.")

if adata_check.n_obs != len(geom_check["cell_ids"]):
    raise ValueError("denoised_adata cell count and cell_data cell count do not match.")

if adata_check.shape != X_raw_counts.shape:
    raise ValueError("denoised_adata shape and X_raw_counts shape do not match.")

raw_check_sum = float(ensure_dense(adata_check.layers["raw"]).sum(dtype=np.float64))

if abs(raw_check_sum - float(X_raw_counts.sum(dtype=np.float64))) > 1:
    raise ValueError("Exported raw layer does not match X_raw_counts sum.")

# Visible marker file
marker = EXPORT_DIR_DRIVE / "_COSMX_STEP4_EXPORT_COMPLETE.txt"

with open(marker, "w") as f:
    f.write("CosMx Step 4 final denoising export completed successfully.\n")
    f.write(f"Export folder: {EXPORT_DIR_DRIVE}\n")
    f.write("denoised_adata.X = denoised counts\n")
    f.write("denoised_adata.layers['raw'] = X_raw_counts\n")

print("\n" + "=" * 60)
print("COSMX FINAL EXPORT COMPLETE")
print("=" * 60)
print("Final Drive export folder:")
print(f"  {EXPORT_DIR_DRIVE}")
print()
print("Core Step 5 files:")
for fname in required_final:
    print(f"  {fname}")
print()
print("Visible marker:")
print("  _COSMX_STEP4_EXPORT_COMPLETE.txt")